---
# >>> Section from `notebooks/p1_data.ipynb`


# Milestone 1: F1 Podium Prediction. Part 1: Data audit, cleaning and EDA

**Target:** `podium = 1` if the driver finishes in the top 3. **One row = one driver in one race.** We predict after qualifying and before the race starts, so only pre-race information can be used as features.

Sections: 1) setup, 2) loading and key audit, 3) EDA before cleaning, 4) cleaning and types, 5) duplicates, 6) Indy 500 and DNQs, 7) EDA after cleaning, 8) `load_clean()` and data dictionary.

## 1. Setup
Fixed random seed and one `DATA_DIR` variable that finds the CSVs on Kaggle or on a laptop, so the notebook runs with a single "Run All".

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")


import os, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42

# Single shared-drive policy switch for the merged notebook.
# False = keep one row per driver-race and flag shared drives; True = exclude those rows.
EXCLUDE_SHARED_DRIVES = False
random.seed(SEED)
np.random.seed(SEED)

SENT = "\\N"                    # the CSVs write a missing value as the text \N
KEYS = ["raceId", "driverId"]   # one row per driver per race
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]


def find_data_dir():
    """Works on Kaggle (/kaggle/input/...) and on a laptop (data/raw)."""
    folders = [Path("data/raw"), Path("../data/raw"), Path("data"), Path(".")]
    if Path("/kaggle/input").exists():
        folders = [p.parent for p in Path("/kaggle/input").rglob("races.csv")] + folders
    for folder in folders:
        if all((folder / f"{t}.csv").exists() for t in TABLES):
            return folder
    raise FileNotFoundError("F1 CSV files not found. Add the Kaggle dataset or put the CSVs in data/raw/")


def load_raw_tables(data_dir):
    """Read every CSV as plain text so nothing is guessed or silently changed."""
    raw = {}
    for t in TABLES:
        raw[t] = pd.read_csv(Path(data_dir) / f"{t}.csv", dtype=str, keep_default_na=False,
                             encoding_errors="replace")
    return raw


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def save_fig(name):
    """Save the current figure as a PNG for the report."""
    os.makedirs("reports", exist_ok=True)
    plt.savefig(f"reports/{name}.png", dpi=130, bbox_inches="tight")


print("data folder:", DATA_DIR)

## 2. Loading the 14 tables and checking the keys

**Why this choice?** We read every file as text. The files write missing values as the text `\N`, and if pandas guessed the types, a numeric column with one `\N` would silently turn into text. Reading everything as text lets us count the `\N`s before changing anything, and the raw tables stay untouched (we only work on copies).

**Why check keys first?** Everything later is a join. If a primary key is duplicated, a join multiplies rows. If a foreign key points to a row that does not exist, a join silently drops rows. So we check that every primary key is unique and not empty, and that every foreign key exists in the table it points to.

In [ ]:
RAW = load_raw_tables(DATA_DIR)
display(pd.DataFrame({"rows": {t: len(df) for t, df in RAW.items()}, "columns": {t: df.shape[1] for t, df in RAW.items()}}))

In [ ]:
primary_keys = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"], "drivers": ["driverId"], "driver_standings": ["driverStandingsId"],
    "races": ["raceId"], "results": ["resultId"], "qualifying": ["qualifyId"],
    "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"], "seasons": ["year"], "status": ["statusId"],
}
rows = []
for table, cols in primary_keys.items():
    df = RAW[table]
    empty = ((df[cols] == SENT) | (df[cols] == "")).any(axis=1).sum()
    duplicated = df.duplicated(cols, keep=False).sum()
    rows.append({"table": table, "primary key": "+".join(cols), "rows": len(df),
                 "empty keys": empty, "rows with duplicate key": duplicated, "OK": empty == 0 and duplicated == 0})
pk_audit = pd.DataFrame(rows)
display(pk_audit)

In [ ]:
foreign_keys = [  # (child table, child column, parent table, parent column)
    ("results", "raceId", "races", "raceId"), ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"), ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"), ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    ("sprint_results", "raceId", "races", "raceId"), ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"), ("sprint_results", "statusId", "status", "statusId"),
    ("driver_standings", "raceId", "races", "raceId"), ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"), ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"), ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"), ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"), ("pit_stops", "driverId", "drivers", "driverId"),
]
rows = []
for child, col, parent, pcol in foreign_keys:
    values = RAW[child][col]
    values = values[(values != SENT) & (values != "")]          # empty foreign keys are "missing", not "broken"
    orphans = (~values.isin(RAW[parent][pcol])).sum()
    rows.append({"child column": f"{child}.{col}", "points to": f"{parent}.{pcol}",
                 "orphans": orphans, "OK": orphans == 0})
fk_audit = pd.DataFrame(rows)
display(fk_audit)
print("primary keys OK:", int(pk_audit["OK"].sum()), "of", len(pk_audit), "| foreign keys OK:", int(fk_audit["OK"].sum()), "of", len(fk_audit))

**What this means.** A table marked OK can safely be the "one" side of a join. Any orphan row is reported, not repaired: we never invent a missing parent. Later we use inner joins on `raceId`, so a result with no race is dropped (it could not be placed in time anyway). Note that a surrogate key like `resultId` is unique by construction, so the real question for our table is whether `(raceId, driverId)` is unique. We check that in section 5.

## 3. EDA before cleaning

Everything here is computed from the raw tables, so we can compare with the cleaned data in section 7.

**3.1 Missing values (`\N`) per column.** We count the `\N` cells in every column and plot the worst columns and the overall distribution.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        n = (df[col] == SENT).sum()
        rows.append({"table": table, "column": col, "missing": n, "pct": 100 * n / len(df)})
missing = pd.DataFrame(rows)
with_missing = missing[missing["missing"] > 0].sort_values("pct", ascending=False)
print(len(with_missing), "of", len(missing), "columns contain at least one \\N")
display(with_missing.head(25))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top = with_missing.head(20)
ax[0].barh(top["table"] + "." + top["column"], top["pct"], color="tab:blue")
ax[0].invert_yaxis(); ax[0].set_xlabel("% of rows that are \\N"); ax[0].set_title("Top 20 columns by missing values")
ax[1].hist(missing["pct"], bins=20, color="tab:orange"); ax[1].set_yscale("log")
ax[1].set_xlabel("% missing in a column"); ax[1].set_ylabel("number of columns (log scale)")
ax[1].set_title("Missingness over all columns")
save_fig("fig_p1_01_missing_values_raw")
plt.show()

**What we see.** Most columns have no `\N` at all and a few have a lot. The big ones are the qualifying times `q2` and `q3` (a driver knocked out in Q1 never drives Q2 or Q3, and before 2006 there was no Q2 or Q3), the fastest-lap columns in old seasons (not recorded), and the newer schedule columns in `races`. The reason differs per column, so in cleaning we only turn `\N` into `NaN` and we do **not** fill anything in. Filling in is done later by the modeling team on the training years only.

**3.2 Field size and podium rate per season.** The podium always has 3 places, so the share of drivers on the podium is about 3 divided by the number of cars (the field size).

In [ ]:
def num(s):
    return pd.to_numeric(s.replace(SENT, np.nan), errors="coerce")

res_raw = RAW["results"][["raceId", "driverId", "positionOrder"]].copy()
res_raw["positionOrder"] = num(res_raw["positionOrder"])
races_raw = RAW["races"][["raceId", "year"]].copy()
races_raw["year"] = num(races_raw["year"])
res_raw = res_raw.merge(races_raw, on="raceId")
res_raw["podium"] = res_raw["positionOrder"] <= 3

season_raw = res_raw.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_raw["rows_per_race"] = season_raw["rows"] / season_raw["races"]
season_raw["unique_drivers_per_race"] = res_raw.groupby(["year", "raceId"])["driverId"].nunique().groupby("year").mean()
season_raw["3_divided_by_field"] = 3 / season_raw["rows_per_race"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(season_raw.index, season_raw["rows_per_race"], label="result rows per race")
ax[0].plot(season_raw.index, season_raw["unique_drivers_per_race"], "--", label="different drivers per race")
ax[0].set_title("Entrants per race (raw)"); ax[0].set_xlabel("season"); ax[0].legend()
ax[1].plot(season_raw.index, season_raw["podium_rate"], label="observed podium rate")
ax[1].plot(season_raw.index, season_raw["3_divided_by_field"], "--", label="3 / field size")
ax[1].set_title("Podium rate per season (raw)"); ax[1].set_xlabel("season"); ax[1].legend()
save_fig("fig_p1_02_field_size_podium_rate_raw")
plt.show()
print("overall podium rate (raw rows):", round(res_raw["podium"].mean(), 3))

**What we see.** The field size changes a lot over the years (big and uneven fields in the 1950s and in the 1980s-90s, about 20 cars today), so the podium rate changes too: about 15% with 20 cars, much lower with 30+. In the early years there are more result rows per race than different drivers. That gap is the first sign of duplicate rows (section 5). So accuracy is not a good metric for this problem, and the class balance will be different in our train, validation and test years.

**3.3 Qualifying coverage per season.**

In [ ]:
quali_raw = RAW["qualifying"]
flags = quali_raw.assign(has_q1=(quali_raw["q1"] != SENT) & (quali_raw["q1"] != ""),
                         has_q3=(quali_raw["q3"] != SENT) & (quali_raw["q3"] != ""))
flags = flags.groupby(KEYS)[["has_q1", "has_q3"]].max().reset_index()
flags["has_quali_row"] = True

cov = res_raw[KEYS + ["year"]].merge(flags, on=KEYS, how="left")
cov[["has_quali_row", "has_q1", "has_q3"]] = cov[["has_quali_row", "has_q1", "has_q3"]].fillna(False).astype(bool)
coverage = cov.groupby("year")[["has_quali_row", "has_q1", "has_q3"]].mean()
first_year = coverage.index[coverage["has_quali_row"] > 0].min()

ax = coverage.plot(figsize=(11, 4.5), lw=2)
ax.axvline(first_year, color="k", ls=":")
ax.set_title(f"Share of result rows with qualifying data (first season: {int(first_year)})")
ax.set_xlabel("season"); ax.set_ylabel("share of rows")
save_fig("fig_p1_03_qualifying_coverage")
plt.show()
print("first season with qualifying rows:", int(first_year))

**What we see.** There is no qualifying data at all before the first season printed above, then coverage grows. Q1 appears first and Q2/Q3 only in the knockout format (2006 onwards).

**Why we do not fill in old qualifying times:** for the old seasons the data is not "randomly missing", the session is simply not in the dataset (and the formats were different, so a 1962 time is not comparable with a 2015 time). Filling it with an average would invent a feature for half of F1 history. Instead `load_clean()` keeps a `has_qualifying` flag, and the grid position (always available) is the fallback for old seasons. The modeling team can also decide to train only on the seasons with qualifying data.

## 4. Cleaning and types

**Why this choice?**
* `\N` is text. In a number column it turns the whole column into text, so we cannot average it, and comparing text like `"10" < "2"` is wrong. We replace `\N` (and blank cells, which turned up in `q2`/`q3`) by a real missing value, and we convert columns to proper types: ids and counts to integers that can hold missing values (`Int64`), dates to datetime, and lap times like `1:27.452` to seconds (`87.452`) so they can be compared and subtracted.
* We only convert, we do not fill in. Why a value is missing differs by column (see 3.1), so one rule would be wrong, and filling with an average from all years could leak information from the future.
* One check beyond the types: a qualifying lap above 200 s cannot be real. One `q1` value is 1,002.64 s (1995 Japanese Grand Prix), while the longest real laps are the wet Belgian Grand Prix of 1994 (about 2:31 to 2:34). Values above 200 s are set to missing, not replaced by a guess. This affects one value.


In [ ]:
INT_COLS = {
    "races": ["year", "round"], "seasons": ["year"], "drivers": ["number"],
    "constructor_standings": ["position", "wins"], "driver_standings": ["position", "wins"],
    "results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap", "rank"],
    "qualifying": ["number", "position"],
    "lap_times": ["lap", "position", "milliseconds"],
    "pit_stops": ["stop", "lap", "milliseconds"],
    "sprint_results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap"],
}
FLOAT_COLS = {
    "circuits": ["lat", "lng", "alt"],
    "constructor_results": ["points"], "constructor_standings": ["points"], "driver_standings": ["points"],
    "results": ["points", "fastestLapSpeed"], "sprint_results": ["points", "fastestLapSpeed"],
}
DATE_COLS = {
    "races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"],
    "drivers": ["dob"],
}
TIME_COLS = {   # text lap times -> seconds (old column: new column)
    "results": {"fastestLapTime": "fastestLapTime_s"},
    "sprint_results": {"fastestLapTime": "fastestLapTime_s"},
    "qualifying": {"q1": "q1_s", "q2": "q2_s", "q3": "q3_s"},
    "lap_times": {"time": "time_s"},
    "pit_stops": {"duration": "duration_s"},
}


MAX_QUALI_LAP_S = 200   # a qualifying lap above 200 s is a data error (one q1 value is 1,002.64 s)


def to_seconds(s):
    """'1:27.452' -> 87.452 (also handles '59.3' and '1:02:03.5'). Anything else -> NaN."""
    parts = s.astype("string").str.extract(r"^(?:(?:(\d+):)?(\d+):)?(\d+(?:\.\d+)?)$")
    hours = pd.to_numeric(parts[0]).fillna(0)
    minutes = pd.to_numeric(parts[1]).fillna(0)
    seconds = pd.to_numeric(parts[2])
    return (hours * 3600 + minutes * 60 + seconds).astype("float64")


def clean_tables(raw):
    """Return cleaned copies of all tables. The raw tables are not touched."""
    clean = {}
    for name, df in raw.items():
        d = df.replace(SENT, np.nan)                      # \N -> NaN
        d = d.replace(r"^\s*$", np.nan, regex=True)       # blank cells are missing too
        for c in d.columns:                               # ids and counts -> integers (can hold NaN)
            if c.endswith("Id") or c in INT_COLS.get(name, []):
                d[c] = pd.to_numeric(d[c], errors="coerce").round().astype("Int64")
        for c in FLOAT_COLS.get(name, []):
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors="coerce")
        for c in DATE_COLS.get(name, []):                 # text -> real dates
            if c in d.columns:
                d[c] = pd.to_datetime(d[c], format="%Y-%m-%d", errors="coerce")
        for old, new in TIME_COLS.get(name, {}).items():  # text lap times -> seconds
            if old in d.columns:
                d[new] = to_seconds(d[old])
                d = d.drop(columns=[old])
        if name == "qualifying":                          # impossible lap times become missing (not filled in)
            for c in ["q1_s", "q2_s", "q3_s"]:
                d[c] = d[c].where(d[c] <= MAX_QUALI_LAP_S)
        clean[name] = d
    return clean

In [ ]:
CLEAN = clean_tables(RAW)

# the lap-time converter on a few examples
test = pd.Series(["1:27.452", "59.3", "1:02:03.5", np.nan, "oops"])
print(dict(zip(test.fillna("NaN"), to_seconds(test).round(3))))

# no \N left anywhere
left = 0
for name, df in CLEAN.items():
    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype) in ("string", "str"):
            left += (df[col] == SENT).sum()
print("\\N left in the cleaned tables:", left)

pairs = [("results", "grid", "grid"), ("results", "positionOrder", "positionOrder"), ("results", "points", "points"),
         ("results", "fastestLapTime", "fastestLapTime_s"), ("qualifying", "q3", "q3_s"),
         ("races", "date", "date"), ("drivers", "dob", "dob")]
display(pd.DataFrame([(t, f"{a} -> {b}", str(RAW[t][a].dtype), str(CLEAN[t][b].dtype)) for t, a, b in pairs],
                     columns=["table", "column", "type before", "type after"]))

# results joined with the race info and the status text (needed for the next sections)
RES = CLEAN["results"].merge(CLEAN["races"][["raceId", "year", "round", "date", "circuitId"]], on="raceId", how="inner")
RES = RES.merge(CLEAN["status"], on="statusId", how="left")
print("result rows:", len(RES), "(raw:", len(RAW["results"]), ")")

### What does `grid = 0` mean?
The project notes say `0` is probably a pit-lane start. We check two ideas:
* **Pit-lane start:** the driver did race (laps > 0) but had no grid slot.
* **No grid slot because he did not start:** DNQ or withdrawn drivers, mostly in old seasons.

In [ ]:
g = RES.copy()
g["grid_zero"] = (g["grid"] == 0).fillna(False).astype(bool)
g["decade"] = (g["year"] // 10 * 10).astype(int)
g["ran"] = g["laps"].fillna(0) > 0

table = g.groupby("decade").agg(rows=("grid", "size"), grid_zero=("grid_zero", "sum"))
table["grid_zero_and_raced"] = g[g.grid_zero & g.ran].groupby("decade").size()
table["grid_zero_no_laps"] = g[g.grid_zero & ~g.ran].groupby("decade").size()
table = table.fillna(0).astype(int)
display(table)

table[["grid_zero_and_raced", "grid_zero_no_laps"]].plot(kind="bar", stacked=True, figsize=(9, 4), color=["tab:green", "tab:red"])
plt.title("Rows with grid = 0, by decade"); plt.ylabel("rows"); save_fig("fig_p1_04_grid_zero_by_decade")
plt.show()

old, new = g[g.grid_zero & (g.year < 1990)], g[g.grid_zero & (g.year >= 2000)]
print("before 1990:", len(old), "rows | share that completed laps:", round(old["ran"].mean(), 2), "| positionText:", old["positionText"].value_counts().head(4).to_dict())
print("2000+      :", len(new), "rows | share that completed laps:", round(new["ran"].mean(), 2), "| top statuses:", new["status"].value_counts().head(4).to_dict())

**Conclusion.** `grid = 0` means "no grid slot", and there are two kinds of rows behind it. In modern seasons these are drivers who raced from the **pit lane** (after a penalty, a car change, or missing the grid). In old seasons they are mostly drivers who **did not start** (failed to qualify or withdrew) or whose slot was not recorded.

**Why this choice for the table:** `0` must never be read as "better than pole position". We remove the non-starters first (section 6), and what is left with `grid = 0` is flagged as `pitlane_start`. We keep the original `grid` column unchanged.

## 5. One row per driver per race (duplicates)

**Why we need exactly one row per `(raceId, driverId)`:** every row is one question for the model: "will this driver finish on the podium in this race?" If a driver has two rows, they can have different labels and different features (two cars, two grid slots), so the model would be asked the same question twice with different answers. It also counts that driver twice in every rate. Two rows for the same driver also means the same race can end up in two different splits.

**Why duplicates exist:** until 1957 a driver could take over a teammate's car during the race (a *shared drive*) and the points were split. The data has one result row per car, so the same driver gets two rows in one race.

In [ ]:
dup_all = RES.duplicated(KEYS, keep=False)        # every row that belongs to a duplicate group
extra = RES.duplicated(KEYS, keep="first")        # the rows beyond the first one
print("duplicate groups (driver-races with more than one row):", RES[dup_all].groupby(KEYS).ngroups)
print("rows in those groups:", int(dup_all.sum()), "| extra rows to remove:", int(extra.sum()))

per_year = extra.groupby(RES["year"]).sum().reindex(range(1950, 2025), fill_value=0)
print("extra rows in 1950-1957:", int(per_year.loc[1950:1957].sum()), "of", int(per_year.sum()))
plt.figure(figsize=(12, 3.8))
plt.bar(per_year.index, per_year.values, color="tab:red")
plt.title("Duplicate (raceId, driverId) rows per season"); plt.xlabel("season"); plt.ylabel("extra rows"); save_fig("fig_p1_05_duplicates_per_season")
plt.show()

examples = RES[dup_all].merge(CLEAN["drivers"][["driverId", "surname"]], on="driverId")
examples = examples.merge(CLEAN["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
display(examples.sort_values(["year", "raceId", "surname"])[["year", "raceId", "surname", "constructor", "number", "grid", "positionOrder", "status"]].head(10))

### The rule for choosing which row to keep
There are two separate questions, and they follow different rules:

1. **Which row do we keep?** Its `grid`, `constructorId` and `number` become features, so we may only use facts that are known **before the race**: first a car that really started (`grid > 0`), then the lowest grid slot, then `constructorId` and car number as arbitrary but fixed tie-breaks. `resultId` is only the very last tie-break, for rows that look identical before the race. We do **not** use `positionOrder` here, because choosing the row by its finishing position would let the race result decide the features.
2. **What is the label?** The label is the outcome, so it may use the result. `podium = 1` if the driver's **best** finish over all his rows in that race is 1, 2 or 3 (a driver who took over a podium car did get that finish). For a normal driver with one row this is just `positionOrder <= 3`.

**Limitation:** when two cars both started, the pre-race columns cannot tell which one was the driver's own, so the kept `grid` may be the teammate's slot. This only affects shared drives, which are a tiny part of the data (85 driver-races before the Indy 500 and the non-starters are removed, 61 rows in the final table, see section 8) and all of them are from 1950-1964, so the validation and test years are not touched. These rows are flagged with `shared_drive`.

In [ ]:
def flag_indy500(races, circuits):
    """Mark the Indianapolis 500 rounds that counted for the championship (1950-1960).
    The same circuit hosted a real F1 US Grand Prix in 2000-2007, so we also check the year."""
    races = races.copy()
    indy_ids = circuits.loc[circuits["name"].str.contains("Indianapolis", na=False), "circuitId"]
    by_name = races["name"].str.contains("Indianapolis 500", na=False)
    by_place = races["circuitId"].isin(indy_ids) & races["year"].between(1950, 1960)
    races["is_indy500"] = (by_name | by_place).astype(bool)
    return races


def find_non_starters(res):
    """Drivers who did not qualify / withdrew. Only uses facts known before the race."""
    by_code = res["positionText"].isin(["F", "W"])      # F = failed to qualify, W = withdrew
    by_status = res["status"].str.contains("did not (?:pre)?qualify|not qualified|withdrew|107%",
                                           case=False, regex=True, na=False)
    return (by_code | by_status).astype(bool)


def keep_one_row_per_driver(res):
    """One row per (raceId, driverId).
    WHICH row we keep uses only facts known before the race (never the finishing position):
      1) a car that really started (grid > 0), 2) the lowest grid slot, 3) constructorId, car number,
      4) resultId only as a last resort (rows that look identical before the race).
    The LABEL is different: podium = the driver's best finish over all his rows <= 3."""
    res = res.reset_index(drop=True).copy()
    res["n_raw_rows"] = res.groupby(KEYS)["resultId"].transform("size")
    res["best_positionOrder"] = res.groupby(KEYS)["positionOrder"].transform("min")
    res["started"] = res["grid"].fillna(0) > 0
    res["grid_for_sort"] = res["grid"].where(res["started"])
    res = res.sort_values(KEYS + ["started", "grid_for_sort", "constructorId", "number", "resultId"],
                          ascending=[True, True, False, True, True, True, True], kind="mergesort")
    kept = res.drop_duplicates(KEYS, keep="first")
    dropped = res.drop(kept.index).drop(columns=["started", "grid_for_sort"])
    kept = kept.drop(columns=["started", "grid_for_sort"]).copy()
    kept["shared_drive"] = kept["n_raw_rows"] > 1
    kept["podium"] = (kept["best_positionOrder"] <= 3).astype("int8")
    return kept.reset_index(drop=True), dropped

In [ ]:
RES_ONE, DROPPED = keep_one_row_per_driver(RES)
assert not RES_ONE.duplicated(KEYS).any()
assert len(RES_ONE) + len(DROPPED) == len(RES)
print(f"rows: {len(RES)} -> {len(RES_ONE)} (removed {len(DROPPED)}); every (raceId, driverId) pair is still there")

shared = RES_ONE[RES_ONE["shared_drive"]]
own_row_label = (shared["positionOrder"] <= 3).astype(int)
print(f"shared-drive rows (before the Indy 500 and the non-starters are removed): {len(shared)} ({len(shared) / len(RES_ONE):.2%} of all rows), seasons {shared['year'].min()}-{shared['year'].max()}")
print("podium label if we only used the kept row:", int(own_row_label.sum()),
      "| with the driver's best row (used):", int(shared["podium"].sum()),
      "| rows where the two differ:", int((own_row_label != shared["podium"]).sum()))

## 6. Indianapolis 500 and DNQ / non-starters

### Indianapolis 500 (1950-1960)
This race counted for the championship, but almost no F1 drivers entered it, so we **exclude it**.

**Why this choice?** It is a different kind of race (oval, 33 cars, different grid), its drivers are mostly Americans who never raced elsewhere in F1, so it would distort the home-race question (Q2) and the driver form features. We find it by race name and year (not by circuit), because the same Indianapolis circuit hosted a normal F1 US Grand Prix in 2000-2007 that we keep.

In [ ]:
RACES = flag_indy500(CLEAN["races"], CLEAN["circuits"])
print("Indy 500 rounds:", int(RACES["is_indy500"].sum()), "| seasons:", sorted(RACES.loc[RACES.is_indy500, "year"].unique().tolist()))
assert RACES.loc[RACES.year.between(2000, 2007), "is_indy500"].sum() == 0     # the modern US GP stays

RES_ONE = RES_ONE.merge(RACES[["raceId", "is_indy500"]], on="raceId", how="left")
d = RES_ONE.merge(CLEAN["drivers"][["driverId", "nationality"]], on="driverId", how="left")
d["american"] = d["nationality"] == "American"
us_circuits = CLEAN["circuits"].loc[CLEAN["circuits"]["country"].isin(["USA", "United States"]), "circuitId"]
home_rows = d[d["american"] & d["circuitId"].isin(us_circuits)]
print("rows of American drivers at US circuits:", len(home_rows), "| of which Indy 500 rows:", int(home_rows["is_indy500"].sum()),
      f"({home_rows['is_indy500'].mean():.0%})")
print("drivers who only ever appear at the Indy 500:", int(d.groupby("driverId")["is_indy500"].all().sum()))

by_year = d[d.year.between(1950, 1960) & d.american].groupby(["year", "is_indy500"]).size().unstack(fill_value=0)
by_year.plot(kind="bar", stacked=True, figsize=(9, 3.8))
plt.title("American driver entries per season, 1950-1960"); plt.legend(["other F1 races", "Indy 500"]); plt.ylabel("rows"); save_fig("fig_p1_06_american_entries_indy500")
plt.show()

### DNQ / non-starters
In the years with more cars than grid places, slow drivers *failed to (pre)qualify* but still have a row in `results`. Withdrawn drivers are the same. We **separate them from the modeling table**.

**Why this choice?** The model predicts after the grid is set, so a driver who is not on the grid is not part of the question, and their label is always 0. They also make the podium rate artificially low in the big-field years. They never have a podium finish, so removing them does not remove any positives. We find them only with facts known before the race (failed to qualify, withdrew, 107% rule). We do not use the number of laps or the race status of drivers who raced, because dropping rows based on how the race went would use the outcome.

In [ ]:
RES_ONE["is_non_starter"] = find_non_starters(RES_ONE)
ns = RES_ONE[RES_ONE["is_non_starter"] & ~RES_ONE["is_indy500"]]
print("non-starter rows (outside Indy 500):", len(ns), "of", int((~RES_ONE["is_indy500"]).sum()))
print("by decade:", {int(k): int(v) for k, v in ns.groupby(ns["year"] // 10 * 10).size().items()})
print("positionText:", ns["positionText"].value_counts().head(4).to_dict(), "| status:", ns["status"].value_counts().head(3).to_dict())
print("non-starters with a podium finish:", int((ns["podium"] == 1).sum()), "(should be 0)")

RES_FINAL = RES_ONE[~RES_ONE["is_indy500"] & ~RES_ONE["is_non_starter"]].copy()
if EXCLUDE_SHARED_DRIVES:
    RES_FINAL = RES_FINAL[~RES_FINAL["shared_drive"]]
RES_FINAL = RES_FINAL.reset_index(drop=True)
print(f"modeling population: {len(RES_FINAL)} rows | shared-drive policy: {'EXCLUDED' if EXCLUDE_SHARED_DRIVES else 'INCLUDED'}")

dec = RES_ONE[~RES_ONE["is_indy500"]].copy()
dec["decade"] = dec["year"] // 10 * 10
rate = pd.DataFrame({"with non-starters": dec.groupby("decade")["podium"].mean(),
                     "starters only": dec[~dec["is_non_starter"]].groupby("decade")["podium"].mean()})
rate.plot(kind="bar", figsize=(9, 3.8)); plt.title("Podium rate per decade"); plt.ylabel("podium rate"); save_fig("fig_p1_07_podium_rate_per_decade")
plt.show()

## 7. EDA after cleaning and before/after comparison

We repeat the metrics from section 3 on the cleaned data (`RES_FINAL`) and put them next to the raw ones.

**First check: did turning `\N` into `NaN` lose or invent anything?** For every column that had missing values we compare the number of missing cells before (`\N` or blank) with the number of `NaN` after. They must be equal.

In [ ]:
rows = []
for table, df in RAW.items():
    for col in df.columns:
        before = ((df[col] == SENT) | (df[col].str.strip() == "")).sum()
        if before == 0:
            continue
        new_col = TIME_COLS.get(table, {}).get(col, col)
        after = CLEAN[table][new_col].isna().sum()
        rows.append({"table": table, "column": col, "missing before": before, "NaN after": after, "difference": after - before})
check = pd.DataFrame(rows)
print("columns checked:", len(check), "| columns with a difference:", int((check["difference"] != 0).sum()))
display(check[check["difference"] != 0])

top = check.sort_values("missing before", ascending=False).head(12)
x = np.arange(len(top))
plt.figure(figsize=(11, 4))
plt.bar(x - 0.2, top["missing before"], 0.4, label="missing before (\\N or blank)")
plt.bar(x + 0.2, top["NaN after"], 0.4, label="NaN after")
plt.xticks(x, top["table"] + "." + top["column"], rotation=70, ha="right"); plt.yscale("log"); plt.legend()
plt.title("Missing values before and after cleaning"); save_fig("fig_p1_08_missing_before_after")
plt.show()

In [ ]:
# field size and podium rate per season: raw vs cleaned
season_clean = RES_FINAL.groupby("year").agg(rows=("driverId", "size"), races=("raceId", "nunique"), podium_rate=("podium", "mean"))
season_clean["rows_per_race"] = season_clean["rows"] / season_clean["races"]

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0, 0].plot(season_raw.index, season_raw["rows_per_race"], label="raw rows per race")
ax[0, 0].plot(season_clean.index, season_clean["rows_per_race"], label="clean rows per race (starters)")
ax[0, 0].set_title("Field size per season"); ax[0, 0].legend()
ax[0, 1].plot(season_raw.index, season_raw["podium_rate"], label="raw")
ax[0, 1].plot(season_clean.index, season_clean["podium_rate"], label="clean")
ax[0, 1].plot(season_clean.index, 3 / season_clean["rows_per_race"], "k:", label="3 / clean field size")
ax[0, 1].set_title("Podium rate per season"); ax[0, 1].legend()

podium_rows_raw = res_raw.groupby("raceId")["podium"].sum().value_counts().sort_index()
podium_rows_clean = RES_FINAL.groupby("raceId")["podium"].sum().value_counts().sort_index()
pd.concat([podium_rows_raw.rename("raw"), podium_rows_clean.rename("clean")], axis=1).fillna(0).plot(kind="bar", ax=ax[1, 0])
ax[1, 0].set_title("Podium rows per race (3 expected)"); ax[1, 0].set_xlabel("podium rows in a race"); ax[1, 0].set_ylabel("races")

lap = CLEAN["results"]["fastestLapTime_s"].dropna()
ax[1, 1].hist(lap.clip(30, 200), bins=60, color="tab:green")
ax[1, 1].set_title("fastestLapTime_s after conversion (seconds)")
save_fig("fig_p1_09_before_after_field_podium_laptimes")
plt.show()

print("races without exactly 3 podium rows after cleaning:", int((podium_rows_clean.drop(3, errors="ignore")).sum()), "of", int(podium_rows_clean.sum()))
for col, df in [("fastestLapTime_s", CLEAN["results"]), ("q1_s", CLEAN["qualifying"]), ("q3_s", CLEAN["qualifying"])]:
    s = df[col].dropna()
    print(f"{col:17s} n={len(s):6d}  min={s.min():7.2f}  median={s.median():7.2f}  max={s.max():8.2f}")

In [ ]:
# what each cleaning rule removed
def podium_rows(df):
    return int((df["positionOrder"] <= 3).sum())

removed_indy = RES_ONE[RES_ONE["is_indy500"]]
removed_ns = RES_ONE[~RES_ONE["is_indy500"] & RES_ONE["is_non_starter"]]
removed_shared = RES_ONE[~RES_ONE["is_indy500"] & ~RES_ONE["is_non_starter"] & RES_ONE["shared_drive"]] if EXCLUDE_SHARED_DRIVES else RES_ONE.iloc[0:0]
summary = pd.DataFrame([
    ("typed results (start)", len(RES), podium_rows(RES)),
    ("removed: duplicate rows", len(DROPPED), podium_rows(DROPPED)),
    ("removed: Indy 500", len(removed_indy), podium_rows(removed_indy)),
    ("removed: non-starters", len(removed_ns), podium_rows(removed_ns)),
    ("removed: shared drives (if enabled)", len(removed_shared), int(removed_shared["podium"].sum())),
    ("modeling population", len(RES_FINAL), int(RES_FINAL["podium"].sum())),
], columns=["step", "rows", "rows with podium label <= 3"])
display(summary)
print("podium label = 1 in the final table:", int(RES_FINAL["podium"].sum()))

**What the cleaning achieved**
* **Missing values:** the number of missing cells before and after is the same in every column except `q1` (one more `NaN` after: the impossible 1,002.64 s lap set to missing on purpose, see section 4), so nothing else was lost or invented. The lap times are now numbers in a sensible range (tens of seconds, no conversion errors).
* **Duplicates:** the rows per race now match the number of different drivers, and almost every race has exactly 3 podium rows. The few races that do not are shared-drive cases where two drivers were credited with the same podium car.
* **Podium rate:** the change is concentrated in the early decades (duplicates, the Indy 500 and non-starters). Recent seasons, which are our validation and test years, are practically unchanged. The non-starter rule removes no podium rows, and the duplicates are only removed as extra rows, never as a driver-race.

## 8. `load_clean()` and the data dictionary

**Why one function?** If every teammate cleaned the data in their own way (one keeps the Indy 500, another drops DNQs), the row counts and scores would no longer be comparable. `load_clean()` runs all the steps above in the same order, so everyone starts from identical tables.

It returns:
* `model_base`: one row per driver per race with **pre-race columns only** plus the target `podium`.
* `outcomes`: the **post-race** columns of the same rows (status, points, positions, laps...). They are used for the data-engineering questions and must **not** be used as features.
* `tables`: all other cleaned tables. The standings tables are post-race snapshots, so use the previous round before using them as features.
* `meta`: how many rows each step removed.

The function stops with an error if a driver-race has more than one row or if a post-race column ends up in `model_base`. It does not fill in or scale anything, so nothing can be fitted on the test years.

In [ ]:
PRE_RACE_COLS = ["raceId", "driverId", "constructorId", "year", "round", "date", "circuitId", "number",
                 "grid", "pitlane_start", "has_qualifying", "qual_position", "q1_s", "q2_s", "q3_s"]
OUTCOME_COLS = ["raceId", "driverId", "resultId", "positionOrder", "podium", "positionText", "position",
                "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime_s",
                "fastestLapSpeed", "statusId", "status", "n_raw_rows", "shared_drive", "best_positionOrder"]
POST_RACE_COLS = {"positionOrder", "positionText", "position", "points", "laps", "time", "milliseconds",
                  "fastestLap", "rank", "fastestLapTime_s", "fastestLapSpeed", "statusId", "status",
                  "resultId", "n_raw_rows", "shared_drive", "best_positionOrder"}


def load_clean(data_dir=None, exclude_indy500=True, drop_non_starters=True, exclude_shared_drives=False):
    """Raw CSVs -> clean tables. Everybody on the team should start from this function.
    model_base = pre-race columns + target `podium` (one row per driver per race)
    outcomes   = post-race columns (status, points, ...) - never use these as features."""
    random.seed(SEED)
    np.random.seed(SEED)
    tables = clean_tables(load_raw_tables(data_dir or find_data_dir()))
    tables["races"] = flag_indy500(tables["races"], tables["circuits"])

    res = tables["results"].merge(tables["races"][["raceId", "year", "round", "date", "circuitId", "is_indy500"]],
                                  on="raceId", how="inner")
    res = res.merge(tables["status"], on="statusId", how="left")
    meta = {"raw_result_rows": len(tables["results"]), "after_race_join": len(res)}

    if exclude_indy500:
        meta["indy500_rows_removed"] = int(res["is_indy500"].sum())
        res = res[~res["is_indy500"]]
    res, dropped = keep_one_row_per_driver(res)
    meta["duplicate_rows_removed"] = len(dropped)
    res["is_non_starter"] = find_non_starters(res)
    if drop_non_starters:
        meta["non_starter_rows_removed"] = int(res["is_non_starter"].sum())
        res = res[~res["is_non_starter"]]
    if exclude_shared_drives:
        meta["shared_drive_rows_removed"] = int(res["shared_drive"].sum())
        res = res[~res["shared_drive"]]
    res = res.reset_index(drop=True)
    meta["final_rows"] = len(res)

    quali = tables["qualifying"].sort_values("qualifyId")
    meta["qualifying_duplicates_dropped"] = int(quali.duplicated(KEYS, keep="last").sum())
    quali = quali.drop_duplicates(KEYS, keep="last").rename(columns={"position": "qual_position"})
    quali = quali[KEYS + ["qual_position", "q1_s", "q2_s", "q3_s"]]

    res["pitlane_start"] = (res["grid"] == 0).fillna(False).astype(bool)
    base = res.merge(quali, on=KEYS, how="left", indicator=True)
    base["has_qualifying"] = base["_merge"] == "both"

    model_base = base[PRE_RACE_COLS].copy()
    model_base["podium"] = base["podium"].values
    outcomes = base[OUTCOME_COLS].copy()

    # safety checks
    assert not model_base.duplicated(KEYS).any(), "more than one row for a driver in a race"
    assert model_base[KEYS].notna().all().all(), "missing key"
    assert not (POST_RACE_COLS & set(model_base.columns)), "post-race column in the base table"
    return {"model_base": model_base, "outcomes": outcomes, "tables": tables, "meta": meta}

In [ ]:
CL = load_clean(DATA_DIR, exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
model_base, outcomes = CL["model_base"], CL["outcomes"]
print(CL["meta"])
display(model_base.head())

# same result as the step-by-step version above
assert len(model_base) == len(RES_FINAL) and model_base["podium"].sum() == RES_FINAL["podium"].sum()
print(f"{len(model_base)} rows, {model_base['raceId'].nunique()} races, seasons {model_base['year'].min()}-{model_base['year'].max()}, podium rate {model_base['podium'].mean():.1%}")

split = np.select([model_base["year"] <= 2019, model_base["year"].between(2020, 2021)], ["train (<=2019)", "validation (2020-21)"], "test (>=2022)")
display(model_base.groupby(split)["podium"].agg(rows="size", podium_rate="mean").round(3))

**Counts for the final table.** Shared-drive rows, how many podium labels the best-row rule changed, and how much of each split has qualifying data (these numbers are used in the report).


In [ ]:
sd = CL["outcomes"][CL["outcomes"]["shared_drive"]].merge(model_base[["raceId", "driverId", "year"]], on=KEYS)
print(f"shared-drive rows in the final table: {len(sd)} ({len(sd) / len(model_base):.2%}), seasons {sd['year'].min()}-{sd['year'].max()}, after 1957: {int((sd['year'] > 1957).sum())}")

changed = CL["outcomes"][(CL["outcomes"]["positionOrder"] <= 3).astype(int) != CL["outcomes"]["podium"]]
print("podium labels that differ from positionOrder <= 3:", len(changed), "| all of them shared-drive rows:", bool(changed["shared_drive"].all()))

split_name = np.select([model_base["year"] <= 2019, model_base["year"].between(2020, 2021)], ["train", "validation"], "test")
print("share of rows with qualifying data by split:", model_base.groupby(split_name)["has_qualifying"].mean().round(3).to_dict())
print("share of rows with qualifying data: before 1994 =", round(model_base.loc[model_base["year"] < 1994, "has_qualifying"].mean(), 3),
      "| 2003 onwards =", round(model_base.loc[model_base["year"] >= 2003, "has_qualifying"].mean(), 3))

In [ ]:
descriptions = {
    "raceId": ("key", "race id (links to races)"), "driverId": ("key", "driver id (links to drivers)"),
    "constructorId": ("pre-race", "team of the kept row"), "year": ("pre-race", "season"), "round": ("pre-race", "round number in the season"),
    "date": ("pre-race", "race date"), "circuitId": ("pre-race", "circuit (links to circuits)"), "number": ("pre-race", "car number"),
    "grid": ("pre-race", "starting slot after penalties; 0 = no grid slot (pit-lane start for drivers who started)"),
    "pitlane_start": ("pre-race", "True if grid = 0 (non-starters were already removed)"),
    "has_qualifying": ("pre-race", "True if the driver has a qualifying row"),
    "qual_position": ("pre-race", "qualifying position (NaN when there is no qualifying data)"),
    "q1_s": ("pre-race", "Q1 best lap in seconds"), "q2_s": ("pre-race", "Q2 best lap in seconds (NaN if knocked out in Q1 or before 2006)"),
    "q3_s": ("pre-race", "Q3 best lap in seconds (NaN if knocked out earlier or before 2006)"),
    "podium": ("TARGET", "1 if the driver's best finish in the race is top 3"),
    "resultId": ("outcome", "id of the kept result row"), "positionOrder": ("outcome", "final ranking of every entrant"),
    "positionText": ("outcome", "position or letter code (R, D, E, W, F, N)"), "position": ("outcome", "classified position"),
    "points": ("outcome", "championship points"), "laps": ("outcome", "laps completed"), "time": ("outcome", "race time text"),
    "milliseconds": ("outcome", "race time in ms"), "fastestLap": ("outcome", "lap of the fastest lap"), "rank": ("outcome", "fastest-lap rank"),
    "fastestLapTime_s": ("outcome", "fastest lap in seconds"), "fastestLapSpeed": ("outcome", "fastest lap speed"),
    "statusId": ("outcome", "why the race ended (links to status)"), "status": ("outcome", "status text"),
    "n_raw_rows": ("audit", "number of raw result rows of this driver in this race"),
    "shared_drive": ("audit", "True if the driver had more than one raw row (car swap, 1950-1964)"),
    "best_positionOrder": ("audit", "best positionOrder over the driver's rows (used for the label)"),
}
dictionary = pd.DataFrame([(tbl, col, str(df[col].dtype), *descriptions[col], round(100 * df[col].isna().mean(), 1))
                           for tbl, df in [("model_base", model_base), ("outcomes", outcomes)] for col in df.columns],
                          columns=["table", "column", "type", "role", "meaning", "% missing"])
display(dictionary)

dictionary.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False)
model_base.to_csv(OUTPUT_DIR / "model_base.csv", index=False)
outcomes.to_csv(OUTPUT_DIR / "outcomes.csv", index=False)

### Notes for the team
* Features come from `model_base` (and from earlier races). Columns marked `outcome` or `audit` are never features.
* Standings tables are after-the-race: shift them to the previous round. Rolling form uses `shift(1)`.
* Fit imputers, scalers and encoders on the training years only (<= 2019).
* Shared-drive policy is controlled by `EXCLUDE_SHARED_DRIVES` at the top of P1. The merged notebook uses this same setting in every section; `True` removes flagged rows and `False` keeps them flagged.

---
# >>> Section from `notebooks/p2_de.ipynb`


## Part 2: Data-engineering questions (P2)

Three questions answered with multi-table joins. Each one ends with the answer, the plots, and short notes on
**grain** (what one row is), **missingness** (what is empty and why) and the **pre-race cutoff** (what is known before the race starts).

In [ ]:
import pandas as pd               # tables
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # heatmap (backup section)

clean = load_clean(exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
base = clean["model_base"]    # pre-race columns + the target `podium`
outcomes = clean["outcomes"]  # post-race columns (status, points...). Never features
tables = clean["tables"]      # all other cleaned tables (races, circuits, drivers...)

### Q1: Which circuits convert a front-row start into a podium most often?

**In plain words:** when a driver starts 1st or 2nd (the front row), how often do they finish in the top 3, at each circuit?
Does that ranking change after 2014? We answer it twice: once with **grid** (where the car really started) and once with
**qualifying position** (where it qualified).

**Why 2014?** In 2014 F1 switched to **hybrid engines** (a small turbo engine plus batteries and electric motors).
One team (Mercedes) became much faster than the rest, and every engine part became limited, which caused many more grid penalties.

**Which years count as "before 2014"?** We make three comparisons, each one for grid and for qualifying (6 in total):

| Before | After | Why |
|---|---|---|
| **2004-2013** | 2014-2024 | **Main, the fairest:** similar length (10 vs 11 seasons), grid penalties exist in both, full qualifying data, similar car reliability |
| 1994-2013 | 2014-2024 | from the first season with qualifying data |
| 1950-2013 | 2014-2024 | all of F1 history |

**Tables and joins (multi-hop):**

```
results ──raceId──► races ──circuitId──► circuits        (which circuit was the race at?)
   │
   └──(raceId, driverId)──► qualifying                     (where did the driver qualify?)
```

- `results ⋈ races ⋈ qualifying` is already done inside `load_clean()` → `model_base`
  (`grid`, `podium`, `year`, `circuitId`, `qual_position`). `load_clean()` also used `status` to remove drivers who did not start,
  removed the Indianapolis 500 and kept one row per driver per race.
- Here we add the last hop: `model_base ⋈ circuits` on `circuitId` (circuit name and country).
- `drivers` and `races` are joined only to show names in an example table; `outcomes` only to explain the result (never as a feature).
- Not used: `sprint_results` (a different, shorter race with its own grid), standings, lap times, pit stops.

In [ ]:
# 1) take only the 3 columns we need from the circuits table
circuits = tables["circuits"][["circuitId", "name", "country"]]
# 2) rename "name" to "circuit" so the column name is clear
circuits = circuits.rename(columns={"name": "circuit"})

# 3) join: every row of base gets the name and country of its circuit
q1 = base.merge(circuits, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks: the join must not add or lose rows, and every race must find its circuit
assert len(q1) == len(base), "the join changed the number of rows"
assert q1["circuit"].notna().all(), "a race has no circuit name"

print("rows (one per driver per race):", len(q1))
print("seasons:", q1["year"].min(), "-", q1["year"].max(), "| races:", q1["raceId"].nunique(), "| circuits:", q1["circuit"].nunique())

#### Grid vs qualifying position

- **Qualifying position** = the result of Saturday's timed session (fastest lap = 1st).
- **Grid** = where the car really started on Sunday.
- **Before 2004 they were the same:** the grid was simply the qualifying order.
- **From 2004** the rules limit engines (and from 2008 gearboxes). A driver who changes one early is moved back on the grid
  (a **penalty**), and the drivers behind move up. A car changed after qualifying starts from the **pit lane** (`grid = 0`).
- **After 2014** every part of the hybrid engine is limited, so penalties became about **twice as common**.

The chart shows this per season (from 1994, the first season with qualifying data, so we can see the "before").

In [ ]:
# keep only rows that have a qualifying position (1994 and later)
with_quali = q1[q1["has_qualifying"]].copy()

# places lost between qualifying and the start (positive = started further back)
with_quali["places_lost"] = with_quali["grid"] - with_quali["qual_position"]

# pushed back = lost 3 or more places, or started from the pit lane (grid 0)
with_quali["pushed_back"] = (with_quali["places_lost"] >= 3) | (with_quali["grid"] == 0)

# per season: number of races, number of pushed-back drivers, and drivers per race
per_year = with_quali.groupby("year").agg(races=("raceId", "nunique"), pushed_back=("pushed_back", "sum"))
per_year["per_race"] = per_year["pushed_back"] / per_year["races"]

plt.figure(figsize=(11, 4))
plt.bar(per_year.index, per_year["per_race"], color="tab:red", label="drivers pushed back per race")
plt.axvline(2003.5, color="grey", linestyle="--", label="2004: first engine rule")
plt.axvline(2013.5, color="black", linestyle="--", label="2014: hybrid engines")
plt.title("Drivers who started 3+ places behind their qualifying position (or from the pit lane)")
plt.xlabel("season")
plt.ylabel("drivers per race")
plt.legend()
plt.show()

**Why 3 places and not 1?** The table counts, per race, the drivers who started 1, 2, 3-4 or 5+ places behind their
qualifying position. Losing **1** place happens about equally often in every span, even before 2004 when there were no
penalties, so it is not a penalty (small recording differences, or someone moved ahead after qualifying). Losing **3 or more**
is zero before 2004 and then grows with the penalty rules. 3 places is also the smallest common real penalty
(for example, blocking another driver in qualifying).

In [ ]:
# three time spans: before penalties, first engine/gearbox rules, hybrid era
with_quali["span"] = "1994-2003"
with_quali.loc[with_quali["year"] >= 2004, "span"] = "2004-13"
with_quali.loc[with_quali["year"] >= 2014, "span"] = "2014-24"

# only drivers who started behind their qualifying position (pit-lane starts have a negative value, so they are left out)
lost = with_quali[with_quali["places_lost"] >= 1].copy()

# put every driver in a group by how many places they lost
lost["places"] = "1"
lost.loc[lost["places_lost"] == 2, "places"] = "2"
lost.loc[lost["places_lost"] >= 3, "places"] = "3-4"
lost.loc[lost["places_lost"] >= 5, "places"] = "5+"

# count the drivers per span and group, then divide each row by the number of races in that span
counts = pd.crosstab(lost["span"], lost["places"])
races_per_span = with_quali.groupby("span")["raceId"].nunique()
places_lost_per_race = counts.div(races_per_span, axis=0)

print("drivers per race who started 1, 2, 3-4 or 5+ places behind their qualifying position:")
places_lost_per_race.round(2)

On the **front row**, every driver who qualified 1st/2nd or started 1st/2nd is one of three cases:
**same** (qualified and started on the front row), **penalty** (qualified on the front row but started further back),
or **moved up** (qualified 3rd or lower but started on the front row because someone ahead was penalised).

In [ ]:
# True / False for every row: qualified 1st or 2nd? started 1st or 2nd?
qualified_front = (with_quali["qual_position"] == 1) | (with_quali["qual_position"] == 2)
started_front = (with_quali["grid"] == 1) | (with_quali["grid"] == 2)

# give every row a readable name for its case
SAME = "qualified 1st/2nd and started 1st/2nd (same)"
PENALTY = "qualified 1st/2nd but started further back (penalty)"
MOVED_UP = "qualified 3rd or lower but started 1st/2nd (moved up)"
with_quali["case"] = "not on the front row"
with_quali.loc[qualified_front & started_front, "case"] = SAME
with_quali.loc[qualified_front & ~started_front, "case"] = PENALTY
with_quali.loc[~qualified_front & started_front, "case"] = MOVED_UP

# only drivers on the front row in at least one of the two, counted per span
front_cases = with_quali[qualified_front | started_front]
case_table = pd.crosstab(front_cases["case"], front_cases["span"])
case_table

In [ ]:
# a few real examples, with driver and race names
drivers = tables["drivers"][["driverId", "forename", "surname"]]
races = tables["races"][["raceId", "name"]].rename(columns={"name": "race"})

examples = front_cases[front_cases["case"] != SAME]
examples = examples.merge(drivers, on="driverId", how="left")
examples = examples.merge(races, on="raceId", how="left")
examples = examples.sort_values("year", ascending=False)
examples[["year", "race", "forename", "qual_position", "grid", "case"]].head(6)

In [ ]:
# missing data: % of drivers with a known qualifying position, per season
coverage = q1.groupby("year")["has_qualifying"].mean() * 100

plt.figure(figsize=(11, 3.5))
plt.bar(coverage.index, coverage.values, color="tab:blue", label="% of drivers with a known qualifying position")
plt.title("Qualifying data per season: none before 1994, patchy 1996-2002, complete from 2003")
plt.xlabel("season")
plt.ylabel("% of drivers")
plt.legend(loc="upper left")
plt.show()

#### Filling the qualifying gaps with the grid (only before 2004)

Qualifying positions are missing before 1994 and for many races in 1996-2002. Qualifying **did** happen in those years and
**decided the grid**; the dataset just does not record it. Before 2004 there were no grid penalties, so the grid is the
qualifying order. The check below proves it on the years where we have both. So we make a new column `qual_or_grid`:
the qualifying position, and where it is missing **before 2004**, the grid. Missing values from 2004 on are **not** filled,
because from then on the grid can differ from qualifying (penalties).

In [ ]:
# check: in 1994-2003 (before penalties), is the grid the same as the qualifying position?
before_penalties = q1[q1["has_qualifying"] & (q1["year"] <= 2003)]
same_position = before_penalties["grid"] == before_penalties["qual_position"]
print("1994-2003: grid = qualifying position for", round(same_position.mean() * 100, 1), "% of drivers")

qualified_front_old = (before_penalties["qual_position"] == 1) | (before_penalties["qual_position"] == 2)
started_front_old = (before_penalties["grid"] == 1) | (before_penalties["grid"] == 2)
print("front row: qualified 1st/2nd:", qualified_front_old.sum(),
      "| of these, started 1st/2nd:", (qualified_front_old & started_front_old).sum())

# new column: the qualifying position; where it is missing before 2004, use the grid instead
q1["qual_or_grid"] = q1["qual_position"]
fill = q1["qual_or_grid"].isna() & (q1["year"] < 2004)
q1.loc[fill, "qual_or_grid"] = q1.loc[fill, "grid"]

print("rows filled with the grid:", fill.sum())
print("still missing (2004 and later, not filled):", q1["qual_or_grid"].isna().sum())

#### Helper functions (our toolbox)

| Function | What it does |
|---|---|
| `add_era(df, start_year)` | keeps the seasons from `start_year` on and labels them "start_year-2013" or "2014-24" |
| `front_row_rate(df, position_column)` | per circuit and era: **starts** (drivers in position 1 or 2), **podiums**, **rate** = podiums ÷ starts, **enough_starts** (at least `MIN_STARTS`) |
| `overall_rate(df, position_column)` | the same, but all circuits together |
| `plot_top10(ax, table, era, title)` | bar chart of the 10 circuits with the highest rate in one era |
| `compare_eras(table, before)` | circuits with enough starts in **both** eras, and the **Spearman** correlation of their two rankings |
| `slope_chart(ax, rates, spearman, title)` | the rank-change chart: one line per circuit from its rank before to its rank after |

**Spearman** compares two rankings: **1** = exactly the same order, **0** = the old order says nothing about the new order,
**−1** = the order is reversed.

In [ ]:
MIN_STARTS = 10   # a circuit needs at least 10 front-row starts (about 5 races) in an era to be ranked


def add_era(df, start_year):
    """Keep the seasons from start_year on and label each row with its era."""
    part = df[df["year"] >= start_year].copy()
    part["era"] = str(start_year) + "-2013"
    part.loc[part["year"] >= 2014, "era"] = "2014-24"
    return part


def front_row_rate(df, position_column):
    """For every circuit and era: front-row starts, podiums and the rate."""
    # keep only drivers in position 1 or 2 (a missing position is not front row)
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    # count the starts and add up the podiums (podium is 1 or 0) per circuit and era
    table = front.groupby(["circuit", "era"])["podium"].agg(starts="count", podiums="sum")
    table = table.reset_index()
    # rate = podiums / starts
    table["rate"] = table["podiums"] / table["starts"]
    # enough starts to be ranked?
    table["enough_starts"] = table["starts"] >= MIN_STARTS
    return table


def overall_rate(df, position_column):
    """Front-row starts, podiums and rate per era, all circuits together."""
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    table = front.groupby("era")["podium"].agg(starts="count", podiums="sum")
    table["rate"] = table["podiums"] / table["starts"]
    return table


def plot_top10(ax, table, era, title):
    """Bar chart of the 10 circuits with the highest rate in one era (only circuits with enough starts)."""
    rows = table[(table["era"] == era) & table["enough_starts"]]
    # sort from low to high and keep the last 10, so the best circuit is drawn at the top
    top10 = rows.sort_values(["rate", "starts"]).tail(10)
    ax.barh(top10["circuit"], top10["rate"] * 100, color="tab:green", label="% of front-row starters on the podium")
    # write "podiums of starts" on each bar
    for i in range(len(top10)):
        ax.text(1, i, str(top10["podiums"].iloc[i]) + " of " + str(top10["starts"].iloc[i]),
                va="center", color="white", fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_title(title)
    ax.set_xlabel("% of front-row starters who finished on the podium")
    ax.legend(loc="lower right", fontsize=8)


def compare_eras(table, before, after="2014-24", min_starts=MIN_STARTS):
    """Rates of the circuits that have enough starts in both eras, and the Spearman correlation of their ranks."""
    rates = table.pivot(index="circuit", columns="era", values="rate")[[before, after]]
    starts = table.pivot(index="circuit", columns="era", values="starts")[[before, after]]
    in_both = (starts[before] >= min_starts) & (starts[after] >= min_starts)
    rates = rates[in_both]
    ranks = rates.rank(ascending=False)   # 1 = highest rate
    spearman = ranks[before].corr(ranks[after])
    return rates, spearman


def slope_chart(ax, rates, spearman, title):
    """Left column = order before, right column = order after, one line per circuit."""
    before = rates.columns[0]
    after = rates.columns[1]
    # position of every circuit in each column (0 = top)
    left = pd.Series(range(len(rates)), index=rates.sort_values(before, ascending=False).index)
    right = pd.Series(range(len(rates)), index=rates.sort_values(after, ascending=False).index)
    for circuit in rates.index:
        if right[circuit] < left[circuit]:
            color = "tab:green"
        elif right[circuit] > left[circuit]:
            color = "tab:red"
        else:
            color = "grey"
        ax.plot([0, 1], [left[circuit], right[circuit]], marker="o", color=color)
        ax.text(-0.03, left[circuit], circuit + " (" + str(round(rates.loc[circuit, before] * 100)) + "%)",
                ha="right", va="center", fontsize=8)
        ax.text(1.03, right[circuit], "(" + str(round(rates.loc[circuit, after] * 100)) + "%) " + circuit,
                ha="left", va="center", fontsize=8)
    ax.plot([], [], color="tab:green", label="better rank after")
    ax.plot([], [], color="tab:red", label="worse rank after")
    ax.plot([], [], color="grey", label="same rank")
    ax.invert_yaxis()                       # best at the top
    ax.set_xlim(-1.2, 2.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([before, after])
    ax.set_yticks([])
    ax.set_ylabel("rank (best at the top)")
    ax.set_title(title + "\n" + str(len(rates)) + " circuits, Spearman = " + str(round(spearman, 2)))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=8)

#### Minimum number of starts

Some circuits hosted only a few races in an era. A circuit with 2 front-row starts and 2 podiums shows 100%, but that is
2 cars, not a pattern, and it would wrongly top the ranking. So a circuit is ranked only if it has **at least 10 front-row
starts in that era (about 5 races)**: then one car can change the rate by at most 10 points. A later cell checks that
choosing 6, 8 or 15 instead does not change the answer.

In [ ]:
all_seasons = add_era(q1, 1950)
all_grid_table = front_row_rate(all_seasons, "grid")

# plt.figure(figsize=(8, 4))
# plt.hist(all_grid_table["starts"], bins=30, color="tab:blue", edgecolor="white", label="circuit-eras")
# plt.axvline(MIN_STARTS, color="red", linestyle="--", label="minimum = 10 starts")
# plt.title("Front-row starts per circuit and era (grid 1st/2nd, 1950-2013 and 2014-24)")
# plt.xlabel("front-row starts")
# plt.ylabel("number of circuit-eras")
# plt.legend()
# plt.show()

small = all_grid_table[~all_grid_table["enough_starts"]]
print("circuit-eras with fewer than", MIN_STARTS, "starts:", len(small), "of", len(all_grid_table))
print("examples (these are left out of the ranking):")
small.sort_values("rate", ascending=False).head(5)

#### Main comparison: 2004-2013 vs 2014-2024

The fairest comparison: grid penalties exist in both eras (so grid and qualifying mean the same thing on both sides),
the eras have a similar length, and qualifying data is complete. Top row = grid, bottom row = qualifying.

In [ ]:
fair = add_era(q1, 2004)
grid_fair = front_row_rate(fair, "grid")
qual_fair = front_row_rate(fair, "qual_or_grid")

print("Front row -> podium, all circuits together:")
print("grid:")
display(overall_rate(fair, "grid").loc[["2004-2013", "2014-24"]].round(3))
print("qualifying:")
display(overall_rate(fair, "qual_or_grid").loc[["2004-2013", "2014-24"]].round(3))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plot_top10(axes[0, 0], grid_fair, "2004-2013", "Top 10 circuits, started 1st/2nd (grid), 2004-2013")
plot_top10(axes[0, 1], grid_fair, "2014-24", "Top 10 circuits, started 1st/2nd (grid), 2014-2024")
plot_top10(axes[1, 0], qual_fair, "2004-2013", "Top 10 circuits, qualified 1st/2nd, 2004-2013")
plot_top10(axes[1, 1], qual_fair, "2014-24", "Top 10 circuits, qualified 1st/2nd, 2014-2024")
plt.tight_layout()
plt.show()

**Does the ranking change?** We take the circuits with at least 10 front-row starts in **both** eras and rank them in each era.
Each line is one circuit: going up (green) = better rank after 2014, going down (red) = worse.

In [ ]:
grid_rates, grid_spearman = compare_eras(grid_fair, "2004-2013")
qual_rates, qual_spearman = compare_eras(qual_fair, "2004-2013")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
slope_chart(axes[0], grid_rates, grid_spearman, "Rank change 2004-13 -> 2014-24, started 1st/2nd (grid)")
slope_chart(axes[1], qual_rates, qual_spearman, "Rank change 2004-13 -> 2014-24, qualified 1st/2nd")
plt.tight_layout()
plt.show()

#### All six comparisons

The same calculation for the three "before" windows (1950-2013, 1994-2013, 2004-2013), each with grid and with
qualifying (`qual_or_grid`), and each compared with **2014-24**.

In [ ]:
# compute the 6 comparisons: 3 "before" windows x 2 positions
rows = []
for start_year in [1950, 1994, 2004]:
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    for position_column in ["grid", "qual_or_grid"]:
        overall = overall_rate(data, position_column)
        rates, spearman = compare_eras(front_row_rate(data, position_column), before)
        rows.append([before, position_column, overall.loc[before, "rate"] * 100,
                     overall.loc["2014-24", "rate"] * 100, spearman])

six = pd.DataFrame(rows, columns=["before", "position", "rate before", "rate 2014-24", "spearman"])
six["position"] = six["position"].replace("qual_or_grid", "qualifying")   # nicer name for the charts

**1) The podium rate: each "before" window next to 2014-24.** One chart per window. In each chart the grey bar is the
window before 2014 and the green bar is 2014-24, for grid and for qualifying.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, before in zip(axes, ["1950-2013", "1994-2013", "2004-2013"]):
    # the 2 rows of this window (grid and qualifying), with the position as row name
    window = six[six["before"] == before].set_index("position")
    # keep the two rates and name them after their years
    pair = window[["rate before", "rate 2014-24"]]
    pair.columns = [before, "2014-24"]

    pair.plot(kind="bar", ax=ax, color=["tab:grey", "tab:green"])
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.0f", fontsize=9)   # write the value on top of each bar

    # difference between the two grid bars, using the same rounded numbers that are written on the bars
    gain = round(pair.loc["grid", "2014-24"]) - round(pair.loc["grid", before])
    ax.set_title(before + " vs 2014-24\n(grid: +" + str(gain) + " points after 2014)")
    ax.set_xlabel("front row by")
    ax.set_ylabel("% of front-row starters on the podium")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

**2) The circuit ranking: how similar is each "before" window to 2014-24?** Spearman: 1 = same order, 0 = unrelated.

In [ ]:
spearman_table = six.pivot(index="before", columns="position", values="spearman")

ax = spearman_table.plot(kind="bar", color=["tab:blue", "tab:orange"], figsize=(8, 4.5))
ax.axhline(0, color="black", linewidth=1)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", fontsize=9)   # write the value on top of each bar
ax.set_title("How similar is the circuit ranking to 2014-24?")
ax.set_xlabel('"before 2014" window')
ax.set_ylabel("Spearman (1 = same order, 0 = unrelated)")
ax.set_ylim(-0.5, 1)
ax.tick_params(axis="x", rotation=0)
plt.show()

#### Why did the front row convert more often?

Every front-row starter ends in one of three ways: **podium**, **did not finish** (the car broke down, crashed or was
disqualified), or **finished 4th or lower**. We also count how often both front-row cars came from the same team.
`outcomes` is used here only to explain the result, never as a model feature.

In [ ]:
# add positionText from outcomes: a number ("1", "2", ...) if the driver was classified, a letter (R = retired...) if not
why = q1.merge(outcomes[["raceId", "driverId", "positionText"]], on=["raceId", "driverId"], how="left")
why = why[(why["grid"] == 1) | (why["grid"] == 2)].copy()

# three spans of years
why["span"] = "1950-2003"
why.loc[why["year"] >= 2004, "span"] = "2004-13"
why.loc[why["year"] >= 2014, "span"] = "2014-24"

# put every front-row starter into one of three groups
why["result"] = "finished 4th or lower"
why.loc[~why["positionText"].str.isnumeric(), "result"] = "did not finish"
why.loc[why["podium"] == 1, "result"] = "podium"

# % of each group per span (normalize="index" makes every row add up to 100%)
result_share = pd.crosstab(why["span"], why["result"], normalize="index") * 100
result_share = result_share[["podium", "finished 4th or lower", "did not finish"]]
display(result_share.round(1))

result_share.plot(kind="bar", stacked=True, color=["tab:green", "tab:orange", "tab:red"], figsize=(7, 4))
plt.title("What happened to the front-row starters (grid 1st/2nd)?")
plt.xlabel("years")
plt.ylabel("% of front-row starters")
plt.xticks(rotation=0)
plt.legend(title="result", bbox_to_anchor=(1, 1))
plt.show()

# how often were both front-row cars from the same team?
per_race = why.groupby(["raceId", "span"]).agg(cars=("constructorId", "count"), teams=("constructorId", "nunique"))
per_race = per_race.reset_index()
per_race = per_race[per_race["cars"] == 2]
per_race["same_team"] = per_race["teams"] == 1
print("% of races where both front-row cars were from the same team:")
print((per_race.groupby("span")["same_team"].mean() * 100).round(1))

#### Check: does our choice of 10 starts change the answer?

The minimum of 10 starts is our own decision. We repeat the main comparison (2004-13 vs 2014-24) with 6, 8 and 15
to see whether the answer (how similar the rankings are, and which circuits are on top after 2014) stays the same.

In [ ]:
rows = []
for min_starts in [6, 8, 10, 16]:
    for position_column, table in [("grid", grid_fair), ("qual_or_grid", qual_fair)]:
        rates, spearman = compare_eras(table, "2004-2013", min_starts=min_starts)
        top5 = rates.sort_values("2014-24", ascending=False).head(5).index
        rows.append([position_column, min_starts, len(rates), round(spearman, 2), ", ".join(top5)])

check = pd.DataFrame(rows, columns=["position", "min starts", "circuits", "spearman", "top 5 in 2014-24"])
with pd.option_context("display.max_colwidth", None):
    display(check)

#### Answer to Q1

**1. Best circuits in the hybrid era (2014-24): Monaco and Suzuka.** About 9 in 10 front-row starters finished on the podium
there: Monaco 17 of 19, Suzuka 16 of 18 (89%). They are the **top 2 in all six comparisons**, with grid and with qualifying.
Next come Yas Marina, Barcelona and Spa (82-86%).

**2. Main comparison, 2004-13 vs 2014-24 (the fairest).** The front row converts more often after 2014:

| Front row by | 2004-13 | 2014-24 |
|---|---|---|
| grid (started 1st/2nd) | 69% (254 of 366) | **76%** (345 of 453) |
| qualifying (qualified 1st/2nd) | 69% (251 of 366) | **74%** (336 of 454) |

The ranking is **only partly the same**. Spearman is **0.28 (grid)** and **0.45 (qualifying)**, for 14 circuits.
- **Moved up:** Yas Marina (60% → 86%), Barcelona, Interlagos, Canada.
- **Moved down:** Monza, Silverstone, Bahrain, Hungaroring.
- Monza and Silverstone barely changed their own rate (75% → 73%, 75% → 75% on grid). They dropped because other circuits rose.
- With a minimum of 6, 8 or 15 starts, Spearman stays positive (0.09-0.39 grid, 0.24-0.54 qualifying), and Monaco and Suzuka
  stay the top 2.

**3. The further back "before" goes, the more different the ranking looks.**

| Before | Spearman grid | Spearman qualifying |
|---|---|---|
| 2004-2013 | 0.28 | 0.45 |
| 1994-2013 | -0.04 | -0.00 |
| 1950-2013 | -0.08 | -0.11 |

Compared with the last decade, part of the order is kept. Compared with all of F1 history, the old order says nothing about the new one.

**4. Why the front row converts more often.**
- Over the long run, front-row cars stopped breaking down: **31%** did not finish in 1950-2003, **10%** in 2004-13, **7%** in 2014-24.
- Between 2004-13 and 2014-24, the share that finished but lost the podium on track also fell (20% → 17%).
- Both front-row cars came from the same team more often (32% → 44% of races). One dominant team kept both spots.

**5. Grid vs qualifying.**
- In 1994-2003 the grid was the qualifying order. 98.5% of drivers started where they qualified, and all 164 front-row
  qualifiers started on the front row.
- Grid penalties start in 2004. Drivers per race who started 3+ places behind their qualifying spot: **0.01** (1994-2003),
  **0.76** (2004-13), **1.05** (2014-24)
  (the 3-4 and 5+ columns of the places-lost table added together).
- On the front row, 0 → 9 → 28 qualifiers were pushed back. Example: Verstappen, Belgium 2024, qualified 1st and started 11th.
- This is why the grid and qualifying answers are identical before 2004 and differ a little after.
- Qualifying gives a slightly lower rate after 2014 (74% vs 76%), because a fast car pushed off the front row still counts
  in qualifying.

**Grain**
- Input: one row per driver per race (`model_base`).
- Front-row subset: 2 rows per race (position 1 and 2).
- Result tables: one row per circuit and era.
- The circuits join is many-to-one, and the row count is checked to stay the same.

**Missingness**
- `qual_position` is empty before 1994 and patchy in 1996-2002 (see the coverage chart).
- `qual_or_grid` fills these gaps with the grid, **only before 2004**, when grid = qualifying order (checked: 164 of 164
  front-row qualifiers started on the front row).
- 21 missing values from 2004 on are not filled.
- `grid` has no gaps. `grid = 0` (pit-lane start) is not counted as front row.

**Pre-race cutoff**
- `grid` and `qual_position` are both known before the race starts.
- `podium` is the outcome we measure, not an input.
- `outcomes` (positionText) is used only to explain why the rate rose, never as an input.

**Limitations**
- Some circuits hosted only a few races in an era (fewer than 10 front-row starts), so they are left out of the ranking.
- One `circuitId` covers all layout changes of a track (e.g. Silverstone, Spa).
- Circuits come and go. Magny-Cours and Istanbul topped 2004-13 but are not on the calendar after 2014.
- Shared drives (two drivers, one car) exist only before 1965 and are counted once, with the best finish.

#### Backup: longer "before" windows and four periods

Extra detail, not needed for the answer above.

In [ ]:
# rank change for the two longer windows: 1994-2013 and 1950-2013 (top = grid, bottom = qualifying)
fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for column, start_year in enumerate([1994, 1950]):
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    rates, spearman = compare_eras(front_row_rate(data, "grid"), before)
    slope_chart(axes[0, column], rates, spearman, "Rank change " + before + " -> 2014-24, grid")
    rates, spearman = compare_eras(front_row_rate(data, "qual_or_grid"), before)
    slope_chart(axes[1, column], rates, spearman, "Rank change " + before + " -> 2014-24, qualifying")
plt.tight_layout()
plt.show()

In [ ]:
# four periods: the rate for all circuits together (grid)
q1["period"] = "1950-93"
q1.loc[q1["year"] >= 1994, "period"] = "1994-2003"
q1.loc[q1["year"] >= 2004, "period"] = "2004-13"
q1.loc[q1["year"] >= 2014, "period"] = "2014-24"

grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
by_period = grid_front.groupby("period")["podium"].agg(starts="count", podiums="sum")
by_period["rate"] = by_period["podiums"] / by_period["starts"]
display(by_period.round(3))

plt.figure(figsize=(7, 4))
plt.bar(by_period.index, by_period["rate"] * 100, color="tab:blue", label="all circuits together")
plt.ylim(0, 100)
plt.title("Front row (grid 1st/2nd) -> podium, per period")
plt.xlabel("period")
plt.ylabel("% of front-row starters on the podium")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# heatmap: rate per circuit and period, only where the circuit had at least 10 front-row starts
grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
period_table = grid_front.groupby(["circuit", "period"])["podium"].agg(starts="count", podiums="sum").reset_index()
period_table["rate"] = period_table["podiums"] / period_table["starts"]
period_table = period_table[period_table["starts"] >= MIN_STARTS]
heat = period_table.pivot(index="circuit", columns="period", values="rate") * 100

# keep circuits that appear in at least 2 periods, best in 2014-24 at the top
heat = heat[heat.notna().sum(axis=1) >= 2]
heat = heat.sort_values("2014-24", ascending=False)

plt.figure(figsize=(8, 9))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100,
            cbar_kws={"label": "% of front-row starters on the podium"})
plt.title("Front row (grid 1st/2nd) -> podium per circuit and period\n(blank = fewer than 10 starts)")
plt.xlabel("period")
plt.ylabel("circuit")
plt.show()

### Q2: Do drivers get more podiums at their home circuit?

**In plain words:** does a British driver racing in Britain finish on the podium more often than when racing elsewhere?

**The catch:** if home drivers simply start further forward, they will of course finish better. So we compare
**drivers who started from the same grid row**: a home driver who started 3rd-4th is compared with away drivers who
also started 3rd-4th.

**Tables and joins (multi-hop):**

```
results ──driverId──► drivers                              (nationality of the driver)
   │
   └──raceId──► races ──circuitId──► circuits              (country of the circuit)
```

- `results ⋈ races` is already done inside `load_clean()` → `model_base` (`grid`, `podium`, `year`, `circuitId`).
- Here we add `drivers` (on `driverId`) for the nationality and `circuits` (on `circuitId`) for the country.
- Nationality ("British") and country ("UK") are written differently, so we connect them with a mapping written by hand.
- Not used: qualifying (we control with `grid`, which exists for every race since 1950), standings, lap times, pit stops.
- All seasons, **1950-2024**: nationality, country and grid are never missing.

In [ ]:
# driver nationality (from drivers.csv) -> country name as written in circuits.csv, written by hand
NATIONALITY_TO_COUNTRY = {
    "American": "USA",
    "American-Italian": "USA", 
    "Argentine": "Argentina", 
    "Argentine-Italian": "Argentina",
    "Argentinian": "Argentina", 
    "Australian": "Australia",
    "Austrian": "Austria", 
    "Belgian": "Belgium",
    "Brazilian": "Brazil",
    "British": "UK",
    "Canadian": "Canada", 
    "Chilean": "Chile", 
    "Chinese": "China",
    "Colombian": "Colombia", 
    "Czech": "Czech Republic", 
    "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany",
    "Finnish": "Finland", 
    "French": "France",
    "German": "Germany",
    "Hungarian": "Hungary",
    "Indian": "India",
    "Indonesian": "Indonesia",
    "Irish": "Ireland", 
    "Italian": "Italy", 
    "Japanese": "Japan",
    "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", 
    "Mexican": "Mexico", 
    "Monegasque": "Monaco",
    "New Zealander": "New Zealand", 
    "Polish": "Poland", 
    "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia",
    "Russian": "Russia", 
    "South African": "South Africa",
    "Spanish": "Spain",
    "Swedish": "Sweden",
    "Swiss": "Switzerland",
    "Thai": "Thailand",
    "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}

# drivers with two nationalities have a second home country
SECOND_COUNTRY = {"American-Italian": "Italy", 
                  "Argentine-Italian": "Italy"}

# show the mapping as a table (shared with P3 for the home-race feature)
mapping = pd.DataFrame(list(NATIONALITY_TO_COUNTRY.items()), columns=["nationality", "home_country"])
mapping["second_home_country"] = mapping["nationality"].map(SECOND_COUNTRY)
mapping

In [ ]:
# 1) driver nationality; .str.strip() removes spaces at the ends ("Argentinian " -> "Argentinian")
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip() #  `strip()` removes spaces at the start and end

# 2) circuit country; "United States" and "USA" are the same country written two ways
circuit_country = tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit"})
circuit_country["country"] = circuit_country["country"].replace("United States", "USA")

# 3) joins: every row of base gets the driver's nationality and the circuit's country
q2 = base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
q2 = q2.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks
assert len(q2) == len(base), "a join changed the number of rows"
assert q2["nationality"].notna().all(), "a driver has no nationality"
assert q2["country"].notna().all(), "a circuit has no country"
unmapped = set(q2["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, "nationalities missing from the mapping: " + str(unmapped)

# 5) home = the circuit is in the driver's (first or second) home country
q2["home_country"] = q2["nationality"].map(NATIONALITY_TO_COUNTRY) # the mapping is happening here, not in the join, 
q2["second_home_country"] = q2["nationality"].map(SECOND_COUNTRY) # because some drivers have two nationalities and a second home country
q2["is_home"] = (q2["country"] == q2["home_country"]) | (q2["country"] == q2["second_home_country"])

print("rows:", len(q2), "| home rows:", q2["is_home"].sum(), "| away rows:", (~q2["is_home"]).sum())

#### Demo: how the Q2 joins work, step by step

Six real rows, one for each case: a normal home race, a normal away race, two drivers with two nationalities,
a circuit written "United States", and a nationality with a hidden space. Then two fake rows show what would happen
if a driver had no nationality or a circuit had no country. These cells do not change `q2`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows, one for each case. Nothing here changes q2.

# the 6 example rows: (raceId, driverId)
examples = [
    (1132, 1),     # Hamilton (British) at the British GP 2024        -> home
    (1136, 1),     # Hamilton (British) at the Italian GP 2024        -> away
    (754, 491),    # Thiele (American-Italian) at the Italian GP 1960 -> home by his SECOND country
    (776, 573),    # de Tomaso (Argentine-Italian) at the Argentine GP 1957 -> home by his FIRST country
    (1119, 858),   # Sargeant (American) at Las Vegas 2023: circuit written "United States"
    (1136, 861),   # Colapinto ("Argentinian " with a space) at the Italian GP 2024 -> away
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the three tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "circuitId", "year", "grid", "podium"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_drivers = tables["drivers"][tables["drivers"]["driverId"].isin(demo_base["driverId"])][["driverId", "forename", "surname", "nationality"]].copy()
demo_drivers["nationality in quotes"] = '"' + demo_drivers["nationality"] + '"'   # quotes make the hidden space visible
print("STEP 0b: DRIVERS table, only these drivers (before .str.strip())")
display(demo_drivers)

demo_circuits = tables["circuits"][tables["circuits"]["circuitId"].isin(demo_base["circuitId"])][["circuitId", "name", "country"]]
print("STEP 0c: CIRCUITS table, only these circuits (before the 'United States' -> 'USA' fix)")
display(demo_circuits)

# STEP 1: base + drivers, matched on driverId -> every row gets the nationality (already stripped)
step1 = demo_base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
print("STEP 1: base + drivers (matched on driverId)")
display(step1)

# STEP 2: + circuits, matched on circuitId -> every row gets the circuit name and its country (already fixed)
step2 = step1.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")
print("STEP 2: + circuits (matched on circuitId)")
display(step2)

# STEP 3: home countries and is_home, exactly like the real cell
step2["home_country"] = step2["nationality"].map(NATIONALITY_TO_COUNTRY)
step2["second_home_country"] = step2["nationality"].map(SECOND_COUNTRY)
step2["is_home"] = (step2["country"] == step2["home_country"]) | (step2["country"] == step2["second_home_country"])
print("STEP 3: home country (first and second) and is_home")
display(step2[["driverId", "nationality", "circuit", "country", "home_country", "second_home_country", "is_home"]])

In [ ]:
# mapping check: which countries never meet?
race_countries = set(q2["country"])
driver_countries = set(q2["home_country"]) | set(q2["second_home_country"].dropna())

print("driver countries that never hosted a race (their drivers are always away):")
print(sorted(driver_countries - race_countries))
print("race countries with no driver from that country (nobody is ever at home there):")
print(sorted(race_countries - driver_countries))
print()
print("home starts per country (top 10):")
q2[q2["is_home"]]["country"].value_counts().head(10)

#### Grid groups

We group the starting positions by **grid row**. On an F1 grid the cars stand in rows of two (1st and 2nd side by side,
then 3rd and 4th, ...), so each group is one row: **1-2, 3-4, 5-6, ... 19-20**. Everything further back, and pit-lane
starts (`grid = 0`), go into the last group **"21+ / pit lane"**. Each group has about 150 home starts, enough to compare.
(The work split suggested 1 / 2-3 / 4-6 / 7-10 / 11+; grid rows give the same answer with a finer view.)

In [ ]:
# the groups in grid order (used to sort tables and charts)
GRID_GROUPS = ["1-2", "3-4", "5-6", "7-8", "9-10", "11-12", "13-14", "15-16", "17-18", "19-20", "21+ / pit lane"]

# every row starts in the last group (grid 21 or further back, or pit lane = grid 0)
q2["grid_group"] = "21+ / pit lane"

# then each pair of grid positions gets its own group
q2.loc[q2["grid"].between(1, 2), "grid_group"] = "1-2"
q2.loc[q2["grid"].between(3, 4), "grid_group"] = "3-4"
q2.loc[q2["grid"].between(5, 6), "grid_group"] = "5-6"
q2.loc[q2["grid"].between(7, 8), "grid_group"] = "7-8"
q2.loc[q2["grid"].between(9, 10), "grid_group"] = "9-10"
q2.loc[q2["grid"].between(11, 12), "grid_group"] = "11-12"
q2.loc[q2["grid"].between(13, 14), "grid_group"] = "13-14"
q2.loc[q2["grid"].between(15, 16), "grid_group"] = "15-16"
q2.loc[q2["grid"].between(17, 18), "grid_group"] = "17-18"
q2.loc[q2["grid"].between(19, 20), "grid_group"] = "19-20"

# count home and away starts in each group
group_counts = pd.crosstab(q2["grid_group"], q2["is_home"])                              # count each (group, home/away) pair
group_counts = group_counts.loc[GRID_GROUPS]                                             # put the groups in grid order
group_counts = group_counts.rename(columns={False: "away starts", True: "home starts"})  # readable column names
group_counts

#### Without grid groups vs with grid groups

First the plain comparison (all home starts vs all away starts). Then the fair comparison inside each grid row.
**"difference"** = home % − away %: how many percentage points higher the home rate is in that row
(for example 45% home and 40% away gives +5; a negative number means away is higher).

In [ ]:
# plain comparison, no grid groups

# split the rows into two groups
home_rows = q2[q2["is_home"]]       # only home races
away_rows = q2[~q2["is_home"]]      # not home races      # only away races (~ means NOT)

# build a small table: one row for home, one row for away
plain = pd.DataFrame({
    "starts": [len(home_rows), len(away_rows)],                          # how many races
    "podiums": [home_rows["podium"].sum(), away_rows["podium"].sum()],   # how many podiums (podium is 1 or 0)
}, index=["home", "away"])

# podium % = podiums / starts * 100
plain["podium %"] = plain["podiums"] / plain["starts"] * 100
plain.round(1)

In [ ]:
# fair comparison: podium rate for home and away inside each grid group
# (home_rows and away_rows come from the plain-comparison cell above)

# count starts and podiums per grid group, separately for home and away
home_away = pd.DataFrame({
    "home starts": home_rows.groupby("grid_group")["podium"].count(),    # how many home races in each group
    "home podiums": home_rows.groupby("grid_group")["podium"].sum(),     # how many of them ended on the podium
    "away starts": away_rows.groupby("grid_group")["podium"].count(),
    "away podiums": away_rows.groupby("grid_group")["podium"].sum(),
})

# put the groups in grid order (1-2, 3-4, ... instead of alphabetical)
home_away = home_away.loc[GRID_GROUPS]

# % = podiums / starts * 100, then difference = home % minus away %
home_away["home %"] = home_away["home podiums"] / home_away["home starts"] * 100
home_away["away %"] = home_away["away podiums"] / home_away["away starts"] * 100
home_away["difference"] = home_away["home %"] - home_away["away %"]
home_away.round(1)

In [ ]:
# grouped bar chart: home (orange) next to away (blue) for each grid row
home_away[["home %", "away %"]].plot(kind="bar", color=["tab:orange", "tab:blue"], figsize=(14, 5))

plt.title("Podium rate at home vs away, for drivers who started from the same grid row (1950-2024)")
plt.xlabel("starting position (grid row)")
plt.ylabel("% of drivers who finished on the podium")
plt.ylim(0, 100)                                       # y axis from 0 to 100%
plt.xticks(rotation=0)                                 # keep the group names horizontal
plt.legend(["home (racing in own country)", "away"])   # legend names, same order as the columns
plt.show()

#### One number for all grid groups: 2 methods

Each method turns the 11 differences into one answer. If both say "about 0", there is no home advantage.

1. **Simple average:** the average of the 11 differences in the table (every grid row counts the same).
2. **Exact grid position:** 1st vs 1st, 2nd vs 2nd ... 20th vs 20th, instead of grid rows.

**How the numbers are calculated, step by step (numbers from the table above):**

**Step 1: the % in each row** = podiums ÷ starts × 100. A sample of three rows:

| grid row | home starts | home podiums | home % | away starts | away podiums | away % |
|---|---|---|---|---|---|---|
| 1-2 | 160 | 96 | 96 ÷ 160 × 100 = **60.0** | 2067 | 1254 | 1254 ÷ 2067 × 100 = **60.7** |
| 3-4 | 144 | 65 | 65 ÷ 144 × 100 = **45.1** | 2080 | 838 | 838 ÷ 2080 × 100 = **40.3** |
| 9-10 | 180 | 8 | 8 ÷ 180 × 100 = **4.4** | 2033 | 145 | 145 ÷ 2033 × 100 = **7.1** |

**Step 2: the difference in each row** = home % − away %:

| grid row | home % − away % | difference | meaning |
|---|---|---|---|
| 1-2 | 60.0 − 60.7 | **−0.7** | away slightly higher |
| 3-4 | 45.1 − 40.3 | **+4.9** | home higher |
| 9-10 | 4.4 − 7.1 | **−2.7** | away higher |

In [ ]:
# METHOD 1: simple average of the 11 differences (every grid row counts the same)
average_difference = home_away["difference"].mean()
print("average difference over the 11 grid rows:", round(average_difference, 2), "percentage points")

In [ ]:
# METHOD 2: exact grid position (1st vs 1st, 2nd vs 2nd, ... 20th vs 20th) instead of grid rows

# only grid positions 1 to 20
home_1_20 = home_rows[home_rows["grid"].between(1, 20)]
away_1_20 = away_rows[away_rows["grid"].between(1, 20)]

# podium % for every grid position, at home and away
by_position = pd.DataFrame({
    "home %": home_1_20.groupby("grid")["podium"].mean() * 100,
    "away %": away_1_20.groupby("grid")["podium"].mean() * 100,
})
by_position["difference"] = by_position["home %"] - by_position["away %"]

print("average difference over grid positions 1-20:", round(by_position["difference"].mean(), 2), "percentage points")
by_position.round(1)

#### Answer to Q2

**No, drivers do not get more podiums at home** once we compare drivers who started from the same grid row.

- **Without grid groups**, home even looks worse: 11.9% podiums at home vs 13.8% away. But home drivers start further back
  more often: 15% of home starts are from 21st or further back / the pit lane, vs 11% of away starts
  (291 of 1932 vs 2542 of 22634, see the grid-group table). Many local drivers entered only their home race.
- **With grid groups** (the fair comparison), home and away are almost the same in every row:
  - front row (1-2): 60.0% home vs 60.7% away
  - 5-6: 21.7% vs 21.9%
  - home is clearly higher in only 2 of 11 rows (3-4: +4.9, 11-12: +1.8)
- **One number, 2 methods** (difference = home % − away %, in percentage points):

  | Method | Result |
  |---|---|
  | 1. Simple average of the 11 grid rows | **−0.24** |
  | 2. Exact grid position (1st vs 1st ... 20th vs 20th) | **−0.20** |

  Both are less than 1 percentage point, so home makes practically no difference.
- The +4.9 in row 3-4 comes from only 144 home starts (65 podiums). It is alone among the other rows, so it is not a home advantage.

**Grain**
- Input: one row per driver per race (`model_base`).
- Result: one row per grid row (11 groups), with home and away side by side.
- Method 2: one row per grid position (1-20).
- Both joins (`drivers`, `circuits`) are many-to-one, and the row count is checked to stay the same.

**Missingness**
- Nationality, circuit country and grid are never missing, so all seasons 1950-2024 are used.
- Every nationality is in the mapping (checked with an assert).
- 14 driver countries never hosted a race (e.g. Finland, New Zealand), so their drivers are always "away".
- 9 race countries never had a driver from there (e.g. Bahrain, Singapore), so nobody is "home" there.
- Method 2 uses grid 1-20 only.

**Pre-race cutoff**
- Nationality, circuit country and grid are all known before the race starts.
- `podium` is the outcome we measure, not an input.
- No column from `outcomes` is used.

**Limitations**
- Nationality is not the same as where a driver lives or grew up.
- Only 2 dual nationalities exist in the data. Both of their countries count as home.
- Most home starts are British (446) and Italian (400), so the answer mostly describes those drivers.

### Q3: Which teams' cars break down the most?

**In plain words:** for each team, what share of its race starts ended because the car broke? We compare **2014-2021**
with **2022-2024**.

**Mechanical retirement** = a race start that ended early because **a part of the car failed** (engine, gearbox, brakes,
hydraulics, ...). Not counted: crashes, driver mistakes, and rule decisions (for example a disqualification).

**Where the data comes from:** everything comes from `load_clean()`.
- `model_base`: who started which race, for which team, in which year.
- `outcomes`: how each race ended (`statusId`). The team rule says `outcomes` is *only for looking at the data, never as
  model input*. Q3 only counts how races ended and builds no model, so this is exactly that allowed use.
- `tables["status"]`: the 139 status names, which we label by hand.
- `tables["constructors"]`: the team names.

**Tables and joins (multi-hop):**

```
model_base ──(raceId, driverId)──► outcomes ──statusId──► status        (how did the race end?)
     │
     └──constructorId──► constructors                                     (which team?)
```

`results → races` (for the year) is already done inside `load_clean()`. Not used: sprint results (a different, shorter
race), lap times, pit stops, standings.

In [ ]:
# how each status ended the race, written by hand from the status table (139 values)
ACCIDENT = ["Accident", "Collision", "Collision damage", "Damage", "Debris", "Fatal accident",
            "Puncture", "Tyre puncture", "Front wing", "Rear wing", "Broken wing", "Undertray"]
DRIVER = ["Spun off", "Stalled", "Illness", "Injured", "Injury", "Physical", "Driver unwell", "Eye injury"]
OTHER = ["Disqualified", "Excluded", "Underweight", "Retired", "Not classified", "Withdrew", "Did not qualify",
         "Did not prequalify", "107% Rule", "Not restarted", "Safety", "Safety concerns", "Out of fuel",
         "Refuelling", "Fuel rig", "Wheel nut"]

status_labels = tables["status"][["statusId", "status"]].copy()

# every status starts as "mechanical" (a part of the car failed), then the lists above overwrite it
status_labels["label"] = "mechanical"
status_labels.loc[status_labels["status"].isin(ACCIDENT), "label"] = "accident"
status_labels.loc[status_labels["status"].isin(DRIVER), "label"] = "driver"
status_labels.loc[status_labels["status"].isin(OTHER), "label"] = "other"

# finished the race: "Finished", or "+1 Lap", "+2 Laps", ... (classified, but some laps behind the winner)
finished = (status_labels["status"] == "Finished") | status_labels["status"].str.startswith("+")
status_labels.loc[finished, "label"] = "finished"

# check: every name in the lists really exists in the status table (catches typos)
names_in_lists = set(ACCIDENT + DRIVER + OTHER)
names_in_table = set(status_labels["status"])
assert names_in_lists <= names_in_table, "not in the status table: " + str(names_in_lists - names_in_table)

print("statuses labelled:", len(status_labels))
status_labels["label"].value_counts()

In [ ]:
# the full label table, one line per label (this is the documented mapping)
for label in ["mechanical", "accident", "driver", "other", "finished"]:
    names = status_labels[status_labels["label"] == label]["status"].tolist()
    print(label.upper(), "(" + str(len(names)) + "):", ", ".join(names))
    print()

#### Team renames

Some teams changed their name but stayed the same team (same factory, same staff). We put each name under one
**team history**, so a team's breakdowns are not split over two or three names.

In [ ]:
# team name in constructors.csv -> team history, written by hand (teams not listed keep their own name)
TEAM_HISTORY = {
    "Toro Rosso": "Toro Rosso / AlphaTauri / RB",
    "AlphaTauri": "Toro Rosso / AlphaTauri / RB",
    "RB F1 Team": "Toro Rosso / AlphaTauri / RB",
    "Force India": "Force India / Racing Point / Aston Martin",
    "Racing Point": "Force India / Racing Point / Aston Martin",
    "Aston Martin": "Force India / Racing Point / Aston Martin",
    "Lotus F1": "Lotus / Renault / Alpine",
    "Renault": "Lotus / Renault / Alpine",
    "Alpine F1 Team": "Lotus / Renault / Alpine",
    "Sauber": "Sauber / Alfa Romeo",
    "Alfa Romeo": "Sauber / Alfa Romeo",
    "Marussia": "Marussia / Manor",
    "Manor Marussia": "Marussia / Manor",
}
pd.DataFrame(list(TEAM_HISTORY.items()), columns=["name in the data", "team history"])

**Check: are these really the same team?** For every renamed team we print the seasons and the drivers. The old name
stops exactly when the new one starts, and the same drivers appear under both names (for example Gasly at Toro Rosso
and AlphaTauri, Pérez at Force India and Racing Point, Ocon at Renault and Alpine, Bottas and Zhou at Alfa Romeo and Sauber).

In [ ]:
# seasons and drivers of every renamed team, 2014-2024
check = base[base["year"] >= 2014]
check = check.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
check = check.merge(tables["drivers"][["driverId", "surname"]], on="driverId", how="left")

for name in TEAM_HISTORY:
    rows = check[check["name"] == name]
    seasons = sorted(set(rows["year"].tolist()))     # .tolist() gives plain numbers (2014 instead of np.int64(2014))
    drivers = sorted(set(rows["surname"].tolist()))
    print(TEAM_HISTORY[name], "|", name, "| seasons:", seasons, "| drivers:", ", ".join(drivers))

#### Build the Q3 table (2014-2024)

In [ ]:
# 1) who started which race for which team (2014-2024)
q3 = base[base["year"] >= 2014][["raceId", "driverId", "constructorId", "year"]]

# 2) joins: how the race ended (statusId) -> its status name and label -> the team name
q3 = q3.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left", validate="one_to_one")
q3 = q3.merge(status_labels, on="statusId", how="left", validate="many_to_one")
q3 = q3.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left", validate="many_to_one")

# 3) safety checks
assert len(q3) == (base["year"] >= 2014).sum(), "a join changed the number of rows"
assert q3["label"].notna().all(), "a race end has no label"
assert q3["name"].notna().all(), "a team has no name"

# 4) team history (renames merged) and period
q3["team"] = q3["name"].replace(TEAM_HISTORY)
q3["period"] = "2014-21"
q3.loc[q3["year"] >= 2022, "period"] = "2022-24"

print("race starts:", len(q3), "| teams:", q3["team"].nunique())
print(q3.groupby("period")["raceId"].nunique())

#### Demo: how the Q3 joins work, step by step

Six real rows, one for each kind of race end, including two renamed teams (Aston Martin, AlphaTauri). Then two fake
rows show what would happen if a status or a team could not be found. These cells do not change `q3`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows. Nothing here changes q3.
examples = [
    (963, 1),      # Hamilton, Malaysian GP 2016       -> Engine        -> mechanical
    (952, 3),      # Rosberg, Spanish GP 2016          -> Collision     -> accident
    (1076, 832),   # Sainz, Australian GP 2022         -> Spun off      -> driver
    (1062, 20),    # Vettel (Aston Martin), Hungary 2021 -> Disqualified -> other
    (1038, 842),   # Gasly (AlphaTauri), Italian GP 2020 -> Finished    -> finished (he won)
    (1137, 815),   # Pérez, Azerbaijan GP 2024         -> Retired       -> other (no reason given)
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "constructorId", "year"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_outcomes = outcomes.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "statusId"]]
print("STEP 0b: OUTCOMES, the same 6 rows (only statusId)")
display(demo_outcomes)

print("STEP 0c: STATUS_LABELS, only these statuses")
display(status_labels[status_labels["statusId"].isin(demo_outcomes["statusId"])])

print("STEP 0d: CONSTRUCTORS, only these teams")
display(tables["constructors"][tables["constructors"]["constructorId"].isin(demo_base["constructorId"])][["constructorId", "name"]])

# STEP 1: base + outcomes, matched on raceId AND driverId -> every row gets its statusId
step1 = demo_base.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left")
print("STEP 1: base + outcomes (matched on raceId and driverId)")
display(step1)

# STEP 2: + status_labels, matched on statusId -> every row gets the status name and its label
step2 = step1.merge(status_labels, on="statusId", how="left")
print("STEP 2: + status_labels (matched on statusId)")
display(step2)

# STEP 3: + constructors, matched on constructorId -> team name, then team history and period
step3 = step2.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
step3["team"] = step3["name"].replace(TEAM_HISTORY)
step3["period"] = "2014-21"
step3.loc[step3["year"] >= 2022, "period"] = "2022-24"
print("STEP 3: + constructors (matched on constructorId), team history and period")
display(step3)

#### How did the races end?

Before the team rates, the big picture: how many starts in each period ended in each way.

In [ ]:
# count of starts per period and label
ended = pd.crosstab(q3["period"], q3["label"])
ended = ended[["finished", "mechanical", "accident", "driver", "other"]]
display(ended)

# the same as % of starts (normalize="index" makes every row add up to 100%)
ended_pct = pd.crosstab(q3["period"], q3["label"], normalize="index") * 100
ended_pct = ended_pct[["finished", "mechanical", "accident", "driver", "other"]]
ended_pct.round(1)

#### Mechanical retirement rate per team

**Rate** = mechanical retirements ÷ race starts × 100. Example: 46 breakdowns in 319 starts = 14.4%.

**Minimum 40 starts per team and period** (about one full season with 2 cars). A team with fewer starts raced only
part of a season, so a few breakdowns would give it an extreme rate. Teams below the minimum are shown but not ranked.

In [ ]:
MIN_TEAM_STARTS = 40   # about one full season with 2 cars

# True / False for every start: did it end with a car failure?
q3["mechanical"] = q3["label"] == "mechanical"

# per period and team: starts (count of rows) and breakdowns (sum of True = 1)
starts = q3.groupby(["period", "team"])["mechanical"].count()
breakdowns = q3.groupby(["period", "team"])["mechanical"].sum()

rates = pd.DataFrame({"starts": starts, "mechanical": breakdowns})
rates["rate %"] = rates["mechanical"] / rates["starts"] * 100
rates["enough starts"] = rates["starts"] >= MIN_TEAM_STARTS

print("left out (fewer than", MIN_TEAM_STARTS, "starts):")
display(rates[~rates["enough starts"]].round(1))
rates.round(1)

In [ ]:
# one bar chart per period, teams sorted by rate (the highest at the top)
for period in ["2014-21", "2022-24"]:
    one_period = rates.loc[period]                               # only this period (rows = teams)
    one_period = one_period[one_period["enough starts"]]         # only teams with enough starts
    one_period = one_period.sort_values("rate %")                # low to high, so barh draws the highest at the top

    plt.figure(figsize=(9, 5))
    plt.barh(one_period.index, one_period["rate %"], color="tab:red", label="mechanical retirements per 100 starts")
    plt.title("Mechanical retirement rate per team, " + period)
    plt.xlabel("% of race starts that ended with a car failure")
    plt.ylabel("team")
    plt.xlim(0, 16)                                              # same scale in both charts, so they can be compared
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
MIN_TEAM_STARTS = 30   # about one full season with 2 cars

# True / False for every start: did it end with a car failure?
q3["mechanical"] = q3["label"] == "mechanical"

# per period and team: starts (count of rows) and breakdowns (sum of True = 1)
starts = q3.groupby(["period", "team"])["mechanical"].count()
breakdowns = q3.groupby(["period", "team"])["mechanical"].sum()

rates = pd.DataFrame({"starts": starts, "mechanical": breakdowns})
rates["rate %"] = rates["mechanical"] / rates["starts"] * 100
rates["enough starts"] = rates["starts"] >= MIN_TEAM_STARTS

print("left out (fewer than", MIN_TEAM_STARTS, "starts):")
display(rates[~rates["enough starts"]].round(1))
rates.round(1)

In [ ]:
# one bar chart per period, teams sorted by rate (the highest at the top)
for period in ["2014-21", "2022-24"]:
    one_period = rates.loc[period]                               # only this period (rows = teams)
    one_period = one_period[one_period["enough starts"]]         # only teams with enough starts
    one_period = one_period.sort_values("rate %")                # low to high, so barh draws the highest at the top

    plt.figure(figsize=(9, 5))
    plt.barh(one_period.index, one_period["rate %"], color="tab:red", label="mechanical retirements per 100 starts")
    plt.title("Mechanical retirement rate per team, " + period)
    plt.xlabel("% of race starts that ended with a car failure")
    plt.ylabel("team")
    plt.xlim(0, 16)                                              # same scale in both charts, so they can be compared
    plt.legend(loc="lower right")
    plt.show()

#### Answer to Q3

**2014-2021, highest mechanical-retirement rate:**
1. Toro Rosso / AlphaTauri / RB: **14.4%** (46 of 319 starts)
2. Lotus / Renault / Alpine: 13.8% (44 of 318)
3. McLaren: 13.7% (43 of 315)

Lowest: Mercedes, **3.8%** (12 of 320).

**2022-2024, highest mechanical-retirement rate:**
1. Lotus / Renault / Alpine: **10.4%** (14 of 135)
2. Sauber / Alfa Romeo: 8.8% (12 of 136)
3. Toro Rosso / AlphaTauri / RB: 5.9% (8 of 136)

Lowest: Red Bull and McLaren, **3.7%** (5 of 136 each).

**What changed:**
- Cars broke down less in 2022-24. Mechanical retirements fell from 9.4% to 5.5% of all starts, while crashes stayed the
  same (6.9% → 6.7%). More cars finished (82.0% → 86.2%).
- Lotus / Renault / Alpine is near the top in both periods (2nd → 1st).
- McLaren dropped the most, from 13.7% (3rd) to 3.7% (lowest).

**Grain**
- Input: one row per driver per race start (`model_base`, 2014-2024: 4595 starts).
- Result: one row per team and period.
- All joins are checked: `outcomes` is one-to-one, `status` and `constructors` are many-to-one, and the row count stays the same.

**Missingness**
- Every start has a status, and all 139 status values have a label (checked with asserts).
- Teams with fewer than 40 starts in a period are shown but not ranked (only Caterham 2014: 7 of 33 = 21%).

**Pre-race cutoff**
- The status (how the race ended) is the *outcome* we measure. It comes from `outcomes` and is never used as a model input.

**Limitations**
- "Retired" without a reason counts as other, so a few breakdowns may be missed. It is a similar share in both periods
  (1.5% vs 1.2% of starts, inside "other"), so the comparison stays fair.
- Borderline labels are a choice: punctures and wing damage count as accidents (usually contact), a failed tyre counts as mechanical.
- Team renames are merged by hand (checked: same seasons in a row, same drivers).
- 2022-24 has only 3 seasons (about 135 starts per team), so one or two breakdowns move a team's rate by about 1 point.
- A breakdown is counted for the team, even when the failing part (for example the engine) comes from another manufacturer.

---
# >>> Section from `notebooks/p3_features.ipynb`


## Part 3: Features engineering, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
clean = load_clean(exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

The dataset has been cleaned with 0 duplicated rows for (raceId, driverId). The missing values are present in the qualifying-related columns q1_s, q2_s, q3_s, qual_position.

In [ ]:
import pandas as pd

# Missingness across the full base table (all columns, all years)
miss = base.isna().mean().sort_values(ascending=False)

summary = pd.DataFrame({
    "missing_count": base.isna().sum(),
    "missing_share": miss,
    "dtype": base.dtypes,
})
summary["missing_share"] = summary["missing_share"].round(4)

print("=" * 65)
print(f"base shape: {base.shape[0]:,} rows × {base.shape[1]} columns")
print(f"Columns with ANY missing:  {(summary['missing_count'] > 0).sum()}")
print(f"Columns with NO  missing:  {(summary['missing_count'] == 0).sum()}")
print("=" * 65)

print("\nColumns with missing values (sorted):")
print(summary[summary["missing_count"] > 0].to_string())

In [ ]:
# Grain and target check: one row per (raceId, driverId), podium is 0/1
dupes = base.duplicated(["raceId", "driverId"]).sum()
assert dupes == 0, "more than one row for a driver in a race"
assert set(base["podium"].unique()) <= {0, 1}, "podium is not binary"

print("duplicated (raceId, driverId):", dupes)
print("target values:", sorted(base["podium"].unique()))
print("overall podium rate:", round(base["podium"].mean(), 4))

In [ ]:
import matplotlib.pyplot as plt

# Missing share (%) per year for the qualifying columns
qual_cols = ["qual_position", "q1_s", "q2_s", "q3_s"]
miss_by_year = base.groupby("year")[qual_cols].apply(lambda d: d.isna().mean()) * 100

fig, ax = plt.subplots(figsize=(16, 3.5))
im = ax.imshow(miss_by_year.T, aspect="auto", cmap="Blues", vmin=0, vmax=100)

# Axis labels: columns on y, every 5th year on x
years = list(miss_by_year.index)
ax.set_yticks(range(len(qual_cols)), qual_cols)
ax.set_xticks(range(0, len(years), 5), years[::5])

# Mark the 2006 start year
ax.axvline(years.index(2006) - 0.5, color="red", linestyle="--", linewidth=2)
ax.text(years.index(2006), -0.7, "2006 (knockout starts)", color="red", fontsize=9)

fig.colorbar(im, ax=ax, label="% of rows missing")
ax.set_title("Missing qualifying data per year (dark = more missing)", pad=18)
ax.set_xlabel("Year")
plt.tight_layout()
plt.savefig("reports/fig_p3_qual_missing_per_year.png", dpi=120, bbox_inches="tight")
plt.show()

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-1993),
so we need to choose the earliest stable year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Find the first year with any qualifying data and the first year with high coverage
q_cov_exact = base.groupby("year")["has_qualifying"].mean()

# Years before this have no qualifying data at all
years_with_any = q_cov_exact[q_cov_exact > 0].index.min()
# Years before this have full or near-full coverage
years_with_full = q_cov_exact[q_cov_exact >= 0.90].index.min()

print("=" * 60)
print("First year with ANY qualifying data (has_qualifying > 0):", years_with_any)
print("First year with >=90% coverage:                      ", years_with_full)
print("=" * 60)

print("\nCoverage 1990–1997 (the transition years):")
print(q_cov_exact.loc[1990:1997].round(4))

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1993-2007:")
print(q_cov.loc[1993:2007])

In [ ]:
print("\nq1_s / q2_s / q3_s coverage for 2006-2024:")
print(q_cov.loc[2006:2024])

**From 2006 the qualifying data is stable for q1, q2 and q3, because the knockout format starts in 2006. Earlier seasons have little or no qualifying data, so they are kept but get a lower weight in training (see below).**

In [ ]:
START_YEAR = 1950          # keep every season
FULL_WEIGHT_FROM = 2006    # knockout qualifying: all qualifying columns exist from here
OLD_SEASON_WEIGHT = 0.25   # pre-2006 rows count a quarter in training (tested in 3.11, Part F)
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])
print("\nmodel shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

In [ ]:
import matplotlib.pyplot as plt

# Coverage per season for each qualifying time column
q_cov = (
    base.groupby("year")[["q1_s", "q2_s", "q3_s"]]
        .apply(lambda d: d.notna().mean())
)

# Focus on the era where qualifying data actually exists
q_cov = q_cov.loc[1994:]

fig, ax = plt.subplots(figsize=(12, 5))

for col, color in zip(["q1_s", "q2_s", "q3_s"],
                      ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(q_cov.index, q_cov[col], marker="o", label=col, color=color)

ax.axvline(2006, color="red", linestyle="--", label="full weight from 2006")

ax.set_title("Qualifying time coverage per season, 1994–2024")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with a time")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("reports/fig_p3_qual_coverage_per_year.png",
            dpi=120, bbox_inches="tight")
plt.show()

**Why we keep every season, with a lower weight before 2006**

The plot shows qualifying coverage per season. Before 1994 there is no qualifying data, 1994–95 have near-full q1 coverage, 1996–2002 are patchy, 2003–2005 have q1 only, and from 2006 (knockout format) all three sessions are stable.

| Column | Range 2006–2024 | Meaning |
|---|---|---|
| `q1_s` | 0.97 – 1.00 | Almost every entrant sets a Q1 time |
| `q2_s` | 0.70 – 0.75 | ~15 of ~20 drivers reach Q2 |
| `q3_s` | 0.37 – 0.50 | ~10 of ~20 drivers reach Q3 |

The lower plateaus are structural, not missing data: only the top 15 reach Q2 and the top 10 reach Q3.

**Decision.** We keep every season from 1950 (24,566 rows) instead of dropping 1950–2005. In training, rows from 2006 on get weight 1 and rows before 2006 get weight `OLD_SEASON_WEIGHT = 0.25`, stored in the `sample_weight` column. The old seasons count less because most of them have no qualifying data and come from a different version of the sport (more cars per race, more retirements, other points systems). Section 3.11, Part F compares this with dropping the old seasons and with using them at full weight.

### Section 3.4: Time split

We split by year. The model's job is to predict
future races from pre-race information, so training on later rows and
testing on earlier ones would leak future information into the model. A
time split matches how the model will actually be used.

- **Train**: year ≤ 2019 (22,441 rows: 16,699 from 1950–2005 with weight 0.25, 5,742 from 2006–2019 with weight 1)
- **Validation**: 2020–2021 (774 rows)
- **Test**: year ≥ 2022 (1,351 rows)

**Why these boundaries.**
Training covers every season up to 2019 (pre-COVID), with the seasons before 2006 down-weighted. Validation covers the nearest seasons to the test set and is used for model, threshold and preprocessing choices. Its weakness is that 2020 was shortened by COVID, so we report results per season. Test covers the ground-effect era, a regulation change the model never sees in training, and is used once.

**Weights.** Only training rows before 2006 have a weight below 1. Validation and test rows all have weight 1, so the scores are not affected by the weights.

**Class balance.** Train 13.5% (slightly lower because early races had more cars), validation 15.1%, test 15.1%.

**Alternatives.** A random split leaks same-season information. Random assignment of seasons mixes regulations.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

# Training weight: seasons before 2006 count less, because most of them have
# little or no qualifying data. Validation and test rows (2020+) all have weight 1.
model["sample_weight"] = np.where(model["year"] < FULL_WEIGHT_FROM, OLD_SEASON_WEIGHT, 1.0)

print(model["split"].value_counts())
print("\nRows and weight per split:")
print(model.groupby("split")["sample_weight"].agg(rows="count", weighted_rows="sum", mean_weight="mean").round(2))
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

### Section 3.5: Grid and qualifying features

The prediction cutoff is after qualifying and before the race. Every feature
below is set during or before grid formation; none comes from the race itself.

**Grid features**

- `grid_fixed` is the ordinal starting position. The raw `grid` column uses
  `0` for a pit-lane start, which a linear model would read as ahead of pole. We set those to PIT_GRID = 35, one place past the largest grid in the training years (34, in 1953). The value is derived from training data only, and a range check confirms no validation or test grid exceeds the training maximum.
- pit_lane_flag is 1 for pit-lane starts and 0 otherwise. It keeps the start type that grid_fixed compresses into one ordinal value.

**Qualifying features**

- `qual_position` is the driver's rank in the qualifying session.
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)` across the sessions they reached).
- `gap_to_pole_pct` is the percentage by which the driver’s best time is slower than the fastest time in the same race. Pole is defined by fastest time, not by grid position, because 64 races have a pole starter who was not the fastest qualifier. 61 of them are from 2006 on, and 39 of those from 2006–2009, when Q3 was run on race fuel: the cars were heavier in Q3, so the fastest lap of the day often came from Q2.
- `no_qual_time` is 1 when the driver set no qualifying time at all
  (`best_qual_s` is NaN). There is no qualifying data before 1994, so this
  is 1 for 14,306 rows (58%), almost all before 2006. From 2006 on it is 1
  for only 96 rows (21 of them also have no `qual_position`): drivers
  excluded from qualifying, withdrawn, or unrecorded, who start from the
  back or the pit lane.
- `qual_group` is the last session the driver reached (Q3 / Q2 only /
  Q1 only / no qualifying time), with yes/no columns `qg_q3`,
  `qg_q2_only` and `qg_q1_only`. The model uses only `qg_q3` and
  `qg_q2_only`: since `qg_q3 + qg_q2_only + qg_q1_only + no_qual_time = 1`
  on every row, `qg_q1_only` can be derived from the others and adds nothing. It only has its full meaning
  in the knockout format (2006+).

  **Why `q1_s`, `q2_s`, `q3_s` are not features.** The raw session times
are used to build `best_qual_s` (the minimum across sessions the driver
reached) and `gap_to_pole_pct` (the normalised gap to the fastest lap in
the race). The raw columns are not in `FEATURES` for three reasons:

1. Their missingness is structural: a Q1-eliminated driver has NaN Q2 and
   Q3. `best_qual_s` and `gap_to_pole_pct` handle that by falling back on
   whichever sessions the driver reached.
2. Absolute lap times vary by track, so the
   raw seconds are not comparable across races. `gap_to_pole_pct`
   normalises by the race's fastest time.
3. Including all three would add three highly correlated columns with
   different missingness patterns for the same underlying fact.

In [ ]:
# Grid features: grid_fixed (ordinal) + pit_lane_flag (indicator)

# 1. Max real grid slot, from TRAIN ONLY.
max_grid = model.loc[
    (model["split"] == "train") & (model["grid"] > 0), "grid"
].max()
print("Max real grid slot (train years only):", max_grid)

# 2. Sentinel: one past the largest real grid slot.
PIT_GRID = max_grid + 1
print(f"PIT_GRID = {PIT_GRID}")

# 3. Build the two features.
model["pit_lane_flag"] = (model["grid"] == 0).astype(int)
model["grid_fixed"] = model["grid"].where(model["grid"] > 0, PIT_GRID)

# 4. Crosstab: pit_lane_flag agrees with pitlane_start.
#    pitlane_start is defined as grid == 0 in f1_clean, so this always
#    agrees (consistency check only).
print("\nCrosstab (grid==0 vs pitlane_start):")
print(pd.crosstab(model["grid"] == 0, model["pitlane_start"]))

# 5. Distribution tail.
print("\nGrid value counts (last 5):")
print(model["grid_fixed"].value_counts().sort_index().tail())

# 6. Assertions.
assert model["grid_fixed"].notna().all(), "null grid after recode"
assert model["grid_fixed"].between(1, PIT_GRID).all()
assert (model.loc[model["pit_lane_flag"] == 1, "grid_fixed"] == PIT_GRID).all()

# 7. Range check: no val/test grid exceeds the train max.
leak_check = model.loc[
    (model["split"] != "train") & (model["grid"] > max_grid)
]
assert len(leak_check) == 0, \
    f"val/test rows with real grid > train max_grid: {len(leak_check)}"

print(f"\nPit lane starts: {model['pit_lane_flag'].sum()} rows")
print(f"Rows at sentinel ({PIT_GRID}): {(model['grid_fixed'] == PIT_GRID).sum()}")
print(f"Val/test rows with grid above train max ({max_grid}): {len(leak_check)}")
print("Grid features: OK")

In [ ]:
# Qualifying features
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

# Gap to pole (percentage)
pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

# No-time flag: 1 when the driver set no qualifying time at all
model["no_qual_time"] = model["gap_to_pole_pct"].isna().astype(int)

print(model[[
    "grid_fixed", "pit_lane_flag", "qual_position",
    "best_qual_s", "gap_to_pole_pct", "no_qual_time",
]].describe())

In [ ]:
defined = model["gap_to_pole_pct"].notna()
assert (model.loc[defined, "gap_to_pole_pct"] >= 0).all(), \
    "negative gap found on rows with a defined gap"

# Pole = fastest best_qual_s in the race, not qual_position == 1
model["is_pole_time"] = (
    model["best_qual_s"]
    == model.groupby("raceId")["best_qual_s"].transform("min")
)
assert (model.loc[model["is_pole_time"] & defined, "gap_to_pole_pct"] == 0).all(), \
    "driver with race-minimum best_qual_s does not have gap 0"

print(f"Rows with defined gap: {defined.sum()}")
print(f"Rows with NaN gap (no qualifying time at all): {(~defined).sum()}")
print(f"Rows with pole-time: {model['is_pole_time'].sum()}")

# Optional: how often qual_position == 1 is not the fastest time
mismatch = model[(model["qual_position"] == 1) & ~model["is_pole_time"]]
print(f"Races where qual_position == 1 is not the fastest qualifier: "
      f"{mismatch['raceId'].nunique()}")

# Confirm the NaN gap is fully explained by missing qualifying data
all_q_missing = model[["q1_s", "q2_s", "q3_s"]].isna().all(axis=1)
assert (all_q_missing == model["gap_to_pole_pct"].isna()).all(), \
    "NaN gap does not match all-three-missing"
print("NaN gap == all three qualifying times missing: OK")

In [ ]:
import numpy as np

model["qual_group"] = np.select(
    [model["q3_s"].notna(),
     model["q2_s"].notna(),
     model["q1_s"].notna()],
    ["Q3", "Q2 only", "Q1 only"],
    default="no qualifying time",
)

# Yes/no columns for the table.
for label, col in zip(["Q3", "Q2 only", "Q1 only"], ["qg_q3", "qg_q2_only", "qg_q1_only"]):
    model[col] = (model["qual_group"] == label).astype(int)

# The model uses two of them: qg_q1_only = 1 - qg_q3 - qg_q2_only - no_qual_time,
# so it carries no extra information ("Q1 only" is the case where all three are 0).
assert (model[["qg_q3", "qg_q2_only", "qg_q1_only", "no_qual_time"]].sum(axis=1) == 1).all()
QUAL_GROUP_COLS = ["qg_q3", "qg_q2_only"]

print(
    model.groupby("qual_group", observed=True)
         .agg(
             n_rows=("gap_to_pole_pct", "size"),
             n_defined=("gap_to_pole_pct", "count"),
             mean=("gap_to_pole_pct", "mean"),
             median=("gap_to_pole_pct", "median"),
             max=("gap_to_pole_pct", "max"),
         )
         .round(3)
)

In [ ]:
# Candidate: gap to the fastest Q1 lap in the same race (%).
# Every driver runs Q1, and Q1 was never run on race fuel, so all laps are
# comparable. Tested against gap_to_pole_pct in Section 3.11.
q1_pole = model.groupby("raceId")["q1_s"].transform("min")
model["gap_to_pole_q1_pct"] = (model["q1_s"] - q1_pole) / q1_pole * 100

defined_q1 = model["gap_to_pole_q1_pct"].notna()
assert (model.loc[defined_q1, "gap_to_pole_q1_pct"] >= 0).all(), "negative Q1 gap"
print(model["gap_to_pole_q1_pct"].describe().round(3))
print("missing share:", round(model["gap_to_pole_q1_pct"].isna().mean(), 4))

### Section 3.6: Pre-race standings

The standings tables record cumulative points and rank **after** each
race. Using them directly would leak the current race's result into the
features.

Fix: sort the standings by `(year, round)`, then `shift(1)` within
`(year, driverId)` and `(year, constructorId)` so each row holds the
standings going into that race. The shift never uses the current race.

Round 1 has no previous race: points are 0 (the true value), position is
undefined (NaN), flagged by `standings_missing`. Coverage gaps — the 419
`(raceId, driverId)` pairs inside the modeling window where the driver
raced but has no standings row — are also filled with 0 for points and
flagged with `standings_missing = 1`. A diagnostic confirmed none of them
has an earlier standing in the same season, so 0 is the correct value, not
a lost earlier total. The same holds for the 399 `(raceId, constructorId)`
gaps, most of them before 1958, when there was no constructors' championship.

We assert: one standings row per `(year, id, round)`, round-1 points
are 0, round-1 position is NaN.

In [ ]:
ds = tables["driver_standings"][
    ["raceId", "driverId", "points", "position"]
].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

assert not ds.duplicated(["year", "driverId", "round"]).any(), \
    "duplicate driver standings row"

# Shift within (year, driverId): round N gets round N-1's values.
ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

# Round 1 has no previous race: points are 0, position is undefined.
ds["pts_before"] = ds["pts_before"].fillna(0.0)
ds["standings_missing"] = ds["pos_before"].isna().astype("int8")

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before",
             "standings_missing"]]

print("ds_pre shape:", ds_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["pts_before", "pos_before", "standings_missing",
                      "pts_before_x", "pts_before_y",
                      "pos_before_x", "pos_before_y",
                      "standings_missing_x", "standings_missing_y"]],
    errors="ignore",
)

model = model.merge(ds_pre, on=["raceId", "driverId"], how="left", validate="many_to_one")
model["pts_before"] = model["pts_before"].fillna(0.0)

model["standings_missing"] = model["standings_missing"].fillna(1).astype("int8")   # 🆕
assert model["standings_missing"].notna().all(), "standings flag has empty values"  # 🆕

first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].eq(0.0).all(), "round 1 should have 0 points"
assert model.loc[first_race, "pos_before"].isna().all(), "round 1 position should be NaN"
print("Driver standings: OK")

In [ ]:
cs = tables["constructor_standings"][
    ["raceId", "constructorId", "points", "position"]
].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

assert not cs.duplicated(["year", "constructorId", "round"]).any(), \
    "duplicate constructor standings row"

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs["cons_pts_before"] = cs["cons_pts_before"].fillna(0.0)
cs["cons_standings_missing"] = cs["cons_pos_before"].isna().astype("int8")

cs_pre = cs[["raceId", "constructorId",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]]

print("cs_pre shape:", cs_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["cons_pts_before", "cons_pos_before",
                      "cons_standings_missing",
                      "cons_pts_before_x", "cons_pts_before_y",
                      "cons_pos_before_x", "cons_pos_before_y",
                      "cons_standings_missing_x", "cons_standings_missing_y"]],
    errors="ignore",
)

model = model.merge(cs_pre, on=["raceId", "constructorId"],
                    how="left", validate="many_to_one")
model["cons_pts_before"] = model["cons_pts_before"].fillna(0.0)
model["cons_standings_missing"] = model["cons_standings_missing"].fillna(1).astype("int8")   # 🆕
assert model["cons_standings_missing"].notna().all(), "constructor flag has empty values"   # 🆕


assert model.loc[first_race, "cons_pts_before"].eq(0.0).all(), \
    "round 1 should have 0 constructor points"
assert model.loc[first_race, "cons_pos_before"].isna().all(), \
    "round 1 constructor position should be NaN"
print("Constructor standings: OK")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before", "standings_missing",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]].isna().mean())

In [ ]:
# Points share of the leader going into the race (0 to 1).
# Raw points are not comparable across years: a win was worth 10 points before
# 2010 and 25 after. The share is. Candidate feature, tested in the ablation (3.11).
leader = model.groupby("raceId")["pts_before"].transform("max")
model["pts_share"] = (model["pts_before"] / leader).where(leader > 0, 0.0)

cons_leader = model.groupby("raceId")["cons_pts_before"].transform("max")
model["cons_pts_share"] = (model["cons_pts_before"] / cons_leader).where(cons_leader > 0, 0.0)

assert model[["pts_share", "cons_pts_share"]].notna().all().all(), "empty share values"
assert model["pts_share"].between(0, 1).all() and model["cons_pts_share"].between(0, 1).all()
print(model[["pts_share", "cons_pts_share"]].describe().round(3))

**Recent form (candidate features).** `driver_form5` is the driver's podium
rate over their previous 5 races; `constructor_form5` is the same for the
team (per race, the share of its drivers on the podium). Both are computed on
all years, so an early-2006 row still sees the end of 2005, and both use
`shift(1)` before the rolling window, so a row never sees its own race. For
constructors we first collapse to one row per team-race, so a teammate's
result in the same race cannot leak in. A driver's or team's first race has
no history and stays NaN. Whether form adds anything beyond the standings is
tested in Section 3.11.

In [ ]:
# Safe to rerun.
model = model.drop(columns=["driver_form5", "constructor_form5"], errors="ignore")

# Driver form: podium rate over the previous 5 races (all years).
form_d = base.sort_values(["driverId", "year", "round"]).reset_index(drop=True)
form_d["driver_form5"] = (
    form_d.groupby("driverId")["podium"]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Constructor form: one row per team-race first, then the same window.
form_c = (
    base.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["podium"].mean()
        .sort_values(["constructorId", "year", "round"])
        .reset_index(drop=True)
)
form_c["constructor_form5"] = (
    form_c.groupby("constructorId")["podium"]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

model = model.merge(form_d[["raceId", "driverId", "driver_form5"]],
                    on=["raceId", "driverId"], how="left", validate="one_to_one")
model = model.merge(form_c[["raceId", "constructorId", "constructor_form5"]],
                    on=["raceId", "constructorId"], how="left", validate="many_to_one")

# Leakage checks: no history on a first race, teammates share the team value.
first_d = form_d.groupby("driverId").cumcount() == 0
assert form_d.loc[first_d, "driver_form5"].isna().all(), "driver form on a first career race"
first_c = form_c.groupby("constructorId").cumcount() == 0
assert form_c.loc[first_c, "constructor_form5"].isna().all(), "constructor form on a first race"
assert (model.groupby(["raceId", "constructorId"])["constructor_form5"]
             .nunique(dropna=False) <= 1).all(), "teammates have different constructor form"

print(model[["driver_form5", "constructor_form5"]].describe().round(3))
print("missing share:", model[["driver_form5", "constructor_form5"]].isna().mean().round(4).to_dict())

### Section 3.7: Circuit history + home flag + season progress + era ###

We also need to add these to our features:

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We build five circuit features:
- `circuit_starts_before`: number of previous races by this driver at this circuit
- `circuit_podiums_before`: number of previous podiums by this driver at this circuit
- `circuit_podium_rate`: the driver's smoothed podium rate at this circuit,
  `(podiums + PRIOR × 5) / (starts + 5)`. A driver with few visits is pulled
  toward the average instead of showing 0% or 100%.
- `team_circuit_starts_before`: number of previous races by this constructor at this circuit
- `team_circuit_podiums_before`: the constructor's previous podiums at this circuit
  (per race, the share of its drivers on the podium, so one of two drivers = 0.5)

All five use only past races. The current race is excluded by `cumcount()` and
`shift(1).cumsum()`. They are computed on the full `base` table (all years), so a
2006 row still knows the driver's earlier visits.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone).

**Season progress.** `season_progress = round / last round of the season`
(0 to 1). The calendar is known before the season starts.

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `pre_2006` (1950–2005), `early_v8` (2006–2008), `late_v8` (2009–2013),
`hybrid` (2014–2021), `ground_effect` (2022+), and encode them as `era_ordinal` (0 to 4).

**Note on `era_ordinal`.** It is in `FEATURES` only as a candidate for the
ablation (Section 3.11). It has a structural limitation: the ground-effect era
(2022+, value 4) never appears in training (1950–2019), and validation
(2020–2021) is entirely hybrid, so the effect of value 3 cannot be learned or
checked before the test set. The final decision is made in Section 3.11.

In [ ]:
base_sorted_circuit = (
    base.sort_values(["driverId", "circuitId", "year", "round"])
        .reset_index(drop=True)
)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)
base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["circuit_starts_before", "circuit_podiums_before",
                      "circuit_podium_rate",
                      "circuit_starts_before_x", "circuit_starts_before_y",
                      "circuit_podiums_before_x", "circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before",
                         "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left",
)

# Smoothed podium rate.
PRIOR = 0.15
PRIOR_STRENGTH = 5
model["circuit_podium_rate"] = (
    (model["circuit_podiums_before"] + PRIOR * PRIOR_STRENGTH)
    / (model["circuit_starts_before"] + PRIOR_STRENGTH)
)

print(model[["circuit_starts_before", "circuit_podiums_before",
             "circuit_podium_rate"]].describe().round(3))

In [ ]:
first_at_circuit = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_starts_before"] == 0).all()
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_podiums_before"] == 0).all()
print("Circuit history leakage check: OK")

In [ ]:
base_team_circuit = (
    base
      .groupby(["constructorId", "circuitId", "raceId", "year", "round"],
               as_index=False)["podium"]
      .mean()
)
base_team_circuit = base_team_circuit.sort_values(
    ["constructorId", "circuitId", "year", "round"]
).reset_index(drop=True)

base_team_circuit["team_circuit_starts_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"]).cumcount()
)
base_team_circuit["team_circuit_podiums_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"])["podium"]
                    .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["team_circuit_starts_before",
                      "team_circuit_podiums_before",
                      "team_circuit_starts_before_x",
                      "team_circuit_starts_before_y",
                      "team_circuit_podiums_before_x",
                      "team_circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_team_circuit[["raceId", "constructorId",
                       "team_circuit_starts_before",
                       "team_circuit_podiums_before"]],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one",
)

print(model[["team_circuit_starts_before",
             "team_circuit_podiums_before"]].describe().round(3))

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "UK", "Canadian": "Canada",
    "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France",
    "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Indonesian": "Indonesia", "Irish": "Ireland", "Italian": "Italy",
    "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland",
    "Portuguese": "Portugal", "Rhodesian": "Rhodesia", "Russian": "Russia",
    "South African": "South Africa", "Spanish": "Spain", "Swedish": "Sweden",
    "Swiss": "Switzerland", "Thai": "Thailand", "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace(
    "United States", "USA"
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["nationality", "country", "home_country",
                      "second_home_country", "home_flag",
                      "nationality_x", "nationality_y",
                      "country_x", "country_y"]],
    errors="ignore",
)

model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)
model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

assert model["nationality"].notna().all()
assert model["country"].notna().all()
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"unmapped nationalities: {unmapped}"

# Sanity check: British drivers at Silverstone (circuit 9).
silverstone = model[(model["circuitId"] == 9) & (model["home_flag"] == 1)]
print("UK drivers flagged at Silverstone:", len(silverstone))
assert len(silverstone) > 0, "home_flag broken: no UK drivers flagged at Silverstone"
print(model["home_flag"].value_counts())

model = model.drop(
    columns=["nationality", "country", "home_country", "second_home_country"],
    errors="ignore",
)

In [ ]:
# Season progress: fraction of the season completed before this race.
season_length = model.groupby("year")["round"].transform("max")
model["season_progress"] = model["round"] / season_length

print(model["season_progress"].describe().round(3))

In [ ]:
model["era"] = pd.cut(
    model["year"],
    bins=[1949, 2005, 2008, 2013, 2021, 2024],
    labels=["pre_2006", "early_v8", "late_v8", "hybrid", "ground_effect"],
)

# Ordinal encoding so the column is numeric and usable by the pipeline.
# The model treats eras as ordered in time; this is an approximation, see
# the markdown note.
ERA_ORDINAL = {"pre_2006": 0, "early_v8": 1, "late_v8": 2, "hybrid": 3, "ground_effect": 4}
model["era_ordinal"] = model["era"].map(ERA_ORDINAL).astype("Int64")

print(model["era"].value_counts())
print("\nera and era_ordinal mapping:")
print(model[["era", "era_ordinal"]].drop_duplicates().sort_values("era_ordinal"))

assert model["era_ordinal"].notna().all(), "era mapping missed a category"
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary and feature groups

We keep two lists.

**`CANDIDATE_GROUPS`: the 20 candidate features** of the first version, in six groups. Section 3.11 tests them.

- **grid**: where the car starts.
- **qualifying**: how fast the driver was in qualifying.
- **driver_standings**: the driver's championship points and position going into the race.
- **constructor_standings**: the team's championship points and position going into the race.
- **circuit**: the driver's and team's history at this specific track.
- **context**: home race, era, and season progress.

**`FEATURE_GROUPS`: the final 12 features** (Option 1 without the redundant `qg_q1_only`), chosen by the ablation in Section 3.11:

- **grid**: `grid_fixed`
- **qualifying**: `qual_position`, `gap_to_pole_pct`, `no_qual_time`, `qg_q3`, `qg_q2_only`
- **driver_standings**: `pts_before`, `pos_before`, `standings_missing`
- **constructor_standings**: `cons_pts_before`, `cons_pos_before`, `cons_standings_missing`

Identifiers (`driverId`, `constructorId`, `circuitId`, `number`) and
keys (`raceId`, `year`, `round`, `date`, `split`) are excluded from
`FEATURES`. They are useful for joins and audit, not as model inputs: an ID
is a label, not a quantity (driver 830 is not "more" than driver 815), so a
model would learn false patterns from the numbers.

`sample_weight` is a training weight, not a feature: it is passed to the
model's `fit` (pre-2006 rows count 0.25).

**Saved for the model notebooks:** `p3_feature_groups.json` (final list),
`p3_data_dictionary.csv`, `p3_training_config.json` (start year and weights),
and `p3_modeling_table.csv` (all columns, including `sample_weight`).

In [ ]:
# The 20 candidate features of the first version. Section 3.11 tests them.
CANDIDATE_GROUPS = {
    "grid":                  ["grid_fixed", "pit_lane_flag"],
    "qualifying":            ["qual_position", "best_qual_s", "gap_to_pole_pct", "no_qual_time"],
    "driver_standings":      ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before", "cons_standings_missing"],
    "circuit":               ["circuit_starts_before", "circuit_podiums_before", "circuit_podium_rate",
                              "team_circuit_starts_before", "team_circuit_podiums_before"],
    "context":               ["home_flag", "era_ordinal", "season_progress"],
}
CANDIDATES = [f for cols in CANDIDATE_GROUPS.values() for f in cols]

# Final feature list (Option 1, chosen by the ablation in Section 3.11).
FEATURE_GROUPS = {
    "grid":                  ["grid_fixed"],
    "qualifying":            ["qual_position", "gap_to_pole_pct", "no_qual_time"] + QUAL_GROUP_COLS,
    "driver_standings":      ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before", "cons_standings_missing"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

missing = [f for f in CANDIDATES + FEATURES if f not in model.columns]
assert not missing, f"feature lists reference missing columns: {missing}"
assert "sample_weight" not in FEATURES, "the training weight is not a feature"

print(f"Candidates: {len(CANDIDATES)} features in {len(CANDIDATE_GROUPS)} groups")
print(f"Final:      {len(FEATURES)} features in {len(FEATURE_GROUPS)} groups")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {cols}")

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), \
    f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("All features are leakage-safe.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

FEATURE_SOURCE = {
    "grid_fixed": ("grid", f"Starting slot after penalties; pit-lane starts set to PIT_GRID={PIT_GRID}."),
    "pit_lane_flag": ("grid", "1 if the driver started from the pit lane."),
    "qual_position": ("qualifying", "Rank in the qualifying session."),
    "best_qual_s": ("qualifying", "Best qualifying lap across the sessions reached."),
    "gap_to_pole_pct": ("qualifying", "Percent slower than the fastest lap in the same race."),
    "no_qual_time": ("qualifying", "1 if the driver set no qualifying time at all."),
    "qg_q3": ("qualifying", "1 if the driver reached Q3 (set a Q3 time)."),
    "qg_q2_only": ("qualifying", "1 if the driver's last session was Q2."),
    "qg_q1_only": ("qualifying", "1 if the driver's last session was Q1."),
    "pts_before": ("driver_standings", "Driver points going into the race."),
    "pos_before": ("driver_standings", "Driver championship position going into the race."),
    "standings_missing": ("driver_standings", "1 if the driver has no previous standings row this season."),
    "cons_pts_before": ("constructor_standings", "Constructor points going into the race."),
    "cons_pos_before": ("constructor_standings", "Constructor championship position going into the race."),
    "cons_standings_missing": ("constructor_standings", "1 if the constructor has no previous standings row this season."),
    "pts_share": ("driver_standings", "Driver points before the race divided by the leader's points (0 to 1)."),
    "cons_pts_share": ("constructor_standings", "Constructor points before the race divided by the leading constructor's points (0 to 1)."),
    "gap_to_pole_q1_pct": ("qualifying", "Percent slower than the fastest Q1 lap in the same race."),
    "driver_form5": ("form", "Driver's podium rate over the previous 5 races."),
    "constructor_form5": ("form", "Constructor's podium share over the previous 5 races."),
    "circuit_starts_before": ("derived", "Driver's previous starts at this circuit."),
    "circuit_podiums_before": ("derived", "Driver's previous podiums at this circuit."),
    "circuit_podium_rate": ("derived", "Smoothed podium rate at this circuit."),
    "team_circuit_starts_before": ("derived", "Constructor's previous starts at this circuit."),
    "team_circuit_podiums_before": ("derived", "Constructor's previous podiums at this circuit."),
    "home_flag": ("derived", "1 if the driver's nationality matches the circuit country."),
    "era_ordinal":                         ("derived", "Technical era bucket (pre_2006, early_v8, late_v8, hybrid, ground_effect)."),
    "season_progress": ("derived", "Round divided by season length (0 to 1)."),
}

# 2. Prevent pandas from truncating text columns
pd.set_option('display.max_colwidth', None)

# 3. Loop through and display a separate table for each group
for group, cols in FEATURE_GROUPS.items():
    dict_rows = []
    for f in cols:
        # Safeguard in case a feature isn't in your dataframe yet
        if f in model.columns:
            dtype_val = str(model[f].dtype)
            missing_val = round(model[f].isna().mean(), 4)
        else:
            dtype_val = "N/A"
            missing_val = "N/A"
            
        src, note = FEATURE_SOURCE.get(f, ("unknown", "No description provided"))
        dict_rows.append({
            "feature": f,
            "note": note,
            "missing_share": missing_val,
        })
    
    group_df = pd.DataFrame(dict_rows)
    
    print(f"\nFeature Group: {group.upper()}")
    display(group_df)
    print("-" * 60)

In [ ]:
import json, os

# Build the dictionary from FEATURE_GROUPS and FEATURE_SOURCE.
dict_rows = []
for group, cols in FEATURE_GROUPS.items():
    for f in cols:
        src, note = FEATURE_SOURCE[f]
        dict_rows.append({
            "group": group,
            "feature": f,
            "source": src,
            "missing_share": round(model[f].isna().mean(), 4),
            "note": note,
        })
dictionary = pd.DataFrame(dict_rows)

# Save both artifacts.
os.makedirs("reports", exist_ok=True)

with open("reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

dictionary.to_csv("reports/p3_data_dictionary.csv", index=False)

# Training settings the model notebooks need (rows before 2006 get a lower weight).
TRAINING_CONFIG = {
    "start_year": START_YEAR,
    "full_weight_from": FULL_WEIGHT_FROM,
    "old_season_weight": OLD_SEASON_WEIGHT,
    "weight_column": "sample_weight",
    "how_to_use": "pipe.fit(X_train, y_train, <final step name>__sample_weight=train['sample_weight'])",
}
with open("reports/p3_training_config.json", "w") as f:
    json.dump(TRAINING_CONFIG, f, indent=2)

print("saved:")
print(" - reports/p3_feature_groups.json")
print(" - reports/p3_data_dictionary.csv")
print(" - reports/p3_training_config.json")
print(f" - {len(FEATURES)} features across {len(FEATURE_GROUPS)} groups")

In [ ]:
model.to_csv("reports/p3_modeling_table.csv", index=False)
print("saved modeling table with features, shape:", model.shape)

### Section 3.9: Evidence plots

We produce five plots to justify our feature choices. All of them use the full-weight training seasons (2006–2019), where every qualifying column exists:

1. **Podium rate vs grid bucket** shows that starting position predicts podium.
2. **Podium rate vs qualifying bucket** shows that qualifying position predicts podium.
3. **Feature distributions by target**, in two versions: (a) three candidate features (driver points, team points, circuit podium rate) and (b) three final features (driver and team championship position, gap to pole), showing how each separates podium from non-podium.
4. **Correlation heatmap** of all 20 candidate features checks for redundancy and confirms features relate to `podium`.
5. **Correlation with podium (bar chart)** ranks each feature by its correlation with `podium`, which is easier to read than the full heatmap.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Full-weight training seasons (2006-2019): every qualifying column exists there.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)].copy()

bins = [0, 2, 4, 6, 10, 15, PIT_GRID - 1, PIT_GRID]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+", "pit lane"]
train["grid_bucket"] = pd.cut(train["grid_fixed"], bins=bins, labels=labels, right=True)

stats = train.groupby("grid_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats["mean_pct"] = stats["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats.index.astype(str), stats["mean_pct"], color="tab:blue")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats["mean_pct"], stats["count"])], fontsize=8)
ax.set_title("Podium rate by grid position (training years 2006–2019)")
ax.set_xlabel("Grid position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         "Marginal view, not controlled for qualifying or car strength. "
         f"Pit-lane starters carry the placeholder grid value {PIT_GRID}. "
         f"Pit-lane bar is based on only {int(stats.loc['pit lane', 'count'])} starts.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("reports/fig_p3_podium_vs_grid.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats.round(3))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Full-weight training seasons (2006-2019): every qualifying column exists there.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)].copy()

bins = [0, 2, 4, 6, 10, 15, 25]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
train["qual_bucket"] = pd.cut(train["qual_position"], bins=bins, labels=labels, right=True)

n_missing = train["qual_position"].isna().sum()
stats_q = train.groupby("qual_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats_q["mean_pct"] = stats_q["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats_q.index.astype(str), stats_q["mean_pct"], color="tab:orange")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats_q["mean_pct"], stats_q["count"])], fontsize=8)
ax.set_title("Podium rate by qualifying position (training years 2006–2019)")
ax.set_xlabel("Qualifying position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats_q["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         f"Marginal view, not controlled for grid. {n_missing} training rows with no qualifying position are excluded.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("reports/fig_p3_podium_vs_qual.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats_q.round(3))

In [ ]:
# Same plot for both periods: grid matters in every era, but less before 2006.
train_all = model[model["split"] == "train"].copy()
train_all["grid_bucket"] = pd.cut(train_all["grid_fixed"], bins=bins, labels=labels, right=True)
train_all["period"] = np.where(train_all["year"] >= FULL_WEIGHT_FROM, "2006-2019 (weight 1)",
                               f"1950-2005 (weight {OLD_SEASON_WEIGHT})")

rates = (train_all.groupby(["grid_bucket", "period"], observed=True)["podium"]
                  .mean().unstack("period") * 100)

fig, ax = plt.subplots(figsize=(10, 5))
rates.plot(kind="bar", ax=ax, color=["tab:gray", "tab:blue"], width=0.8, rot=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=7)
ax.set_title("Podium rate by grid position: old vs modern seasons (training rows)")
ax.set_xlabel("Grid position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.legend(title="")
plt.tight_layout()
plt.savefig("reports/fig_p3_podium_vs_grid_by_period.png", dpi=120, bbox_inches="tight")
plt.show()

print(rates.round(1))

In [ ]:
# Plot 3a: three candidate features (points and circuit history).
import numpy as np
import matplotlib.pyplot as plt

# Full-weight training seasons (2006-2019).
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]

features = ["pts_before", "cons_pts_before", "circuit_podium_rate"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, features):
    x_no = train.loc[train["podium"] == 0, feat].dropna()
    x_yes = train.loc[train["podium"] == 1, feat].dropna()

    # Shared bin edges so the two groups are directly comparable
    edges = np.histogram_bin_edges(pd.concat([x_no, x_yes]), bins=20)

    ax.hist(x_no, bins=edges, density=True, histtype="stepfilled",
            alpha=0.35, color="tab:gray", label=f"not podium (n={len(x_no):,})")
    ax.hist(x_yes, bins=edges, density=True, histtype="step",
            linewidth=3, color="tab:red", label=f"podium (n={len(x_yes):,})")

    ax.set_title(feat)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density (within group)")
    ax.legend(fontsize=8)

fig.suptitle("Candidate features: distributions by podium outcome, training years 2006–2019 "
             "(densities normalised within each group; podium rate ≈ 14%)",
             fontsize=10)
plt.tight_layout()
plt.savefig("reports/fig_p3_distributions_candidates.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3b: three final features (championship positions and gap to pole).
import numpy as np
import matplotlib.pyplot as plt

# Full-weight training seasons (2006-2019).
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]

# Three final features that mean the same in every era (positions and a percentage).
features = [
    ("pos_before", "Driver championship position before the race"),
    ("cons_pos_before", "Team championship position before the race"),
    ("gap_to_pole_pct", "Gap to pole (%)"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (feat, title) in zip(axes, features):
    x_no = train.loc[train["podium"] == 0, feat].dropna()
    x_yes = train.loc[train["podium"] == 1, feat].dropna()

    if feat == "gap_to_pole_pct":
        # Cut the long tail so the bulk of the data is visible.
        hi = np.percentile(pd.concat([x_no, x_yes]), 99)
        x_no, x_yes = x_no.clip(upper=hi), x_yes.clip(upper=hi)
        edges = np.histogram_bin_edges(pd.concat([x_no, x_yes]), bins=20)
    else:
        # Whole-number bins, so each bar is one championship position.
        edges = np.arange(0.5, pd.concat([x_no, x_yes]).max() + 1.5, 1)

    ax.hist(x_no, bins=edges, density=True, histtype="stepfilled",
            alpha=0.35, color="tab:gray", label=f"not podium (n={len(x_no):,})")
    ax.hist(x_yes, bins=edges, density=True, histtype="step",
            linewidth=3, color="tab:red", label=f"podium (n={len(x_yes):,})")

    ax.set_title(title)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density (within group)")
    ax.legend(fontsize=8)

fig.suptitle("Final features: distributions by podium outcome, full-weight training years 2006–2019 "
             "(densities normalised within each group)", fontsize=10)
plt.tight_layout()
plt.savefig("reports/fig_p3_distributions_by_target.png", dpi=120, bbox_inches="tight")
plt.show()

# Share of each group in the front of each distribution.
for feat, cut in [("pos_before", 6), ("cons_pos_before", 3), ("gap_to_pole_pct", 1.0)]:
    yes = train.loc[train["podium"] == 1, feat].dropna()
    no = train.loc[train["podium"] == 0, feat].dropna()
    print(f"{feat} <= {cut}: podium {(yes <= cut).mean():.1%}, not podium {(no <= cut).mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlations on full-weight TRAINING rows only (2006-2019), for all 20 candidates,
# so validation and test never influence the feature choice.
train = model[(model["split"] == "train") & (model["year"] >= FULL_WEIGHT_FROM)]
print("Features in heatmap:", len(CANDIDATES))
corr = train[CANDIDATES + ["podium"]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Feature correlation heatmap (training years 2006–2019 only)")
plt.tight_layout()
plt.savefig("reports/fig_p3_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nCorrelation with podium (training rows, sorted):")
print(corr["podium"].drop("podium").sort_values(ascending=False).round(3))

In [ ]:
# Correlation of each feature with podium only (easier to read than the full heatmap)
corr_podium = corr["podium"].drop("podium").sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["tab:blue" if v < 0 else "tab:red" for v in corr_podium]
bars = ax.barh(corr_podium.index, corr_podium.values, color=colors)
ax.bar_label(bars, fmt="%+.2f", padding=3, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-0.65, 0.65)
ax.set_title("Correlation of each feature with podium (training years 2006–2019)")
ax.set_xlabel("Correlation with podium (− = lower value, more podiums)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("reports/fig_p3_corr_with_podium.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.9 findings

All five plots use the full-weight training rows only (2006–2019). The correlations are computed on the same split, so they are not influenced by validation or test.

**Plot 1 (grid).** Podium rate falls with each step down the grid:

| Grid bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 72.9% | 535 |
| 3-4 | 43.7% | 535 |
| 5-6 | 17.5% | 536 |
| 7-10 | 4.9% | 1,068 |
| 11-15 | 1.4% | 1,331 |
| 16+ | 0.9% | 1,717 |
| pit lane (35) | 0.0% | 20 |

The pit-lane group shows no podium finishes, but 20 starts are too few to estimate a rate. Pit-lane starts get `grid_fixed = 35` (`PIT_GRID`), behind every real grid slot; Section 3.11 shows that `pit_lane_flag` adds nothing on top of that.

In the seasons before 2006 the grid matters less (front row 54.1% vs 72.9%, P7–10 11.3% vs 4.9%): races were less predictable, with more retirements. This is one reason the old seasons get a lower training weight.

**Plot 2 (qualifying).** The same shape:

| Qualifying bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 71.8% | 536 |
| 3-4 | 43.2% | 535 |
| 5-6 | 20.0% | 535 |
| 7-10 | 4.3% | 1,067 |
| 11-15 | 1.6% | 1,331 |
| 16+ | 0.8% | 1,718 |

Grid and qualifying position are closely related because most drivers start where they qualify. They are not identical: pit-lane starts have a qualifying position but no grid slot, and grid penalties move some drivers back from their qualifying position. 

**Plot 3a (candidate features).**

- `pts_before`: podium drivers have more pre-race points. The podium
  group has a long right tail; the non-podium group is concentrated near
  zero. Clear separation.
- `cons_pts_before`: the same pattern, slightly stronger. The podium
  distribution is shifted further right than the driver version,
  consistent with car performance being the dominant factor.
- `circuit_podium_rate`: the podium group is shifted slightly right, but
  the overlap is large. The feature has signal but is weaker than the
  standings (the circuit group is dropped in Section 3.11).

**Plot 3b (final features).**

- `pos_before`: 87.1% of podium finishers were in the top 6 of the
  championship before the race, against 18.6% of the other drivers.
- `cons_pos_before`: 84.0% of podium finishers drove for a top-3 team,
  against 18.9% of the others, consistent with car performance being the
  dominant factor.
- `gap_to_pole_pct`: 83.9% of podium finishers qualified within 1% of the
  fastest lap, against 19.6% of the others.

**Plot 4 (correlations with podium, training rows only).**

The ranking matches the expected structure. Grid and qualifying are the
strongest features (|r| ≈ 0.50). Driver and constructor standings follow
(|r| ≈ 0.44–0.47). Circuit features are moderate (0.10–0.29). The flags
and `season_progress` are near zero.

- `circuit_podium_rate` (0.285) is stronger than the raw counts
  (`circuit_podiums_before` 0.250, `circuit_starts_before` 0.168),
  supporting the smoothed rate over the counts.
- `best_qual_s` (-0.057) is weak because absolute lap times vary by
  track; `gap_to_pole_pct` (-0.290) is the normalised version and carries
  five times the signal.
- `season_progress` has zero correlation with podium. It was meant to
  help interpret `pts_before` (50 points at round 3 means more than at
  round 20), but logistic regression cannot combine two features on its
  own, so this effect is not captured. It is tested on its own in the
  ablation (Section 3.11).
  `era_ordinal` is 0.012, essentially zero. Era is a time bucket, not a
per-race property, so a linear correlation cannot capture it. It is
tested in the ablation, with the limitation noted in Section 3.7
(validation contains only the hybrid era).
- The rare-event flags (`no_qual_time`, `pit_lane_flag`,
  `*_missing`) have small correlations because they fire on few rows.
  Correlation is the wrong measure for them; the ablation is the right
  one.

No feature pair exceeds 0.95 correlation, so no two features are
redundant enough to drop on correlation alone. `grid_fixed` and
`qual_position` are the highest pair at ~0.9, and both stay: they carry
distinct information, and the ablation (Section 3.11) shows that removing
the grid group lowers validation PR-AUC.

**Plot 5 (correlation with podium, bar chart).** The same correlations as
the `podium` column of the heatmap, sorted, so the ranking is easier to read.

### Section 3.10: Preprocessing experiments

We run four experiments, each with the same logistic regression as the
baseline. Only the preprocessing step changes.

- **1a: scaling vs no scaling.** Does `StandardScaler` change performance
  or convergence?
- **1b: order (impute→scale vs scale→impute).** Does the order matter?
- **2: median vs mean imputation.** Which fill value is better for this
  feature set?
- **3: fit-on-train vs fit-on-all (leakage demo).** Does fitting the
  preprocessor on validation rows inflate the score?

All experiments use the final feature list, the same train/validation split,
the training weights (`sample_weight`, pre-2006 rows 0.25) and the same
`LogisticRegression(max_iter=2000, random_state=SEED)`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# Ensure splits are defined 
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

X_tr = train_df[FEATURES];  y_tr = train_df["podium"].values
w_tr = train_df["sample_weight"].values   # pre-2006 rows count less
X_va = val_df[FEATURES];    y_va = val_df["podium"].values

print("X_tr:", X_tr.shape, " X_va:", X_va.shape)
print("\n")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def lr_pipeline(scale: bool):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("lr", LogisticRegression(max_iter=2000, random_state=SEED)))
    return Pipeline(steps)

def run_and_count_warnings(pipe, X_fit, y_fit, X_eval, y_eval, w_fit=None):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        pipe.fit(X_fit, y_fit, lr__sample_weight=w_fit)
        proba = pipe.predict_proba(X_eval)[:, 1]
    n_warn = sum(1 for x in w if issubclass(x.category, ConvergenceWarning))
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
        "warnings": n_warn,
    }

import pandas as pd
rows = []
for name, scale, iters in [("impute→scale", True, 2000),
                           ("impute-only", False, 2000),
                           ("impute-only (max_iter=10000)", False, 10000)]:
    pipe = lr_pipeline(scale=scale)
    pipe.set_params(lr__max_iter=iters)
    r = run_and_count_warnings(pipe, X_tr, y_tr, X_va, y_va, w_fit=w_tr)
    rows.append({"pipeline": name, **r, "steps used": pipe.named_steps["lr"].n_iter_[0]})

exp1a = pd.DataFrame(rows)
print("Experiment 1a: scaling vs no scaling (validation)")
print(exp1a.to_string(index=False))

In [ ]:
pipe_imp_sc = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipe_sc_imp = Pipeline([
    ("sc",  StandardScaler()),
    ("imp", SimpleImputer(strategy="median")),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])

print("Experiment 1b: order of imputation and scaling")
for name, pipe in [("impute→scale", pipe_imp_sc), ("scale→impute", pipe_sc_imp)]:
    pipe.fit(X_tr, y_tr, lr__sample_weight=w_tr)
    pr = average_precision_score(y_va, pipe.predict_proba(X_va)[:, 1])
    print(f"  {name}: validation PR-AUC = {pr:.4f}")

In [ ]:
def lr_pipeline_imputer(strategy: str):
    return Pipeline([
        ("imputer", SimpleImputer(strategy=strategy)),
        ("scaler",  StandardScaler()),
        ("lr",      LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

rows = []
for name, strategy in [("median", "median"), ("mean", "mean")]:
    pipe = lr_pipeline_imputer(strategy)
    pipe.fit(X_tr, y_tr, lr__sample_weight=w_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    rows.append({
        "imputer": name,
        "PR-AUC":  round(average_precision_score(y_va, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_va, proba), 4),
    })

exp1c = pd.DataFrame(rows)
print("Experiment 2: median vs mean imputation (validation)")
print(exp1c.to_string(index=False))

In [ ]:
median_imp = SimpleImputer(strategy="median").fit(X_tr)
mean_imp   = SimpleImputer(strategy="mean").fit(X_tr)

compare = pd.DataFrame({
    "feature":       FEATURES,
    "median_fill":   median_imp.statistics_.round(3),
    "mean_fill":     mean_imp.statistics_.round(3),
    "diff":          (mean_imp.statistics_ - median_imp.statistics_).round(3),
    "missing_share": [round(X_tr[f].isna().mean(), 4) for f in FEATURES],
})
print(compare[compare["missing_share"] > 0].to_string(index=False))

In [ ]:
# DEMO ONLY: the "leaky" version below shows what NOT to do.
# The real pipeline (used everywhere else) is fitted on train only.
from sklearn.base import clone

prep_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

def lr_on(prep, X, y):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(
        prep.transform(X), y, sample_weight=w_tr)

prep_correct = clone(prep_template).fit(X_tr)
clf_correct  = lr_on(prep_correct, X_tr, y_tr)
proba_correct = clf_correct.predict_proba(prep_correct.transform(X_va))[:, 1]

X_tr_va = pd.concat([X_tr, X_va])
prep_leaky = clone(prep_template).fit(X_tr_va)
clf_leaky  = lr_on(prep_leaky, X_tr, y_tr)
proba_leaky = clf_leaky.predict_proba(prep_leaky.transform(X_va))[:, 1]

pr_correct = average_precision_score(y_va, proba_correct)
pr_leaky   = average_precision_score(y_va, proba_leaky)

print("Experiment 3: preprocessing fit on train vs train+val")
print(f"  Correct (train only) : PR-AUC = {pr_correct:.4f}")
print(f"  Leaky   (train+val)  : PR-AUC = {pr_leaky:.4f}")
print(f"  Difference           : {pr_leaky - pr_correct:+.4f}")

scaler_correct = prep_correct.named_steps["scaler"]
scaler_leaky   = prep_leaky.named_steps["scaler"]
mean_shift_std  = np.abs(scaler_correct.mean_ - scaler_leaky.mean_) / scaler_correct.scale_
scale_shift_pct = np.abs(scaler_correct.scale_ - scaler_leaky.scale_) / scaler_correct.scale_ * 100

print(f"\nScaler drift: mean shift {mean_shift_std.mean():.4f} std (max {mean_shift_std.max():.4f})")
print(f"              scale change {scale_shift_pct.mean():.2f}% (max {scale_shift_pct.max():.2f}%)")

### 3.10 findings

All experiments use the final 12 features, training rows 1950–2019 with the
season weights, and validation 2020–2021.

**Experiment 1a: scaling vs no scaling.**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings | Iterations used |
|---|---:|---:|---:|---:|
| impute→scale | 0.7351 | 0.9273 | 0 | 30 |
| impute-only | 0.7351 | 0.9272 | 0 | 696 |
| impute-only (max_iter=10000) | 0.7351 | 0.9272 | 0 | 696 |

Both pipelines converge and score the same. We keep `impute→scale`
because it converges about 23 times faster (30 vs 696 iterations), makes
the L2 penalty treat all features equally (otherwise a feature in the
hundreds, like `pts_before`, is penalised differently from a 0/1 flag), and
is required by the other models we train (e.g. neural networks).

**Experiment 1b: order of imputation and scaling.**

| Order | Validation PR-AUC |
|---|---:|
| impute→scale | 0.7351 |
| scale→impute | 0.7349 |

The same to 3 decimals. We use `impute→scale`, the standard scikit-learn
order, which produces a final distribution with mean 0 and std 1.

**Experiment 2: median vs mean imputation (validation).**

| Feature | Median fill | Mean fill | Difference | Missing in train |
|---|---:|---:|---:|---:|
| `qual_position` | 11.000 | 11.345 | +0.345 | 63.1% |
| `gap_to_pole_pct` | 1.961 | 2.503 | +0.542 | 63.7% |
| `pos_before` | 12.000 | 13.072 | +1.072 | 12.4% |
| `cons_pos_before` | 6.000 | 6.162 | +0.162 | 12.4% |

Validation PR-AUC is identical for both (0.7351 / 0.7351). The missing
shares are high because of the old seasons (no qualifying data before 1994,
no constructors' championship before 1958). Those rows are marked by
`no_qual_time` and the standings flags and have weight 0.25, while the
validation rows (2020–21) are almost never missing, so the fill value hardly
moves the score. We choose **median**, the standard choice for skewed
features (`gap_to_pole_pct` is right-skewed). Note that `SimpleImputer` is
not weighted: the medians are computed over all training rows.

**Experiment 3: fit on train vs train+val (leakage demo).**

| Setup | Validation PR-AUC |
|---|---:|
| Fit on train only (correct) | 0.7351 |
| Fit on train+val (leaky) | 0.7350 |
| Difference | −0.0001 |

The metric impact is small, but the scaler's parameters drift measurably:
per-feature mean shift 0.0188 std on average (max 0.0433), scale change
2.24% on average (max 5.45%). The leak is real. We fit on train only because
it is the honest rule and the effect could be larger under a bigger
distribution shift.

### Section 3.11: Feature ablation

We measure what each feature and each feature group adds by changing the
feature set and retraining the same model. Everything else stays fixed: the
same pipeline (median imputation, scaling, logistic regression), the same
training data (1950 onwards, pre-2006 rows weighted 0.25) and the same
metric (validation PR-AUC from `evaluate()` in `src/f1_eval.py`). The model
is fitted here with the weights, then scored with `evaluate()`, because
`train_eval()` does not take weights.

**Three validation periods.** A single validation period can move by about
0.001 by chance, so every test is run on three expanding time folds. The test
years (2022+) are never used.

| Fold | Train | Validate |
|---|---|---|
| 1 | 1950–2015 | 2016–2017 |
| 2 | 1950–2017 | 2018–2019 |
| 3 | 1950–2019 | 2020–2021 (the official validation split) |

**Feature lists.** `CANDIDATE_GROUPS` (20 candidates) and `FEATURE_GROUPS`
(final list) are defined in Section 3.8.

- **Part A: group ablation.** Remove one candidate group at a time, plus grid and
  qualifying together (they are about 0.9 correlated, so each can cover for the other).
- **Part B: single-feature tests**, starting from the 20 candidates: *remove*
  (is the feature needed?), *swap* (is a new version better?) and *add*
  (does a new candidate help?).
- **Part D: three focused questions**: grid encoding, qualifying rank vs round
  reached, season points vs last-5-races form.
- **Part F: using the seasons before 2006**: dropped, full weight, or down-weighted.
- **Part C: the final list** against the 20 candidates, and a group ablation on it.
- **Part E: extra checks** on the final list: add each unused column, three new feature
  ideas (teammate gap, reliability, places gained), every subset of the final list, and a
  grid fallback for the missing qualifying positions of the old seasons.

**Decision rule.** A change counts only if it moves mean PR-AUC by more than
`MIN_GAIN = 0.002` and goes the same way in at least 2 of the 3 folds.
Otherwise the simpler feature set wins.

**Limitation for `era_ordinal`.** No fold contains the ground-effect era, so
the ablation can only show its effect on earlier eras, not whether the test
value (4) is safe.

In [ ]:
import sys
# copied from src/f1_eval.py
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5, sample_weight=None):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`.
    `sample_weight` (optional) weights the rows, as in training."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    w = sample_weight
    return {
        "ROC-AUC": roc_auc_score(y, proba, sample_weight=w),
        "PR-AUC": average_precision_score(y, proba, sample_weight=w),
        "F1": f1_score(y, pred, sample_weight=w, zero_division=0),
        "precision": precision_score(y, pred, sample_weight=w, zero_division=0),
        "recall": recall_score(y, pred, sample_weight=w, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model", sample_weight=None):
    """Fit `model` on the training years only, then score it on train and validation.
    `sample_weight` (optional): one weight per training row (P3's `sample_weight` column). It is passed to fit
    (to the last step of a pipeline) and used for the training scores. Validation rows are not weighted.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    if sample_weight is None:
        model.fit(X_train, y_train)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        key = f"{model.steps[-1][0]}__sample_weight" if hasattr(model, "steps") else "sample_weight"
        model.fit(X_train, y_train, **{key: sample_weight})
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold, sample_weight)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)


import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# CANDIDATE_GROUPS / CANDIDATES (20) and FEATURE_GROUPS / FEATURES (final) come from Section 3.8.
abl = model.copy()

# Three expanding time folds. Training starts in START_YEAR; test years (2022+) are never used.
FOLDS = [
    ("2016-17", abl["year"] <= 2015, abl["year"].between(2016, 2017)),
    ("2018-19", abl["year"] <= 2017, abl["year"].between(2018, 2019)),
    ("2020-21", abl["year"] <= 2019, abl["year"].between(2020, 2021)),
]
FOLD_NAMES = [name for name, _, _ in FOLDS]
MIN_GAIN = 0.002

def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

def cv_pr_auc(features, start=START_YEAR, old_weight=OLD_SEASON_WEIGHT):
    """Validation PR-AUC on each of the three folds.
    Training rows from `start`; rows before FULL_WEIGHT_FROM get `old_weight`."""
    scores = []
    for name, tr, va in FOLDS:
        tr = tr & (abl["year"] >= start)
        w = np.where(abl.loc[tr, "year"] < FULL_WEIGHT_FROM, old_weight, 1.0)
        pipe = make_pipeline()
        pipe.fit(abl.loc[tr, features], abl.loc[tr, "podium"].values, lr__sample_weight=w)
        scores.append(evaluate(pipe, abl.loc[va, features], abl.loc[va, "podium"].values)["PR-AUC"])
    return np.array(scores)

base_scores = cv_pr_auc(CANDIDATES)
print(f"Training from {START_YEAR}, pre-{FULL_WEIGHT_FROM} weight {OLD_SEASON_WEIGHT}")
print(f"Candidates: {len(CANDIDATES)} features in {len(CANDIDATE_GROUPS)} groups")
for name, s in zip(FOLD_NAMES, base_scores):
    print(f"  baseline PR-AUC, validate {name}: {s:.4f}")
print(f"  mean: {base_scores.mean():.4f}")

In [ ]:
# Part A: remove one group at a time (plus grid and qualifying together).
# drop = baseline - score without the group, so a positive drop means the group helps.
group_tests = dict(CANDIDATE_GROUPS)
group_tests["grid + qualifying"] = CANDIDATE_GROUPS["grid"] + CANDIDATE_GROUPS["qualifying"]

rows = []
for group, cols in group_tests.items():
    drop = base_scores - cv_pr_auc([f for f in CANDIDATES if f not in cols])
    rows.append({
        "dropped group": group,
        **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
        "mean drop": drop.mean(),
        "folds where it helps": int((drop > 0).sum()),
    })

group_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print(group_ablation.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Mean drop per group, with each fold as a dot so the spread is visible.
plot_df = group_ablation.sort_values("mean drop")
fold_cols = [f"drop {n}" for n in FOLD_NAMES]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["dropped group"], plot_df["mean drop"], color="tab:blue", alpha=0.8,
        label="mean over 3 folds")
for k, col in enumerate(fold_cols):
    ax.scatter(plot_df[col], plot_df["dropped group"], color="black", s=14,
               marker="o", zorder=3, label="single fold" if k == 0 else None)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xscale("symlog", linthresh=0.005)
ax.set_title("Group ablation: PR-AUC drop when the group is removed (3 validation folds)")
ax.set_xlabel("PR-AUC drop vs baseline (larger = more important; symlog scale)")
ax.grid(alpha=0.3, axis="x")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("reports/fig_p3_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Part B: single-feature tests, each starting from the 20 candidates.
# gain = score after the change - baseline, so a positive gain means the change helps.
def without(*cols):
    return [f for f in CANDIDATES if f not in cols]

def swap(old, new):
    return [new if f == old else f for f in CANDIDATES]

TESTS = [
    ("remove", "best_qual_s",                  without("best_qual_s")),
    ("remove", "raw circuit counts (2)",       without("circuit_starts_before", "circuit_podiums_before")),
    ("remove", "circuit_podium_rate",          without("circuit_podium_rate")),
    ("remove", "team circuit history (2)",     without("team_circuit_starts_before", "team_circuit_podiums_before")),
    ("remove", "home_flag",                    without("home_flag")),
    ("remove", "season_progress",              without("season_progress")),
    ("remove", "era_ordinal",                  without("era_ordinal")),
    ("swap",   "pts_before -> pts_share",      swap("pts_before", "pts_share")),
    ("swap",   "cons_pts_before -> cons_pts_share", swap("cons_pts_before", "cons_pts_share")),
    ("swap",   "gap_to_pole_pct -> gap_to_pole_q1_pct", swap("gap_to_pole_pct", "gap_to_pole_q1_pct")),
    ("add",    "qual_group (2 yes/no columns)", CANDIDATES + QUAL_GROUP_COLS),
    ("add",    "driver_form5",                 CANDIDATES + ["driver_form5"]),
    ("add",    "constructor_form5",            CANDIDATES + ["constructor_form5"]),
    ("add",    "both form features",           CANDIDATES + ["driver_form5", "constructor_form5"]),
]

def decide(kind, gain):
    clear_gain = gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2
    clear_loss = gain.mean() < -MIN_GAIN and (gain < 0).sum() >= 2
    if kind == "remove":
        return "keep" if clear_loss else "drop"
    if kind == "swap":
        return "use new" if clear_gain else "keep current"
    return "add" if clear_gain else "do not add"

rows = []
for kind, name, features in TESTS:
    gain = cv_pr_auc(features) - base_scores
    rows.append({
        "test": kind,
        "change": name,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "decision": decide(kind, gain),
    })

feature_tests = pd.DataFrame(rows)
print(feature_tests.round(4).to_string(index=False))

In [ ]:
# Part D: focused comparisons, each starting from the 20 candidates.
# Grid encoding: is the fix (pit lane = 25), the flag, or both doing the work?
# Rank vs round reached: qual_position, qual_group, or both?
# Points vs recent form: season points, last-5-races form, or both?
# (pos_before / cons_pos_before stay in every variant; only the points columns change.)
def replace(features, old, new):
    out = [f for f in features if f not in old]
    return out + [f for f in new if f not in out]

COMPARISONS = {
    "Grid encoding": [
        ("grid_fixed + pit_lane_flag (current)", CANDIDATES),
        ("grid_fixed only",                      without("pit_lane_flag")),
        ("raw grid (0 = pit lane) + pit_lane_flag", replace(CANDIDATES, ["grid_fixed"], ["grid"])),
        ("raw grid only (original version)",     replace(CANDIDATES, ["grid_fixed", "pit_lane_flag"], ["grid"])),
    ],
    "Qualifying rank vs round reached": [
        ("qual_position only (current)",         CANDIDATES),
        ("qual_group only",                      replace(CANDIDATES, ["qual_position"], QUAL_GROUP_COLS)),
        ("qual_position + qual_group",           CANDIDATES + QUAL_GROUP_COLS),
    ],
    "Points vs recent form": [
        ("season points (current)",              CANDIDATES),
        ("driver: form instead of points",       replace(CANDIDATES, ["pts_before"], ["driver_form5"])),
        ("team: form instead of points",         replace(CANDIDATES, ["cons_pts_before"], ["constructor_form5"])),
        ("both: form instead of points",         replace(CANDIDATES, ["pts_before", "cons_pts_before"],
                                                         ["driver_form5", "constructor_form5"])),
        ("points + form",                        CANDIDATES + ["driver_form5", "constructor_form5"]),
    ],
}

comparison_tables = {}
for topic, variants in COMPARISONS.items():
    rows = []
    for name, features in variants:
        scores = cv_pr_auc(features)
        gain = scores - base_scores
        rows.append({
            "variant": name,
            **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
            "mean PR-AUC": scores.mean(),
            "gain vs current": gain.mean(),
            "folds better": int((gain > 0).sum()),
        })
    comparison_tables[topic] = pd.DataFrame(rows)
    print(f"\n{topic}")
    print(comparison_tables[topic].round(4).to_string(index=False))

In [ ]:
# Part F: using the seasons before 2006.
# Same 3 folds and features; only the training years and the weight of pre-2006 rows change.
SEASON_SETUPS = [
    ("2006-2019 only (old seasons dropped)",    FULL_WEIGHT_FROM, 1.0),
    ("1950-2019, no weights",                   1950, 1.0),
    ("1950-2019, pre-2006 weight 0.5",          1950, 0.5),
    ("1950-2019, pre-2006 weight 0.25 (used)",  1950, 0.25),
    ("1950-2019, pre-2006 weight 0.1",          1950, 0.1),
]

season_tables = {}
for set_name, features in [("20 candidates", CANDIDATES), ("Final list", FEATURES)]:
    rows = []
    for name, start, w_old in SEASON_SETUPS:
        scores = cv_pr_auc(features, start=start, old_weight=w_old)
        rows.append({"training data": name,
                     **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
                     "mean PR-AUC": scores.mean()})
    table = pd.DataFrame(rows)
    no_w = table.loc[table["training data"] == "1950-2019, no weights", "mean PR-AUC"].iloc[0]
    table["vs 1950 no weights"] = table["mean PR-AUC"] - no_w
    season_tables[set_name] = table
    print(f"\n{set_name} ({len(features)} features)")
    print(table.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Part F chart: mean PR-AUC for each way of using the old seasons (final list).
t = season_tables["Final list"]
fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ["tab:gray", "tab:red", "tab:blue", "tab:blue", "tab:blue"]
bars = ax.barh(t["training data"], t["mean PR-AUC"], color=colors)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(t["mean PR-AUC"].min() - 0.01, t["mean PR-AUC"].max() + 0.006)
ax.set_title("Final list: mean validation PR-AUC over 3 periods, by training data")
ax.set_xlabel("Mean PR-AUC (zoomed axis)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("reports/fig_p3_seasons_weights.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Part C: the final list (FEATURE_GROUPS, Option 1) against the 20 candidates,
# then a group ablation on the final list.
final_scores = cv_pr_auc(FEATURES)
compare_final = pd.DataFrame({
    "feature set": [f"20 candidates", f"Final list ({len(FEATURES)})"],
    **{f"PR-AUC {n}": [b, f] for n, b, f in zip(FOLD_NAMES, base_scores, final_scores)},
    "mean PR-AUC": [base_scores.mean(), final_scores.mean()],
})
print(compare_final.round(4).to_string(index=False))

rows = []
for group, cols in FEATURE_GROUPS.items():
    drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in cols])
    rows.append({"dropped group": group,
                 **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
                 "mean drop": drop.mean(), "folds where it helps": int((drop > 0).sum())})
gq = FEATURE_GROUPS["grid"] + FEATURE_GROUPS["qualifying"]
drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in gq])
rows.append({"dropped group": "grid + qualifying",
             **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
             "mean drop": drop.mean(), "folds where it helps": int((drop > 0).sum())})
final_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print("\nGroup ablation on the final list:")
print(final_ablation.round(4).to_string(index=False))

In [ ]:
# Part C (continued): remove each final feature on its own.
# drop = final score - score without the feature, so a positive drop means the feature helps.
# The two qual_group columns are also removed together, since they describe one thing.
single_tests = [(f, [f]) for f in FEATURES] + [("qg_q3 + qg_q2_only (removed together)", QUAL_GROUP_COLS)]

rows = []
for name, cols in single_tests:
    drop = final_scores - cv_pr_auc([f for f in FEATURES if f not in cols])
    group = next((g for g, fs in FEATURE_GROUPS.items() if cols[0] in fs), "")
    rows.append({
        "removed feature": name,
        "group": group,
        **{f"drop {n}": d for n, d in zip(FOLD_NAMES, drop)},
        "mean drop": drop.mean(),
        "helps in": f"{int((drop > 0).sum())} of 3",
        "verdict": ("clearly needed" if drop.mean() > MIN_GAIN and (drop > 0).sum() >= 2
                    else "small or shared"),
    })

feature_ablation = pd.DataFrame(rows).sort_values("mean drop", ascending=False)
print(f"Final list: {len(FEATURES)} features, mean PR-AUC {final_scores.mean():.4f}\n")
print(feature_ablation.round(4).to_string(index=False))

In [ ]:
# Part E1: add each unused column of the table to the final list, one at a time.
# Includes the dropped candidates, the extra candidates, raw columns and IDs
# (IDs are labels, not quantities, so they are only here to show they add nothing).
UNUSED = [c for c in CANDIDATES + ["pts_share", "cons_pts_share", "gap_to_pole_q1_pct",
                                   "driver_form5", "constructor_form5"] if c not in FEATURES]
UNUSED += ["grid", "q1_s", "q2_s", "q3_s", "year", "round", "number",
           "driverId", "constructorId", "circuitId"]

rows = []
for col in UNUSED:
    gain = cv_pr_auc(FEATURES + [col]) - final_scores
    rows.append({
        "added column": col,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "folds better": int((gain > 0).sum()),
        "decision": "add" if gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2 else "do not add",
    })
add_one = pd.DataFrame(rows).sort_values("mean gain", ascending=False)
print(f"Final list: {len(FEATURES)} features, mean PR-AUC {final_scores.mean():.4f}\n")
print(add_one.round(4).to_string(index=False))

In [ ]:
# Part E2: three new feature ideas, built here as candidates only (on the ablation copy).
# - qual_gap_to_teammate: the driver's gap to pole minus the teammate's, same race (pre-race).
# - team_dnf_rate10 / driver_dnf_rate10: share of the previous 10 races not finished.
# - pos_gained_avg10: average places gained (grid - finish) over the previous 10 finished races.
# The last three use race results (status, finishing order), so they are shifted by one race
# before the rolling window: a row only ever sees earlier races.
team = abl.groupby(["raceId", "constructorId"])["gap_to_pole_pct"]
others_sum = team.transform("sum") - abl["gap_to_pole_pct"].fillna(0)
others_n = team.transform("count") - abl["gap_to_pole_pct"].notna().astype(int)
abl["qual_gap_to_teammate"] = abl["gap_to_pole_pct"] - (others_sum / others_n.replace(0, np.nan))

hist = base[["raceId", "driverId", "constructorId", "year", "round", "grid"]].merge(
    outcomes[["raceId", "driverId", "positionOrder", "status"]],
    on=["raceId", "driverId"], how="left", validate="one_to_one")
finished = hist["status"].eq("Finished") | hist["status"].str.match(r"^\+\d+ Laps?$", na=False)
hist["dnf"] = (~finished).astype(float)
hist["places_gained"] = (hist["grid"] - hist["positionOrder"]).where(finished & (hist["grid"] > 0))

team_races = (hist.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["dnf"].mean()
                  .sort_values(["constructorId", "year", "round"]).reset_index(drop=True))
team_races["team_dnf_rate10"] = team_races.groupby("constructorId")["dnf"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())

hist = hist.sort_values(["driverId", "year", "round"]).reset_index(drop=True)
hist["driver_dnf_rate10"] = hist.groupby("driverId")["dnf"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())
hist["pos_gained_avg10"] = hist.groupby("driverId")["places_gained"].transform(
    lambda s: s.shift(1).rolling(10, min_periods=3).mean())

# Leakage checks: no history on a first race.
assert team_races.loc[team_races.groupby("constructorId").cumcount() == 0, "team_dnf_rate10"].isna().all()
first_d = hist.groupby("driverId").cumcount() == 0
assert hist.loc[first_d, ["driver_dnf_rate10", "pos_gained_avg10"]].isna().all().all()

abl = abl.drop(columns=["team_dnf_rate10", "driver_dnf_rate10", "pos_gained_avg10"], errors="ignore")
abl = (abl.merge(team_races[["raceId", "constructorId", "team_dnf_rate10"]],
                 on=["raceId", "constructorId"], how="left", validate="many_to_one")
          .merge(hist[["raceId", "driverId", "driver_dnf_rate10", "pos_gained_avg10"]],
                 on=["raceId", "driverId"], how="left", validate="one_to_one"))

NEW_IDEAS = {
    "qual_gap_to_teammate": ["qual_gap_to_teammate"],
    "team_dnf_rate10": ["team_dnf_rate10"],
    "driver_dnf_rate10": ["driver_dnf_rate10"],
    "pos_gained_avg10": ["pos_gained_avg10"],
    "all four": ["qual_gap_to_teammate", "team_dnf_rate10", "driver_dnf_rate10", "pos_gained_avg10"],
}
rows = []
for name, cols in NEW_IDEAS.items():
    gain = cv_pr_auc(FEATURES + cols) - final_scores
    rows.append({
        "added": name,
        **{f"gain {n}": g for n, g in zip(FOLD_NAMES, gain)},
        "mean gain": gain.mean(),
        "folds better": int((gain > 0).sum()),
        "decision": "add" if gain.mean() > MIN_GAIN and (gain > 0).sum() >= 2 else "do not add",
    })
new_ideas = pd.DataFrame(rows)
print(new_ideas.round(4).to_string(index=False))

In [ ]:
# Part E3: every subset of the final list (the two qg columns count as one unit),
# to check that no smaller combination clearly beats it. Takes a few minutes.
import itertools

UNITS = {f: [f] for f in FEATURES if f not in QUAL_GROUP_COLS}
UNITS["qg_q3 + qg_q2_only"] = QUAL_GROUP_COLS
unit_names = list(UNITS)

rows = []
for k in range(1, len(unit_names) + 1):
    for combo in itertools.combinations(unit_names, k):
        scores = cv_pr_auc([c for u in combo for c in UNITS[u]])
        rows.append({
            "dropped": ", ".join(u for u in unit_names if u not in combo) or "(none: final list)",
            **{f"PR-AUC {n}": s for n, s in zip(FOLD_NAMES, scores)},
            "mean PR-AUC": scores.mean(),
            "vs final": scores.mean() - final_scores.mean(),
            "folds better": int((scores > final_scores).sum()),
        })

subsets = pd.DataFrame(rows).sort_values("mean PR-AUC", ascending=False).reset_index(drop=True)
rank = int(subsets.index[subsets["dropped"] == "(none: final list)"][0]) + 1
clear_winners = subsets[(subsets["vs final"] > MIN_GAIN) & (subsets["folds better"] >= 2)]

print(f"Subsets tested: {len(subsets)}  |  rank of the final list by mean: {rank}")
print(f"Subsets beating the final list by more than {MIN_GAIN} in at least 2 folds: {len(clear_winners)}\n")
print("Top 10 by mean PR-AUC:")
print(subsets.head(10).round(4).to_string(index=False))
best_2021 = subsets.sort_values("PR-AUC 2020-21", ascending=False).head(3)
print("\nTop 3 on the official validation period (2020-21):")
print(best_2021.round(4).to_string(index=False))

In [ ]:
# Part E4: grid fallback for the missing qualifying positions of the old seasons.
# Before 2006 most rows have no qualifying position; today they get the median (via the
# imputer) plus no_qual_time = 1. Alternative: use the starting grid slot instead.
abl["qual_position_grid_fallback"] = abl["qual_position"].fillna(
    abl["grid"].where(abl["grid"] > 0)).astype(float)

train_rows = abl["year"] <= 2019
print(f"qual_position missing in training rows: {abl.loc[train_rows, 'qual_position'].isna().mean():.1%} "
      f"-> with grid fallback: {abl.loc[train_rows, 'qual_position_grid_fallback'].isna().mean():.1%}")

fallback_scores = cv_pr_auc([("qual_position_grid_fallback" if f == "qual_position" else f)
                             for f in FEATURES])
grid_fallback = pd.DataFrame({
    "qual_position filled with": ["median + no_qual_time flag (final list)", "grid slot (fallback)"],
    **{f"PR-AUC {n}": [a, b] for n, a, b in zip(FOLD_NAMES, final_scores, fallback_scores)},
    "mean PR-AUC": [final_scores.mean(), fallback_scores.mean()],
})
print(grid_fallback.round(4).to_string(index=False))

### 3.11 findings

All results: 3 validation periods, training from 1950 with pre-2006 rows
weighted 0.25, logistic regression. Drop / gain = change in validation PR-AUC.

**Part A: group ablation** (20 candidates, baseline mean 0.7248).

| Dropped group | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Helps in |
|---|---:|---:|---:|---:|---:|
| grid + qualifying | +0.1041 | +0.0934 | +0.0948 | +0.0974 | 3 of 3 |
| grid | +0.0288 | +0.0193 | +0.0137 | +0.0206 | 3 of 3 |
| driver_standings | −0.0136 | +0.0141 | +0.0400 | +0.0135 | 2 of 3 |
| qualifying | +0.0014 | +0.0114 | +0.0058 | +0.0062 | 3 of 3 |
| context | +0.0038 | +0.0005 | +0.0044 | +0.0029 | 3 of 3 |
| circuit | −0.0022 | +0.0030 | +0.0046 | +0.0018 | 2 of 3 |
| constructor_standings | −0.0021 | +0.0013 | +0.0039 | +0.0010 | 2 of 3 |

Grid and qualifying cover for each other: removed one at a time they cost
0.021 and 0.006, removed together 0.097.

**Part B: single-feature tests.**

- *Remove*: `best_qual_s`, the raw circuit counts, `circuit_podium_rate`,
  team circuit history, `home_flag` and `era_ordinal` are not needed
  (|mean change| ≤ 0.0017). Only `season_progress` passes the rule: removing it
  costs 0.0024, in all 3 periods.
- *Swap*: `pts_share` (−0.0030), `cons_pts_share` (−0.0033) and the Q1-only gap
  (−0.0018) are worse than the current versions.
- *Add*: `qual_group` +0.0007 (a tie); the form features hurt (−0.0034 to −0.0058).

**Part D: focused questions.**

- *Grid encoding*: `grid_fixed` alone scores the same as `grid_fixed` + `pit_lane_flag`
  (0.7248); without any pit-lane fix (the original bug) the score drops by 0.0039.
- *Rank vs round reached*: `qual_group` only +0.0004, both +0.0007: a tie.
- *Points vs form*: season points win; form instead of points costs 0.0053–0.0080.

**Part F: using the seasons before 2006** (final list).

| Training data | 2016-17 | 2018-19 | 2020-21 | Mean | vs 1950 no weights |
|---|---:|---:|---:|---:|---:|
| 2006–2019 only (old seasons dropped) | 0.7336 | 0.7250 | 0.7415 | 0.7334 | +0.0131 |
| 1950–2019, no weights | 0.7360 | 0.6971 | 0.7278 | 0.7203 | — |
| 1950–2019, pre-2006 weight 0.5 | 0.7348 | 0.7017 | 0.7309 | 0.7225 | +0.0022 |
| **1950–2019, pre-2006 weight 0.25 (used)** | 0.7358 | 0.7045 | 0.7351 | **0.7251** | **+0.0049** |
| 1950–2019, pre-2006 weight 0.1 | 0.7355 | 0.7055 | 0.7346 | 0.7252 | +0.0049 |

Using every season at full weight is the worst option (0.7203). Down-weighting
the old seasons recovers part of the loss: weight 0.25 is 0.0049 above no
weights, and weight 0.1 gives the same. Dropping the old seasons still
scores highest on validation (+0.0083 above weight 0.25): they have no
qualifying data and come from a different version of the sport, and they help
the oldest validation period (2016–17) but not the two most recent ones. We
keep every season with weight 0.25, so no data is discarded, at a cost of
0.008 mean PR-AUC compared with dropping them. The 20 candidates show the same
pattern (no weights 0.7187, weight 0.25 0.7248, dropped 0.7277).

**Part C: the final list.**

| Feature set | 2016-17 | 2018-19 | 2020-21 | Mean |
|---|---:|---:|---:|---:|
| 20 candidates | 0.7338 | 0.7028 | 0.7377 | 0.7248 |
| Final list (12) | 0.7358 | 0.7045 | 0.7351 | 0.7251 |

The final list ties the 20 candidates with 8 fewer features, so the simpler
list wins. Group ablation on the final list:

| Dropped group | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Helps in |
|---|---:|---:|---:|---:|---:|
| grid + qualifying | +0.0950 | +0.0879 | +0.0856 | +0.0895 | 3 of 3 |
| driver_standings | −0.0169 | +0.0165 | +0.0556 | +0.0184 | 2 of 3 |
| grid | +0.0282 | +0.0135 | +0.0090 | +0.0169 | 3 of 3 |
| constructor_standings | −0.0084 | +0.0113 | +0.0140 | +0.0057 | 2 of 3 |
| qualifying | +0.0042 | −0.0049 | +0.0147 | +0.0047 | 2 of 3 |

All four groups contribute (each costs at least 0.0047 when removed, in at
least 2 periods).

**Single features in the final list.** Removing one feature at a time:

| Removed feature | Drop 2016-17 | Drop 2018-19 | Drop 2020-21 | Mean drop | Verdict |
|---|---:|---:|---:|---:|---|
| `grid_fixed` | +0.0282 | +0.0135 | +0.0090 | +0.0169 | clearly needed |
| `pts_before` | −0.0163 | +0.0099 | +0.0338 | +0.0091 | clearly needed |
| `cons_pts_before` | −0.0128 | +0.0114 | +0.0187 | +0.0058 | clearly needed |
| `pos_before` | +0.0085 | +0.0004 | +0.0017 | +0.0035 | clearly needed |
| `cons_standings_missing` | +0.0074 | +0.0011 | +0.0012 | +0.0033 | clearly needed |
| `cons_pos_before` | −0.0067 | +0.0106 | +0.0058 | +0.0032 | clearly needed |
| `qual_position` | −0.0039 | +0.0050 | +0.0072 | +0.0027 | clearly needed |
| `no_qual_time` | −0.0016 | +0.0009 | +0.0049 | +0.0014 | small or shared |
| `qg_q3` | +0.0047 | −0.0016 | +0.0004 | +0.0012 | small or shared |
| `qg_q3` + `qg_q2_only` (removed together) | +0.0033 | −0.0016 | +0.0016 | +0.0011 | small or shared |
| `gap_to_pole_pct` | +0.0029 | −0.0019 | +0.0019 | +0.0010 | small or shared |
| `qg_q2_only` | +0.0009 | +0.0009 | +0.0010 | +0.0009 | small or shared |
| `standings_missing` | +0.0008 | +0.0001 | −0.0000 | +0.0003 | small or shared |

Seven features are clearly needed on their own. The others are small on
their own because they share information (`gap_to_pole_pct` overlaps with
`qual_position` and `grid_fixed`; the missing flags affect few rows), but none
of them hurts, and the qualifying group as a whole costs 0.0047 when removed.

**`qg_q1_only` removed.** `qg_q3 + qg_q2_only + qg_q1_only + no_qual_time = 1`
on all 24,566 rows, so `qg_q1_only` can be derived from the other three.
With all four columns, removing any single one changed nothing (−0.0002);
without `qg_q1_only` the score is the same (0.7251 vs 0.7249 with 13
features) and every remaining qualifying column now shows a positive effect.

Notes on the final list:
- `qual_group` is kept: it clearly helped when training on the knockout seasons
  only (+0.0048 on 2006–2019), and it is a tie with weighted 1950 training
  because the "round reached" only exists in the knockout format.
- `season_progress` is not added: removing it from the 20 candidates costs
  0.0024, but adding it to the final list gains only 0.0005, below the rule.
- `pit_lane_flag` adds nothing (0.0000): `grid_fixed = PIT_GRID` already handles pit-lane starts.

**Part E: extra checks on the final list** (mean 0.7251).

- *E1, add one unused column* (25 columns): none passes the rule. The best
  are `pts_share` (+0.0012), `round` (+0.0009) and `circuit_podiums_before`
  (+0.0008). The IDs hurt (`number` −0.0160), as expected for labels.
- *E2, new feature ideas*: `qual_gap_to_teammate` +0.0000, `team_dnf_rate10`
  −0.0010, `driver_dnf_rate10` −0.0009, `pos_gained_avg10` +0.0002, all four
  +0.0001. None helps: the standings already reflect reliability, and the
  drivers who gain many places are mostly those who start at the back.
- *E3, every subset of the final list* (2,047 subsets, the two qg columns as
  one unit): the final list ranks 41st by mean. Three subsets pass the rule
  against it (+0.0021 to +0.0027 mean, better in 2 of 3 periods). All three
  drop the season points (`pts_before`, `cons_pts_before`) and keep the
  positions. They win on 2016–17 and 2018–19 but lose 0.015–0.017 on 2020–21,
  the official validation period and the closest to the test years. With
  2,047 subsets tested, a few passing the 0.002 rule by chance is expected,
  and `pts_before` has the largest single drop on 2020–21 (+0.0338). We keep
  the final list.
- *E4, grid fallback for the old seasons*: filling the missing
  `qual_position` (63% of training rows, almost all before 2006) with the
  starting grid slot lowers the mean to 0.7215 (−0.0036, better in 1 of 3
  periods). `grid_fixed` is already a feature, so the fallback only repeats
  it and hides that those rows have no real qualifying result. We keep the
  median plus the `no_qual_time` flag.

**Decision.** Final features = `FEATURE_GROUPS` (12): `grid_fixed`;
`qual_position`, `gap_to_pole_pct`, `no_qual_time`, `qg_q3`, `qg_q2_only`;
`pts_before`, `pos_before`, `standings_missing`;
`cons_pts_before`, `cons_pos_before`, `cons_standings_missing`. Train on
1950–2019 with `sample_weight` (pre-2006 rows 0.25).

---
# >>> Section from `notebooks/p4_models.ipynb`


## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on P3's modeling table, chooses the metrics, the hyperparameters and the decision threshold **without touching the test set**, evaluates the final choice once on the test set, explains the final model and lists its limitations.

**Inputs from Part 3:** the modeling table `model` (one row per driver per race, 1950-2024, with the `split` and `sample_weight` columns), the 12 final features `FEATURES` and their groups `FEATURE_GROUPS`.
**Outputs for P1's inference function:** `FINAL_MODEL` (preprocessing + model in one pipeline; it takes the columns in `FEATURES`), `FINAL_THRESHOLD` and `FINAL_NAME`.

Sections: 4.1) evaluation function, 4.2) data, weights, preprocessing and leakage checks, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models (plus two other model families), 4.6) shallow neural network (plus a wider and a deeper network), 4.7) hyperparameter tuning (plus the number of trees), 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI), 4.12) model limitations, 4.13) hand-off to the inference function.

Every attempt has its own cell and its own result, numbered **Attempt 1, 2, 3...** in the order they were run.

In [ ]:
clean = load_clean(exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
outcomes = clean["outcomes"]    # post-race columns (status, finishing position...). Never features: only used to tell the stories in 4.11-4.12
tables = clean["tables"]        # cleaned tables (driver and race names for the examples)

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (ours and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14-15% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a probability threshold. The threshold is 0.5 until it is tuned on validation in 4.8.

**Training weights (new):** P3 keeps every season from 1950 and gives the rows before 2006 a weight of 0.25 (`sample_weight`). `train_eval()` takes these weights as an optional argument: it passes them to the model's `fit` (for a pipeline, to its last step) and uses them for the training scores too, so the training score measures what the model was asked to learn. Validation and test rows all have weight 1, so their scores are plain. Without the argument the function works exactly as before, so P3's code that calls it does not change.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5, sample_weight=None):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`.
    `sample_weight` (optional) weights the rows, as in training."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    w = sample_weight
    return {
        "ROC-AUC": roc_auc_score(y, proba, sample_weight=w),
        "PR-AUC": average_precision_score(y, proba, sample_weight=w),
        "F1": f1_score(y, pred, sample_weight=w, zero_division=0),
        "precision": precision_score(y, pred, sample_weight=w, zero_division=0),
        "recall": recall_score(y, pred, sample_weight=w, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model", sample_weight=None):
    """Fit `model` on the training years only, then score it on train and validation.
    `sample_weight` (optional): one weight per training row (P3's `sample_weight` column). It is passed to fit
    (to the last step of a pipeline) and used for the training scores. Validation rows are not weighted.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    if sample_weight is None:
        model.fit(X_train, y_train)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        key = f"{model.steps[-1][0]}__sample_weight" if hasattr(model, "steps") else "sample_weight"
        model.fit(X_train, y_train, **{key: sample_weight})
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold, sample_weight)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Data, weights, preprocessing and leakage checks

**Data:** P3's modeling table and P3's time split: train 1950-2019, validation 2020-2021, test 2022-2024. The test rows are separated only in 4.10.

**Features:** P3's final `FEATURES`, 12 numeric columns in 4 groups (grid, qualifying, driver standings, constructor standings), chosen by P3's ablation (3.11). Everything P3 tested and dropped (circuit history, home race, era, recent form, raw lap time) is not used. In the previous version we had to remove `era_ordinal` ourselves; P3's final list no longer contains it.

**Weights:** every training row from 2006 on has weight 1, every row from 1950-2005 has weight 0.25 (P3, 3.3-3.4 and 3.11 Part F). Old seasons have little or no qualifying data and a different sport (more cars, more retirements, other points systems), so they count a quarter. Every model below gets the weights, except k-NN, which cannot use them (4.5).

**Preprocessing (one pipeline, the same for every model):**
* **Column selection:** the first step keeps the columns in `FEATURES` and turns them into plain numbers (P3 stores some as nullable integers). P1's inference function passes exactly these columns.
* **Missing values are structural, not errors:** the championship position is empty in round 1 of every season, and qualifying data does not exist before 1994 and is patchy until 2006. We fill gaps with the **training median** and add a **missing-indicator column** for every feature that has gaps in training, so the model still knows that the value was missing.
* **Scaling:** every feature is standardized (mean 0, standard deviation 1). Logistic regression, k-NN and the network need it; for the tree models it changes nothing.
* **Fitted on the training rows only:** the imputer and the scaler are steps of a `Pipeline`, so `fit()` learns the medians, means and standard deviations from the training years only; validation and test rows are only transformed. (The weights go to the model step; the imputer and scaler use every training row equally.)
* **Difference from P3's experiments:** P3's ablation uses plain median imputation without missing-value flags. Our models add the flags (`add_indicator=True`), so P3's logistic-regression numbers and ours differ slightly. P3 also scores on three validation periods; the team rule for P4 is the single validation period 2020-2021, so our numbers are not directly comparable with her averages.

**Leakage checks (code below):** no post-race column is a feature, the weight is not a feature, each split holds only its own years, and only training rows before 2006 have a weight below 1.

In [ ]:
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

table = model                     # P3's modeling table (read only; the name `model` is kept for Part 3)

# leakage checks: no post-race column and no weight among the features, and the time split is respected
assert not set(FEATURES) & set(outcomes.columns), "a post-race column is used as a feature"
assert "sample_weight" not in FEATURES, "the training weight must not be a feature"
assert table.loc[table["split"] == "train", "year"].max() <= 2019
assert table.loc[table["split"] == "validation", "year"].between(2020, 2021).all()
assert table.loc[table["split"] == "test", "year"].min() >= 2022
assert (table.loc[table["sample_weight"] < 1, "year"] < 2006).all(), "only pre-2006 rows may be down-weighted"

train = table[table["split"] == "train"]
val = table[table["split"] == "validation"]
X_train, y_train, w_train = train[FEATURES].astype(float), train["podium"], train["sample_weight"].to_numpy()
X_val, y_val = val[FEATURES].astype(float), val["podium"]


def keep_features(X):
    """First step of every model: keep the FEATURES columns as plain float numbers."""
    return X[FEATURES].astype(float)

select = FunctionTransformer(keep_features, feature_names_out=lambda transformer, names: FEATURES)
preprocess = make_pipeline(select, SimpleImputer(strategy="median", add_indicator=True), StandardScaler())


def with_preprocessing(estimator):
    """The shared preprocessing followed by `estimator`, as one pipeline (fitted on the training rows only)."""
    return Pipeline([("prep", clone(preprocess)), ("model", estimator)])


print(f"{len(FEATURES)} features in {len(FEATURE_GROUPS)} groups: {FEATURE_GROUPS}")
summary = table[table["split"] != "test"].groupby(["split", table["year"] >= 2006]).agg(
    rows=("podium", "size"), years=("year", lambda s: f"{s.min()}-{s.max()}"),
    podium_rate=("podium", "mean"), weight=("sample_weight", "mean"))
summary.index = summary.index.set_levels(["1950-2005", "2006 and later"], level=1)
display(summary.round(3))
missing = pd.DataFrame({"1950-2005": X_train[train["year"] < 2006].isna().mean(),
                        "2006-2019": X_train[train["year"] >= 2006].isna().mean()})
display(missing.loc[missing.max(axis=1) > 0].round(3).rename_axis("share missing (train)"))

**What we see.**
* **Rows and weights:** 22,441 training rows: 16,699 from 1950-2005 (weight 0.25, podium rate 13.4%) and 5,742 from 2006-2019 (weight 1, 14.0%). Validation: 774 rows from 2020-2021 (15.1%). In weighted terms the old seasons count as about 4,175 rows, a bit less than the modern ones.
* **Why the old seasons get a lower weight:** about 85% of the 1950-2005 rows have no qualifying position or gap (qualifying data starts in 1994 and is patchy until 2006), against less than 2% from 2006 on. The championship position is empty more often too (14% vs 7%), because the early seasons were short.
* **The missing-value flags matter now:** for the old rows, "qualifying missing" is the normal case, not a rare signal. The model sees it through the flags of 4.2.

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation), **PR-AUC = the podium rate** (no better than random ranking) and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)",
                           sample_weight=w_train)
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate (0.151 on validation) and F1 = 0, exactly as expected, so the function computes the metrics correctly (with and without weights). Its accuracy is still 84.9%, although it never finds a single podium. That is why accuracy is not one of our metrics, and why **PR-AUC is the natural primary metric** for this rare positive class. These numbers are the floor every real model must beat.

### 4.4 Baselines

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned.
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it encodes the domain fact that the starting position is the strongest single predictor.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more for a top team").

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a single operating point. It uses P3's `grid_fixed`, where a pit-lane start is slot 35, so a pit-lane start is never a front-row start.

In [ ]:
class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y, sample_weight=None):
        return self

    def predict_proba(self, X):
        p = X["grid_fixed"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3", sample_weight=w_train)
display(rule_results)

#### Attempt 2: logistic regression (12 features, with the weights)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression", sample_weight=w_train)
display(lr_results)

#### Attempt 3: the same logistic regression without the weights
Every row from 1950 counts fully. Does P3's down-weighting of the old seasons also help our pipeline?

In [ ]:
lr_noweight = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_noweight_results = train_eval(lr_noweight, X_train, y_train, X_val, y_val,
                                 name="logistic regression (no weights)")
display(lr_noweight_results)

#### Attempt 4: the same logistic regression on 2006-2019 only
The opposite extreme: old seasons get weight 0 (the setup of our previous version).

In [ ]:
modern = (train["year"] >= 2006).to_numpy()
lr_modern = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_modern_results = train_eval(lr_modern, X_train[modern], y_train[modern], X_val, y_val,
                               name="logistic regression (2006-2019 only)")
display(lr_modern_results)

**What we see (the weights, Attempts 2-4).**
* **Weighted (P3's setup): 0.736.** All rows at full weight: **0.728**. Only 2006-2019: **0.744**.
* **The weights help** compared with treating 1950s races like modern ones (+0.008).
* **Dropping the old seasons scores a little higher still** (+0.008 over the weighted version). P3 found the same on her three validation periods and documented it: the team keeps the old seasons with a low weight because the course asks us not to simply drop them.
* **All three numbers are within 0.016 of each other,** far less than the noise of the validation PR-AUC (standard error about 0.037, 4.9). We follow P3's setup (weighted) for every model below.

#### Attempt 5: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training (it multiplies with P3's season weights). We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = with_preprocessing(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)",
                            sample_weight=w_train)
display(lr_bal_results)

#### Attempt 6: logistic regression on the 4 draft features only
Before P3's table was ready, the models were built on four raw pre-race columns (`grid`, `pitlane_start`, `has_qualifying`, `qual_position`). Same model, same rows, same weights: does P3's feature engineering add anything?

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]
lr_draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))
lr_draft_results = train_eval(lr_draft, train[DRAFT_FEATURES].astype(float), y_train,
                              val[DRAFT_FEATURES].astype(float), y_val,
                              name="logistic regression (4 draft features)", sample_weight=w_train)
display(lr_draft_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_noweight_results, lr_modern_results,
                       lr_bal_results, lr_draft_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4.5))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned: weights of the standardized inputs, largest first
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
display(weights.sort_values(key=abs, ascending=False).head(10).round(3).to_frame("weight (standardized input)"))

**What we see (validation 2020-2021).**
* **Logistic regression is far above both references:** PR-AUC 0.736 vs 0.530 for the rule and 0.151 for the dummy; ROC-AUC 0.929 vs 0.820.
* **P3's features add real information:** the same weighted logistic regression reaches PR-AUC 0.667 with the 4 draft features and **0.736 with P3's 12 features (+0.069)**.
* **At threshold 0.5 the rule still has the best F1** (0.695 vs 0.648): plain logistic regression is cautious at 0.5 (precision 0.73, recall 0.58). The threshold is tuned in 4.8.
* **`class_weight="balanced"` behaves as expected:** PR-AUC barely changes (0.730 vs 0.736), while at 0.5 recall rises to 0.91 and precision falls to 0.46.
* **Weights:** the largest is `grid_fixed` (-1.60): starting nearer the front means a higher podium probability. Next come the missing-value flags of the qualifying columns (-1.06 and +0.69) and `no_qual_time` (+0.69). They mostly describe the old seasons, where qualifying is missing for everyone, and partly cancel each other, so they should not be read one by one. Then `gap_to_pole_pct` (-0.57), the team's (-0.44) and the driver's (-0.38) championship positions. These are associations the model uses, not causes.
* **Train PR-AUC is much lower than validation** (0.626 vs 0.736): the training score includes the 1950-2005 seasons, which are harder to predict (more cars, more retirements, little qualifying data). It is not a sign of a problem with the model.

**Baseline to beat:** logistic regression, validation PR-AUC **0.736**.

### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features (for example "a good grid slot matters more for a team that leads the championship"). Both are in scikit-learn.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* its probabilities are averages of votes (not calibrated), and it cannot extrapolate beyond the values seen in training.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. *Limitation:* more hyperparameters, and it overfits if the trees are too big or too many. scikit-learn's automatic early stopping only starts above 10,000 training rows; we now have 22,441, so it **is** on by default (it holds out 10% of the training rows, never our validation years). To keep the number of trees under our control, we switch it off and set the number of trees ourselves (100 here; Attempt 19 studies it).

**Class imbalance:** both use `class_weight="balanced"` (podiums are about 14% of rows), on top of P3's season weights. Hyperparameters are tuned in 4.7.

#### Attempt 7: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 driver-races, which keeps the probabilities smooth instead of memorizing single rows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = with_preprocessing(RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                               n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest", sample_weight=w_train)
display(rf_results)

#### Attempt 8: gradient boosting
Default tree settings, 100 trees, no automatic early stopping.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = with_preprocessing(HistGradientBoostingClassifier(early_stopping=False, class_weight="balanced",
                                                       random_state=SEED))
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting", sample_weight=w_train)
display(gb_results)

#### Two other model families (exploration)
The brief asks for two statistical models, and we have them. To check that we did not miss an easy win, we try two families that work in a completely different way, each with one sensible setting and no tuning. They are not candidates for the final model unless they clearly beat the others.
* **k-nearest neighbours (k-NN):** no training at all. For a new driver-race it finds the 50 most similar driver-races in the training years, and the probability is the share of them that ended on the podium. *Limitation:* every feature counts equally in "similar", and it cannot use training weights. We therefore give it only the full-weight seasons (2006-2019), which is the same as giving the old seasons weight 0.
* **Gaussian Naive Bayes:** for each class it learns a bell curve per feature and multiplies them, assuming the features are **independent** of each other. *Limitation:* our features are strongly linked (grid and qualifying position, points and championship position), so the same evidence is counted several times.

#### Attempt 9: k-nearest neighbours (k = 50, 2006-2019 rows)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = with_preprocessing(KNeighborsClassifier(n_neighbors=50))
knn_results = train_eval(knn, X_train[modern], y_train[modern], X_val, y_val,
                         name="k-nearest neighbours (k=50)")
display(knn_results)

#### Attempt 10: Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = with_preprocessing(GaussianNB())
nb_results = train_eval(nb, X_train, y_train, X_val, y_val, name="naive Bayes", sample_weight=w_train)
display(nb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap means overfitting).

In [ ]:
def compare(results):
    """Validation scores of every model, plus the train - validation PR-AUC gap."""
    allres = pd.concat(results, ignore_index=True)
    pr = allres.pivot(index="model", columns="split", values="PR-AUC")
    summary = allres[allres["split"] == "validation"].set_index("model").drop(columns="split")
    summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
    return summary.sort_values("PR-AUC", ascending=False)

summary = compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results,
                   rf_results, gb_results, knn_results, nb_results])
display(summary)

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4.5), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models tie with logistic regression:** PR-AUC random forest 0.739, logistic regression 0.736, gradient boosting 0.727; ROC-AUC 0.927-0.933.
* **Gradient boosting no longer overfits badly:** train 0.750 vs validation 0.727. With 22,441 training rows (instead of 5,742) the default trees have much more data to learn from.
* **k-NN improved a lot (0.736, now a tie):** in the previous version (19 features) it got 0.677. With P3's shorter list of strong features, "similar driver-race" means similar grid, qualifying and standings, without weak features blurring the distance. A nice side effect of P3's feature selection.
* **Naive Bayes is still the weakest real model (0.570):** better than with 19 features (0.292), because there are fewer duplicate features, but it still counts the same evidence several times (grid, qualifying position, gap and the Q3 flag all say "fast car").
* **Why no gain from trees:** the strongest signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures.

**Decision:** random forest and gradient boosting stay as our two statistical models and are tuned in 4.7. k-NN ties with the baseline but adds nothing over it and cannot use P3's weights; Naive Bayes is far behind. Neither is taken further.

### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Training weights:** each row's loss is multiplied by its `sample_weight`, so a 1950s row counts a quarter, exactly as in the other models.
* **Class imbalance:** `pos_weight` = (weighted non-podium rows / weighted podium rows) in the loss, the same idea as `class_weight="balanced"`.
* **Preprocessing:** the same pipeline as the other models (4.2), fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread, so Run All gives the same numbers on a laptop and on Kaggle.

*Limitations:* many settings, needs scaling and imputation, a black box without XAI, and on small tables it rarely beats simpler models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the test set gives the fair comparison.

In [ ]:
import torch
from torch import nn


class ShallowNN:
    """Feed-forward network wrapped with fit / predict_proba so train_eval() can score it.
    - hidden: the units of each hidden layer, e.g. (32,) = one layer, (64, 32) = two layers
    - preprocessing: the shared pipeline of 4.2, fitted on the training rows only, inside fit()
    - sample_weight: each training row's loss is multiplied by its weight
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y, sample_weight=None):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine
        self.prep = clone(preprocess).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        wt = torch.ones(len(yt)) if sample_weight is None else torch.tensor(np.asarray(sample_weight), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        self.n_weights = sum(p.numel() for p in self.net.parameters())
        pos_weight = wt[yt == 0].sum() / wt[yt == 1].sum()      # class imbalance, like class_weight="balanced"
        row_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")   # one loss per row
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                losses = row_loss(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss = (losses * wt[idx]).sum() / wt[idx].sum()  # weighted mean: old seasons count less
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = row_loss(self.net(Xv).squeeze(1), yv).mean().item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])


def run_network(hidden, name):
    """Train one network on the training years (with the weights), score it, print its size and stopping epoch."""
    net = ShallowNN(X_val, y_val, hidden=hidden)
    res = train_eval(net, X_train, y_train, X_val, y_val, name=name, sample_weight=w_train)
    display(res)
    print(f"{net.n_weights} weights; stopped after {len(net.history['train'])} epochs, best epoch: {net.best_epoch}")
    return net, res

#### Attempt 11: one hidden layer (32 units)

In [ ]:
nn1, nn1_results = run_network((32,), "neural network (32)")

#### Attempt 12: two hidden layers (64 and 32 units)

In [ ]:
nn2, nn2_results = run_network((64, 32), "neural network (64-32)")

#### Attempt 13: a wider layer (one hidden layer, 128 units)
Same depth as Attempt 11 but four times the units. If the 32-unit layer were too small to hold the patterns, more units would help.

In [ ]:
nn128, nn128_results = run_network((128,), "neural network (128)")

#### Attempt 14: a deeper network (three hidden layers, 128-64-32)
One step beyond "shallow": if 1 and 2 layers are close, does a third layer find anything more? (Kept out of the tuning grid in 4.7, because the brief asks for a shallow network; this attempt only shows what depth does.)

In [ ]:
nn3, nn3_results = run_network((128, 64, 32), "neural network (128-64-32)")

#### Loss curves
Training loss (weighted average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, net, title in [(axes[0, 0], nn1, "1 hidden layer (32)"), (axes[0, 1], nn2, "2 hidden layers (64-32)"),
                       (axes[1, 0], nn128, "1 wide hidden layer (128)"), (axes[1, 1], nn3, "3 hidden layers (128-64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
for ax in axes[:, 0]:
    ax.set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of the networks (validation)

In [ ]:
nets = compare([nn1_results, nn2_results, nn128_results, nn3_results])
sizes = {"neural network (32)": nn1, "neural network (64-32)": nn2, "neural network (128)": nn128,
         "neural network (128-64-32)": nn3}
nets.insert(0, "weights", [sizes[m].n_weights for m in nets.index])
display(nets)

**What we see (validation 2020-2021).**
* **All four networks are a little above logistic regression (0.736):** 0.759 with one layer of 32, 0.759 with two layers (64-32), **0.763 with one wide layer of 128**, 0.755 with three layers.
* **Wider (Attempt 13):** this time the 128-unit layer is the best network, but only by 0.005, with four times the weights of the 32-unit layer (2,305 vs 577). That is far below the noise of the validation PR-AUC (about 0.037), so it is a tie, not a win.
* **Deeper (Attempt 14):** the third layer gives 0.755, no better, with the most weights (12,545). It reaches its best validation loss already at **epoch 8**, then the validation loss creeps up while the training loss keeps falling: the start of overfitting.
* **All four are within 0.008 of each other.** So the honest answer to "why not a bigger or deeper network?" is: we tried; with this table, size does not matter, and a shallow network is enough.
* **Loss curves:** both losses fall quickly in the first 5-10 epochs and then flatten. This time the **training** loss is above the validation loss, because the training rows include the old seasons, which are harder to predict (the loss is weighted, but those rows are still there).
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.90-0.91), precision about 0.50.

### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years (with the weights) and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years, with the weights) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        candidate = make_model(**params)
        res = train_eval(candidate, X_train, y_train, X_val, y_val, name=name,
                         sample_weight=w_train).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = candidate, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 15: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: with_preprocessing(LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 16: tuning the random forest
`min_samples_leaf`: the minimum number of driver-races behind each leaf (larger = smoother, less memorizing). `max_features`: how many inputs each split may look at (`"sqrt"` = a random subset, `1.0` = all of them).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: with_preprocessing(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 17: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees). 100 trees each.

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: with_preprocessing(HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, early_stopping=False,
        class_weight="balanced", random_state=SEED)),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 18: tuning the neural network
`hidden`: the size of the hidden layer(s), including the 128-unit layer of Attempt 13. `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (128,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** `C` hardly matters (0.729-0.737); best `C=10` (0.737). With 12 features and 22,441 rows the model is not short of data.
* **Random forest:** best with `max_features="sqrt"` and `min_samples_leaf=5` (0.740), just above `min_samples_leaf=20` (0.739). But small leaves memorize: train 0.842 vs validation 0.740. Letting every split see all features (`1.0`) is worse (0.711-0.731).
* **Gradient boosting:** best with small trees and the faster rate (`max_leaf_nodes=7`, `learning_rate=0.1`): 0.733. The biggest trees are the worst on validation.
* **Neural network:** best with one hidden layer of 128 and dropout 0.2: 0.763 (Attempt 13). The 128-unit layer is also second with more dropout (0.760). Two layers 0.759, 32 units 0.759.
* **Tuning gains are small** (at most +0.005, for gradient boosting), and the four tuned models end within 0.031 of each other (0.733-0.763), less than one standard error.

#### Attempt 19: how many trees? (random forest and gradient boosting)
Until now the forest always had 300 trees and boosting 100. Here we keep each model's tuned settings and change only the number of trees, scoring train and validation every time. The two families should react differently: a forest **averages** independent trees, so more trees can only make the average steadier; boosting **adds** trees that keep correcting the training errors, so too many trees start to learn the noise of the training years.

In [ ]:
best_rf = tuned["random forest"][1].iloc[0]          # tuned settings from Attempts 16 and 17
best_gb = tuned["gradient boosting"][1].iloc[0]
tree_counts = [10, 25, 50, 100, 200, 400, 800]

rows = []
for n in tree_counts:
    forest = with_preprocessing(RandomForestClassifier(
        n_estimators=n, min_samples_leaf=int(best_rf["min_samples_leaf"]), max_features=best_rf["max_features"],
        class_weight="balanced", n_jobs=-1, random_state=SEED))
    boosting = with_preprocessing(HistGradientBoostingClassifier(
        max_iter=n, early_stopping=False, learning_rate=best_gb["learning_rate"],
        max_leaf_nodes=int(best_gb["max_leaf_nodes"]), class_weight="balanced", random_state=SEED))
    for family, clf in [("random forest", forest), ("gradient boosting", boosting)]:
        res = train_eval(clf, X_train, y_train, X_val, y_val, name=family, sample_weight=w_train).set_index("split")
        rows.append({"model": family, "trees": n, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"]})
trees = pd.DataFrame(rows)
display(trees.pivot(index="trees", columns="model", values=["train PR-AUC", "val PR-AUC"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, family, used in [(axes[0], "random forest", 300), (axes[1], "gradient boosting", 100)]:
    part = trees[trees["model"] == family]
    ax.plot(part["trees"], part["train PR-AUC"], marker="o", label="train")
    ax.plot(part["trees"], part["val PR-AUC"], marker="o", label="validation")
    ax.axvline(used, color="black", linestyle="--", label=f"number used in the models ({used})")
    ax.set_xscale("log")
    ax.set_xticks(tree_counts, [str(n) for n in tree_counts])
    ax.set_title(f"{family}: PR-AUC vs number of trees")
    ax.set_xlabel("number of trees (log scale)")
    ax.legend()
axes[0].set_ylabel("PR-AUC")
plt.tight_layout()
plt.show()

**What we see (number of trees).**
* **Random forest: more trees never hurt, they only steady the average.** Validation PR-AUC rises from 0.707 with 10 trees to 0.739 with 200 and stays there (0.740 with 800); the training score is flat (about 0.84). The 300 trees we use are on the plateau.
* **Gradient boosting: the number of trees is a complexity knob.** The training score keeps rising (0.611 with 10 trees to 0.783 with 800) while validation peaks at 200 trees (0.736) and then falls clearly (0.731 at 400, **0.683 at 800**): every extra tree corrects the remaining training errors, and after a point those errors are noise.
* **We keep the tuned models unchanged.** 200 boosting trees would add +0.004 on validation, far below the noise, and it would not change the final model (4.9). The lesson is the shape: for boosting the number of trees must be limited (or stopped early), for a forest it does not matter.

### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (about 0.8 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (4.4).

In [ ]:
def best_threshold(clf, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = clf.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (clf, _) in tuned.items():
    t = best_threshold(clf, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(clf, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (4.10) and by P1's inference function (4.13).

In [ ]:
COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

within_one_se = [name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se]
FINAL_NAME = within_one_se[0]
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {within_one_se}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from 4.4
rule_pred = X_val["grid_fixed"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")
display(curve.loc[sorted({0.2, 0.3, 0.4, FINAL_THRESHOLD, 0.5, 0.6})].round(3))     # how flat F1 is around the choice

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.29 for logistic regression, 0.69 for the random forest, 0.71 for gradient boosting, 0.72 for the network. The balanced models have their probabilities pushed up by the class weights, so their best threshold is higher.
* **With its own threshold every model reaches a validation F1 of about 0.71-0.74** (network 0.736, gradient boosting 0.722, random forest 0.718, logistic regression 0.710).
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.037 (95% range 0.69-0.83). All four models are within one standard error of the best (the network, 0.763), so the one-standard-error rule picks the simplest: **logistic regression (`C=10`) with threshold 0.29**.
* **The threshold sits on a plateau:** validation F1 is 0.660 at 0.20, **0.710 at 0.29**, 0.708 at 0.30, 0.687 at 0.40 and 0.641 at 0.50. Anything between about 0.25 and 0.35 is almost equally good, which is why the exact value moves between table versions (0.23, then 0.42, now 0.29) while F1 barely changes.
* **More than the rule:** the final model disagrees with the rule `grid <= 3` on 43 of 774 validation rows.

### 4.10 Final evaluation on the test set (used once)

Every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set (2022-2024) with the threshold it got on validation, next to its train and validation scores. Nothing is changed after this point. The rule `grid <= 3` is the reference to beat.

**Honest note on the test set.** This part was run on four versions of P3's table: (1) 18 features with a one-hot `era`, where the test probabilities were pushed up because the test era never occurs in training; (2) 17 features from 2003; (3) 20 features from 2006, of which we used 19; (4) this run: P3's final 12 features, training 1950-2019 with weights. Every change of the table was made by P3 with reasons documented on training and validation years (her ablation never uses the test years). **Nothing in this part was chosen on the test set:** the models, the hyperparameter grids, the threshold rule and the final-model rule are the same in every run, and every choice is made on validation.

In [ ]:
test = table[table["split"] == "test"]
X_test, y_test = test[FEATURES].astype(float), test["podium"]
assert (test["sample_weight"] == 1).all()
print(f"test: {len(X_test)} rows ({test['year'].min()}-{test['year'].max()}), {y_test.sum()} podiums, "
      f"podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (clf, thresholds[name]) for name, (clf, _) in tuned.items()}}
rows = []
for name, (clf, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y, w in [("train", X_train, y_train, w_train), ("validation", X_val, y_val, None),
                           ("test", X_test, y_test, None)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(clf, X, y, t, w)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (clf, _) in tuned.items():
    proba = clf.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (clf, _)) in zip(axes, tuned.items()):
    pred = (clf.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **Ranking on unseen seasons:** PR-AUC network 0.693, logistic regression 0.692 (a tie), gradient boosting 0.682, random forest 0.670; rule 0.474; no-skill floor 0.151. ROC-AUC is 0.93 for all four.
* **Yes/no decisions:** F1 network 0.685, logistic regression 0.676, gradient boosting 0.667, random forest 0.656, rule 0.649. The network's lead (+0.009) is well inside the noise, and switching to it now would be a choice made on the test set, so we do not.
* **The one-standard-error rule held again:** from validation to test the network drops most (0.763 to 0.693, its stopping epoch was chosen on validation) and the forest drops a lot (0.740 to 0.670, its small leaves memorized the training seasons: train 0.842), while the simplest model drops least (0.737 to 0.692).
* **Final model in numbers (threshold 0.29):** it finds 169 of the 204 podiums (recall 0.83) with 127 false alarms (precision 0.57) and 35 missed podiums. The lower threshold favours finding podiums.
* **Compared with the previous run (19 features, 2006-2019, threshold 0.42):** test PR-AUC 0.702 then, 0.692 now; F1 0.673 then, 0.676 now. Both differences are well inside the noise: P3's shorter feature list and the weighted old seasons give practically the same model quality, with 12 features instead of 19.
* **No choice was made on the test set:** the final model and threshold are the ones chosen on validation in 4.9.

### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature. Because several features carry the same information (for example `grid_fixed` and `qual_position`), we also shuffle each of P3's **feature groups** together: shuffling only one of two twin features underestimates both, since the model can lean on the twin.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles)

In [ ]:
from sklearn.inspection import permutation_importance

pr_auc_scorer = lambda clf, X, y: average_precision_score(y, clf.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES)

# the same idea per feature group: shuffle all columns of a group together
rng = np.random.default_rng(SEED)
full_score = pr_auc_scorer(FINAL_MODEL, X_val, y_val)
group_rows = []
for group, cols in FEATURE_GROUPS.items():
    drops = []
    for _ in range(30):
        shuffled = X_val.copy()
        shuffled[cols] = X_val[cols].iloc[rng.permutation(len(X_val))].set_axis(X_val.index)
        drops.append(full_score - pr_auc_scorer(FINAL_MODEL, shuffled, y_val))
    group_rows.append({"group": group, "PR-AUC drop": np.mean(drops), "std": np.std(drops)})
group_perm = pd.DataFrame(group_rows).set_index("group")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 2]})
for ax, imp, title in [(ax1, perm, "per feature"), (ax2, group_perm, "per feature group (shuffled together)")]:
    imp = imp.sort_values("PR-AUC drop")
    ax.barh(imp.index, imp["PR-AUC drop"], xerr=imp["std"], color="tab:blue")
    ax.set_title(f"Permutation importance {title}")
    ax.set_xlabel("drop in validation PR-AUC (bar = mean, line = std)")
fig.suptitle(f"Final model: {FINAL_NAME}")
plt.tight_layout()
plt.show()
display(perm.sort_values("PR-AUC drop", ascending=False).round(4).head(8))
display(group_perm.sort_values("PR-AUC drop", ascending=False).round(4))

#### Global: SHAP summary (validation)
SHAP and LIME call the model with plain number arrays, so a small helper `predict_podium_numeric` first turns them back into a table with the feature names. (Its name differs from P1's inference function `predict_podium` on purpose, so the two do not overwrite each other in the merged notebook.) Rows with a missing value (mostly round 1, without a championship position) are left out of the explanations, so every explained value is real. Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low).

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

def predict_podium_numeric(A):
    """Podium probability for a plain number array in FEATURES order (how SHAP and LIME call the model)."""
    return FINAL_MODEL.predict_proba(pd.DataFrame(np.asarray(A, dtype=float), columns=FEATURES))[:, 1]

complete = val[FEATURES].notna().all(axis=1)
xai_rows = val[complete]                                   # validation rows without missing values
X_xai, y_xai = xai_rows[FEATURES].astype(float), xai_rows["podium"].to_numpy()
print(f"explained rows: {len(X_xai)} of {len(val)} validation rows")

background = X_xai.sample(100, random_state=SEED)          # SHAP's baseline = average prediction over these rows
explainer = shap.Explainer(predict_podium_numeric, shap.maskers.Independent(background), algorithm="permutation",
                           feature_names=FEATURES, seed=SEED)
shap_values = explainer(X_xai, max_evals=2 * len(FEATURES) + 1, silent=True)
print(f"SHAP baseline (average predicted podium probability of the background rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position and status are shown only to tell the story; they are post-race information and never features.

In [ ]:
race_names = tables["races"].set_index("raceId")["name"]
driver_names = tables["drivers"].set_index("driverId")["surname"]
finish = outcomes.set_index(["raceId", "driverId"])[["positionOrder", "status"]]

proba_xai = predict_podium_numeric(X_xai)
non_podium_rows = np.where(y_xai == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y_xai == 1, proba_xai, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba_xai[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for explained row i (finishing position and status from `outcomes`, for the story only)."""
    r = xai_rows.iloc[i]
    pos, status = finish.loc[(r["raceId"], r["driverId"])]
    return (f"{driver_names[r['driverId']]}, {race_names[r['raceId']]} {r['year']}: grid {r['grid']}, "
            f"predicted {proba_xai[i]:.1%}, finished P{pos} ({status})")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

yes_no_columns = [j for j, c in enumerate(FEATURES) if X_xai[c].nunique() <= 2]   # 0/1 features, treated as categories
lime_explainer = LimeTabularExplainer(X_xai.to_numpy(), feature_names=FEATURES,
                                      class_names=["no podium", "podium"], categorical_features=yes_no_columns,
                                      mode="classification", random_state=SEED)
predict_both = lambda A: np.column_stack([1 - predict_podium_numeric(A), predict_podium_numeric(A)])

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=10)
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba_xai[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### The most confident miss
The non-podium row the model was most sure about: what happened in that race?

In [ ]:
most_confident_miss = int(np.argmax(np.where(y_xai == 0, proba_xai, -1)))
print(f"most confident miss: {describe(most_confident_miss)}")

**Interpretation.**
* **The starting slot carries the model.** Per feature, shuffling `grid_fixed` drops the validation PR-AUC by 0.262, `pts_before` by 0.101, `gap_to_pole_pct` by 0.044 and `qual_position` by 0.042; the rest are below 0.03. Per group: **grid (0.261), driver standings (0.165), qualifying (0.122)**, then constructor standings (0.038). P3's ablation (retraining without each group) also puts grid, qualifying and driver standings first.
* **Per feature vs per group:** the qualifying features are twins of each other (position, gap, Q3 flag), so each one alone looks small (0.04) while the group together is 0.12: shuffling one lets the model lean on the others. **Permutation vs ablation:** permutation keeps the trained model and breaks a group; ablation retrains without it, so the retrained model leans on the remaining twins and the drops are smaller.
* **Direction (SHAP summary):** a low (blue) `grid_fixed`, `qual_position`, `gap_to_pole_pct` and championship positions push the probability up, by up to +0.3 for the grid; many points before the race (`pts_before`, red) push it up by up to +0.4. Two signs look odd and come from twin features: many *team* points (`cons_pts_before`, red) push *down*, and reaching Q3 (`qg_q3`) pushes slightly down. Given the driver's own points and qualifying position, the model uses these to correct the twins (for example, a driver with few points in a team with many points is the slower team-mate). The flags (`no_qual_time`, `standings_missing`) show no effect here because every explained row is complete.
* **Podium example (Verstappen, Abu Dhabi 2021, from pole, won):** SHAP starts at the baseline 0.171 and adds 369.5 points (+0.37), grid 1 (+0.31), qualifying P1 (+0.11) and gap to pole 0 (+0.08); his team's high points pull back (-0.20). Result: 0.910. **Non-podium example (Vettel, French GP 2021, grid 12, finished 9th):** grid 12 (-0.08), qualifying 12th (-0.03), only 28 points (-0.03), 2% off pole (-0.02) bring him down to 0.016. Both stories are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** its small linear model estimates 0.33 where the real model says 0.91 (local fit R² 0.50), and it fits the non-podium row poorly (R² 0.10). SHAP values add up exactly to the model's prediction, so we trust SHAP more for the size of each effect.
* **The most confident miss is again Hamilton at the 2020 Italian Grand Prix:** pole position, predicted 81.3%, finished 7th after a stop-and-go penalty during the race. A pre-race model cannot see events inside the race.

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the grid, the championship and qualifying because they are the best pre-race summary of how fast the car and driver are; it does not show that the starting position *causes* podiums.

### 4.12 Model limitations

Four checks give numbers to the main limitations.
1. **Are the predicted probabilities realistic in every split?** Every race has exactly 3 podiums, so the predicted probabilities of one race should add up to about 3. This check needs **no labels**, so it can be run on any future season before its races; we also compare the average predicted probability with the real podium rate.
2. **Era sensitivity:** the test years are a new technical era. Is the final model equally good in each test season?
3. **The false alarms on the test set:** how many of those cars did not finish the race (crash, breakdown), something no pre-race model can see?
4. **Feature drift:** do the feature values of the test years look like those of the full-weight training years (2006-2019)? This also needs no labels.

These checks only describe the final model; nothing is changed because of them.

In [ ]:
# 1) are the probabilities realistic? (podiums per race should be about 3; this part needs no labels)
parts = [("train 1950-2005", train[train["year"] < 2006]), ("train 2006-2019", train[train["year"] >= 2006]),
         ("validation", val), ("test", test)]
calib_rows = []
for split, part in parts:
    p = pd.Series(FINAL_MODEL.predict_proba(part[FEATURES].astype(float))[:, 1], index=part.index)
    calib_rows.append({"split": split, "predicted podiums per race (median)": p.groupby(part["raceId"]).sum().median(),
                       "average predicted probability": p.mean(), "real podium rate": part["podium"].mean()})
display(pd.DataFrame(calib_rows).set_index("split").round(3))

# 2) era sensitivity: the final model on each test season separately
season_rows = []
for year, part in test.groupby("year"):
    season_rows.append({"season": year, "races": part["raceId"].nunique(),
                        **evaluate(FINAL_MODEL, part[FEATURES].astype(float), part["podium"], threshold=FINAL_THRESHOLD)})
display(pd.DataFrame(season_rows).set_index("season").round(3))

# 3) false alarms on the test set: how many of those cars did not finish the race?
status = test[["raceId", "driverId"]].merge(outcomes[["raceId", "driverId", "status"]],
                                            on=["raceId", "driverId"], how="left")["status"]
finished = (status.eq("Finished") | status.str.match(r"^\+\d+ Laps?$")).to_numpy()
said_podium = FINAL_MODEL.predict_proba(X_test)[:, 1] >= FINAL_THRESHOLD
false_alarm = said_podium & (y_test.to_numpy() == 0)
print(f"test false alarms: {false_alarm.sum()}; did not finish the race: {(~finished[false_alarm]).sum()} "
      f"({(~finished[false_alarm]).mean():.0%}), compared with {(~finished).mean():.0%} of all test starters")

# 4) feature drift (no labels): how far does each feature's test mean move, in training standard deviations?
X_modern = X_train[modern]
drift = pd.DataFrame({"train 2006-2019 mean": X_modern.mean(), "test mean": X_test.mean(),
                      "test rows above the training maximum": (X_test > X_train.max()).mean()})
drift["shift (training SDs)"] = (drift["test mean"] - drift["train 2006-2019 mean"]) / X_modern.std()
display(drift.sort_values("shift (training SDs)", key=abs, ascending=False).head(6).round(3))

**What we see.**
* **Probabilities are realistic everywhere, slightly high on test:** median predicted podiums per race 3.09 (1950-2005), 2.93 (2006-2019), 3.02 (validation) and 3.14 (test); average probability 0.159 on test against a real podium rate of 0.151. This is better than the previous version (3.32 on test).
* **The drift check explains the small excess:** the points features moved most. Before a race in 2022-2024 a driver has on average 58.8 points (36.4 in 2006-2019) and a team 118 (73): seasons now have 22-24 races and sprint races. The model learned "more points = more likely podium", so every test driver looks a little stronger. Less than 1% of test rows lie above the training maximum of any single feature, so this is a shift, not values never seen.
* **Era sensitivity, per season:** test PR-AUC 0.685 in 2022, 0.764 in 2023 and 0.635 in 2024. 2022 was the first season of new rules (P3 notes that the first season after a rule change, like 2014, was also harder); 2023 was dominated by one team and is the easiest; 2024 had four winning teams and is the hardest.
* **False alarms:** 127 on the test set; 20% of those cars did not finish the race, compared with 14% of all test starters. Some false alarms are breakdowns or crashes, which no pre-race model can see.

**Limitations of the models**

| Model | Main limitation in this project |
|---|---|
| Logistic regression (final) | Linear: no interactions unless they are built as features; the weights of twin features (points and positions, the qualifying columns, the missing flags) are not individually interpretable; it extrapolates linearly, so drifting features (points) shift its probabilities. |
| Random forest | Overfits with small leaves (train PR-AUC 0.84 vs test 0.67); probabilities are vote shares, not calibrated; cannot extrapolate beyond training values. |
| Gradient boosting | Sensitive to the number of trees (800 trees: train 0.78 vs validation 0.68) and tree size; many hyperparameters for a small validation set. |
| Shallow neural network | Stopping epoch chosen on validation, so its validation score is optimistic (0.763 on validation, 0.693 on test); wider or deeper networks did not help beyond noise; a black box without XAI. |
| k-NN, Naive Bayes (explored) | k-NN cannot use P3's training weights and weighs every feature equally; Naive Bayes assumes independent features, which our twin features break (validation PR-AUC 0.736 and 0.570). |

**Limitations of the data and the method**
* **Old seasons:** training uses 1950-2019 with the 1950-2005 rows at weight 0.25. On our validation, training on 2006-2019 only scores slightly higher (0.744 vs 0.736, within noise); the team keeps the old seasons on purpose (P3, 3.11 Part F).
* **Era sensitivity:** the test years are a new technical era (2022 ground-effect cars) that the model has never seen, and its first season is the hardest (0.685). The relative features (grid, qualifying, championship position) still transfer.
* **Points drift:** `pts_before` and `cons_pts_before` count points, and the number of points available per season has grown. P3 tested a points share (points divided by the leader's points) as a candidate and did not keep it; such a share would be more stable across eras.
* **Missing values carry meaning:** missing qualifying data is normal before 2006 but rare after it, so for a modern driver the model reads "missing" as a bad sign (4.13: a pole-sitter without qualifying data and championship position drops from 68% to 8%). For a record with a data gap the prediction is unreliable.
* **No flag for features that are never missing in training:** if such a feature (for example `grid_fixed`) is missing in a new record, it is silently replaced by the training median. These features are always known at the prediction cutoff, so a record without them should be treated as invalid input.
* **Small validation set:** 117 podiums; the bootstrap standard error of PR-AUC is about 0.04, larger than every difference between the tuned models, so the ranking of the models is uncertain.
* **The threshold is chosen for F1**, which counts a missed podium and a false alarm as equally bad. Another use weighs them differently and needs a different threshold.
* **The task itself:** incidents, penalties, safety cars, weather and strategy happen during the race and are invisible before it (Hamilton, Italy 2020, in 4.11). No pre-race model can predict them.

### 4.13 Hand-off to the inference function (P1)

P1's inference function uses the objects below. `FINAL_MODEL` is one pipeline (column selection + preprocessing + model), so it takes the columns in P3's `FEATURES` and handles missing values itself (4.2). The training weights are only used in training; a prediction needs no weight. Usage:

```python
proba = FINAL_MODEL.predict_proba(record[FEATURES])[:, 1]   # podium probability
podium = proba >= FINAL_THRESHOLD                           # yes / no decision
```

The check below runs the final model on one complete validation record and on the same record with its qualifying and championship values removed (as for a driver without a qualifying time in the first race of a season).

In [ ]:
complete_record = X_val[X_val.notna().all(axis=1)].iloc[[0]]
incomplete_record = complete_record.copy()
incomplete_record[["qual_position", "gap_to_pole_pct", "pos_before", "cons_pos_before"]] = np.nan
check = pd.concat([complete_record, incomplete_record]).assign(record=["complete", "incomplete"]).set_index("record")
check["podium probability"] = FINAL_MODEL.predict_proba(check[FEATURES])[:, 1].round(3)
check["podium (threshold)"] = check["podium probability"] >= FINAL_THRESHOLD
assert check["podium probability"].notna().all()
display(check[["grid_fixed", "qual_position", "pts_before", "pos_before", "podium probability", "podium (threshold)"]])
print(f"FINAL_NAME = {FINAL_NAME!r}, FINAL_THRESHOLD = {FINAL_THRESHOLD}, {len(FEATURES)} features")

---
# >>> Section from `notebooks/p1_inference.ipynb`


## Part 5: Inference Function

`predict_podium(record)` predicts whether one driver will finish on the podium (top 3) in one race. It accepts the engineered **pre-race** features produced by Part 3 and uses the final trained model selected in Part 4. It does not rebuild features from raw driver/race IDs or retrain the model.

The function follows a strict sequence: **input validation → feature validation → model input preparation → prediction**. If the record is incomplete or invalid, it raises a clear error and stops before the model is called. This prevents a prediction from being produced from missing or unusable input.

### 5.1 The function

The function uses `FEATURES` and `FEATURE_GROUPS` from Part 3, so the expected feature names and their training order stay synchronized with the modeling pipeline.

1. **Validate the record structure.** The input must be a dictionary for one driver-race record. Unexpected keys are rejected so IDs or unrelated columns cannot accidentally enter the model.
2. **Prevent target leakage.** Post-race fields and the target label are rejected because they are not known at the prediction cutoff: after qualifying and before the race starts.
3. **Check completeness.** A feature is missing if its key is absent or its value is `None` / `NaN`. If any required feature is missing, the function stops immediately.
4. **Validate supplied values.** Values must be finite numbers and satisfy basic domain checks for binary flags, positions, and non-negative counts/points.
5. **Prepare the model input.** Once validation succeeds, values are placed into a one-row DataFrame in exactly the same column order used for training. Preprocessing is not fitted again because it is already part of `FINAL_MODEL`.
6. **Predict.** The final pipeline returns the podium probability. `FINAL_THRESHOLD`, selected during model evaluation, converts it into the binary podium decision.

**Why reject incomplete records?** The fitted model pipeline includes imputation for its training/evaluation workflow, but an imputed value is not the observed pre-race value. This inference interface makes missing information explicit and does not silently substitute values or fallbacks.

In [ ]:
import numpy as np
import pandas as pd

# Map each final feature to its group so error messages explain where a missing
# input belongs (grid, qualifying, driver standings, or constructor standings).
GROUP_OF = {
    feature: group
    for group, features in FEATURE_GROUPS.items()
    for feature in features
}


def _is_missing(value):
    """Return True for None, NaN, or another scalar pandas missing-value marker."""
    try:
        result = pd.isna(value)
        # Feature values must be scalar; list/array values are handled as invalid.
        return bool(result) if np.isscalar(result) else False
    except (TypeError, ValueError):
        return False


def _validate_feature_value(feature, value):
    """
    Return a readable validation error, or None if the value is acceptable.
    These are basic domain checks based on the definitions in Part 3.
    """
    if isinstance(value, (bool, np.bool_)):
        return "must be numeric, not True/False"

    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return "must be a finite number"

    if not np.isfinite(number):
        return "must be a finite number"

    # These features encode yes/no states, so only 0 and 1 are valid.
    binary_features = {
        "no_qual_time",
        "qg_q3",
        "qg_q2_only",
        "standings_missing",
        "cons_standings_missing",
    }
    if feature in binary_features and number not in (0.0, 1.0):
        return "must be 0 or 1"

    # Positions are one-based; zero and negative positions are not valid.
    position_features = {
        "grid_fixed",
        "qual_position",
        "pos_before",
        "cons_pos_before",
    }
    if feature in position_features:
        if number <= 0:
            return "must be greater than 0"
        if not number.is_integer():
            return "must be a whole number"

    # Points and a percentage gap cannot be negative.
    non_negative_features = {
        "pts_before",
        "cons_pts_before",
        "gap_to_pole_pct",
    }
    if feature in non_negative_features and number < 0:
        return "cannot be negative"

    return None


def predict_podium(record, model=None, threshold=None):
    """
    Predict the probability of a podium finish for one driver-race record.

    Pipeline:
        1. Validate input type and keys.
        2. Reject post-race information.
        3. Reject missing required features.
        4. Reject invalid feature values.
        5. Arrange a one-row model input in training feature order.
        6. Call the final model and apply its selected threshold.

    Missing or invalid input raises an explanatory error. The model is not
    called unless every required feature is present and valid.
    """
    # Stage 1: validate the input container before inspecting its contents.
    if not isinstance(record, dict):
        raise TypeError(
            "Invalid input: record must be a dictionary of engineered "
            f"pre-race features, not {type(record).__name__}."
        )

    non_string_keys = [key for key in record if not isinstance(key, str)]
    if non_string_keys:
        raise ValueError(
            "All input keys must be feature-name strings. Invalid key(s): "
            + ", ".join(repr(key) for key in non_string_keys)
        )

    # Use the final Part 4 model and threshold unless test doubles are supplied.
    selected_model = FINAL_MODEL if model is None else model
    selected_threshold = FINAL_THRESHOLD if threshold is None else threshold

    # Stage 2: reject post-race data and the target to prevent future information
    # from leaking into a prediction that should be made before race start.
    leaky = set(record) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(
            "Post-race information is not allowed. Remove these field(s): "
            + ", ".join(sorted(leaky))
        )

    # Only the model features are accepted. Driver/race IDs and descriptive
    # context are used upstream and are not part of the model input.
    unexpected = set(record) - set(FEATURES)
    if unexpected:
        raise ValueError(
            "Unexpected field(s): "
            + ", ".join(sorted(unexpected))
            + ". Supply only the engineered features listed in FEATURES."
        )

    # Stage 3: check for absent keys and explicit missing markers before any
    # preprocessing or model call. An incomplete record stops here.
    missing = [
        feature for feature in FEATURES
        if feature not in record or _is_missing(record[feature])
    ]
    if missing:
        grouped = [
            f"{feature} ({GROUP_OF.get(feature, 'unknown group')})"
            for feature in missing
        ]
        raise ValueError(
            f"Prediction stopped: {len(missing)} required feature(s) are missing: "
            + ", ".join(grouped)
            + f". Supply all {len(FEATURES)} features and try again. "
              "The model was not called."
        )

    # Stage 4: validate values only after confirming that every feature exists.
    invalid = {}
    values = {}
    for feature in FEATURES:
        problem = _validate_feature_value(feature, record[feature])
        if problem:
            invalid[feature] = problem
        else:
            values[feature] = float(record[feature])

    if invalid:
        details = [
            f"{feature}: {reason}"
            for feature, reason in invalid.items()
        ]
        raise ValueError(
            "Prediction stopped because feature value(s) are invalid: "
            + "; ".join(details)
            + ". Correct these values and try again. The model was not called."
        )

    # The decision threshold must be a valid probability cutoff.
    try:
        selected_threshold = float(selected_threshold)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError("threshold must be a finite number between 0 and 1.") from error
    if not np.isfinite(selected_threshold) or not 0 <= selected_threshold <= 1:
        raise ValueError("threshold must be a finite number between 0 and 1.")

    # Stage 5: create the exact one-row, ordered input expected by the pipeline.
    # Preprocessing is already fitted inside FINAL_MODEL, so do not fit it here.
    X = pd.DataFrame(
        [[values[feature] for feature in FEATURES]],
        columns=FEATURES,
    )

    # Stage 6: run prediction only after all earlier validation stages pass.
    try:
        probabilities = selected_model.predict_proba(X)
        probability = float(probabilities[0, 1])
    except Exception as error:
        raise RuntimeError(
            "The final model failed during prediction. Check that it is fitted "
            "and uses the same FEATURES list supplied by Part 3."
        ) from error

    if not np.isfinite(probability) or not 0 <= probability <= 1:
        raise RuntimeError(
            f"The model returned an invalid podium probability: {probability!r}."
        )

    return {
        "status": "ok",
        "podium_probability": probability,
        "podium_prediction": int(probability >= selected_threshold),
        "missing_features": [],
        "invalid_features": [],
        "message": "Prediction completed successfully; all required features were present and valid.",
    }

### 5.2 Example A: Complete Record

This example selects a validation record for which all final model features are present. The actual finishing position is printed only to identify the historical example; it is not included in the input dictionary.

A successful call returns the estimated podium probability and the binary decision produced using the threshold selected in Part 4.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All required features are supplied, so this record follows the successful path through validation, model-input preparation, and prediction. The probability is an estimate from the final model, not a guarantee of the actual result. Overall model performance is evaluated separately in Part 4 using validation and unseen test data.

### 5.3 Example B: Incomplete Record

This example selects a validation record with at least one missing final feature. Missing fields are omitted from the input dictionary to represent an incomplete feature record.

The function should identify the missing feature names and their groups, raise `ValueError`, and stop before calling the model. This demonstrates how the inference interface handles incomplete input rather than silently imputing values.

In [ ]:
# Select a validation record that has at least one missing final feature.
incomplete_rows = table[
    (table["split"] == "validation")
    & ~table[FEATURES].notna().all(axis=1)
]
assert not incomplete_rows.empty, (
    "No incomplete validation row is available to demonstrate input error handling."
)

row_B = incomplete_rows.iloc[0]
record_B = {
    feature: value
    for feature, value in row_B[FEATURES].to_dict().items()
    if pd.notna(value)
}

print(describe_row(row_B))
print("input:", record_B)

try:
    predict_podium(record_B)
except ValueError as error:
    print("Prediction correctly refused.")
    print("Reason:", error)
    expected_missing = [
        feature for feature in FEATURES if pd.isna(row_B[feature])
    ]
    assert all(feature in str(error) for feature in expected_missing)
    assert "model was not called" in str(error)
else:
    raise AssertionError("The incomplete record should have raised ValueError.")

### 5.4 Validation Checks

The checks below cover both the normal inference path and expected failure cases:

* Complete validation rows must produce the same probability as a direct call to the final model with the same ordered features.
* Incomplete rows must raise `ValueError` and identify every missing feature.
* A tracking model verifies that missing input is rejected before `predict_proba` is called.
* Invalid numeric values, post-race columns, unexpected fields, and non-dictionary inputs must be rejected with clear errors.

In [ ]:
val = table[table["split"] == "validation"]
is_complete = val[FEATURES].notna().all(axis=1)

# Complete rows: the wrapper must agree with a direct call to the final pipeline.
val_ok = val[is_complete]
assert not val_ok.empty, "No complete validation rows are available for this check."

direct = FINAL_MODEL.predict_proba(val_ok[FEATURES])[:, 1]
results_ok = [
    predict_podium(row[FEATURES].to_dict())
    for _, row in val_ok.iterrows()
]
one_by_one = np.array([result["podium_probability"] for result in results_ok])

max_difference = float(np.max(np.abs(direct - one_by_one)))
print(
    "complete validation rows:", len(val_ok),
    "| largest probability difference from direct model call:", max_difference
)
assert all(result["status"] == "ok" for result in results_ok)
assert np.allclose(direct, one_by_one)

# Incomplete rows: verify each is rejected and every missing field is named.
val_bad = val[~is_complete]
for _, row in val_bad.iterrows():
    record = {
        feature: value
        for feature, value in row[FEATURES].to_dict().items()
        if pd.notna(value)
    }
    try:
        predict_podium(record)
    except ValueError as error:
        expected_missing = [
            feature for feature in FEATURES if pd.isna(row[feature])
        ]
        assert all(feature in str(error) for feature in expected_missing)
        assert "model was not called" in str(error)
    else:
        raise AssertionError(
            "An incomplete validation record unexpectedly produced a prediction."
        )

print(
    "incomplete validation rows:", len(val_bad),
    "| all were rejected before prediction"
)

# A spy model proves that missing input is rejected before predict_proba is called.
class ModelMustNotBeCalled:
    def predict_proba(self, X):
        raise AssertionError("predict_proba must not run for an incomplete record.")

incomplete_record = {
    feature: value
    for feature, value in row_B[FEATURES].to_dict().items()
    if pd.notna(value)
}
try:
    predict_podium(incomplete_record, model=ModelMustNotBeCalled())
except ValueError:
    print("early-abort check: passed; model was not called")
else:
    raise AssertionError("Incomplete input was not rejected.")

# Invalid feature values must be refused before prediction.
try:
    predict_podium({**record_A, "grid_fixed": "front"})
except ValueError as error:
    assert "grid_fixed" in str(error)
    print("invalid feature value correctly refused:", error)
else:
    raise AssertionError("A non-numeric grid_fixed value should have been rejected.")

# Post-race fields are forbidden because they would leak information from the outcome.
for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError as error:
        assert col in str(error)
    else:
        raise AssertionError(f"Post-race field {col} should have been rejected.")
print("post-race leakage checks: passed")

# Unrecognized fields and wrong input types must also fail clearly.
try:
    predict_podium({**record_A, "unexpected_column": 1})
except ValueError as error:
    assert "unexpected_column" in str(error)
    print("unexpected field correctly refused:", error)
else:
    raise AssertionError("Unexpected input fields should have been rejected.")

try:
    predict_podium(None)
except TypeError as error:
    print("wrong input type correctly refused:", error)
else:
    raise AssertionError("A non-dictionary input should have been rejected.")

### 5.5 Missing-Feature Analysis

This section reports how many validation records have at least one missing final feature and which features are missing most often. It helps show how often the strict inference input contract would require upstream data to be completed before prediction.

The analysis is descriptive: it does not change the feature set or model, and it does not use the test set to make input-policy decisions.

In [ ]:
share = len(val_bad) / len(val)
print(f"validation rows with missing features: {len(val_bad)} of {len(val)} ({share:.1%}); complete rows: {len(val_ok)}")

missing_counts = val[FEATURES].isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
display(
    missing_counts.rename("rows with this feature missing")
    .to_frame()
    .assign(group=lambda d: d.index.map(GROUP_OF))
)


### 5.6 Limitations

* The function accepts engineered features from Part 3; it does not construct them from raw race tables or driver/race identifiers.
* A prediction requires all final model features to be present and valid. Missing values are reported and rejected rather than silently imputed.
* Basic validation can detect missing, non-numeric, non-finite, and clearly out-of-domain values, but it cannot verify that upstream feature engineering is factually correct.
* The returned probability is the final model's estimate of a podium finish, not a guarantee of the actual race outcome.